# Wind VISION × NVIDIA PhysicsNeMo — Stage 34
## Evidence-preserving eigenspectrum failure attribution and controlled next experiment

**Start here after Stage 33 CUDA V2.** This is a **new** Colab notebook. Do not run historical notebooks or overwrite any Stage 32 / Stage 33 artifacts.

**Known Stage 33 outcome:** NVIDIA L4, 4,500 original-training fields, 500 internal development fields, 100 full-batch optimizer steps, PhysicsNeMo backend recorded, independent test **unaccessed**. Learned model spatial-correlation off-diagonal MAE **0.52026617** versus empirical-eigenspectrum baseline **0.00268187**; development gate **failed**. The two arms have nearly matching scalar temporal / spectral energy statistics.

**What this notebook does**
1. Mount Drive; check exact Stage 33 output and its SHA-256 hashes **before loading a checkpoint**.
2. Read Stage 33 training-only eigenvalues / eigenvectors and learned MLP weights using `torch.load(weights_only=True)`; verify the MLP reproduces the saved spectra.
3. Isolate the covariance failure analytically **without needing raw training HDF5**; compare target vs learned eigenmode allocations and implied spatial correlation.
4. Independently replay generated fields with the verified Stage 33 synthesis and matched random seeds; verify the eight-field preview.
5. If the original **training** HDF5 is still available, reconstruct the exact 500-field internal development partition, check its parent hash and reproduce Stage 33 metrics. **Never open the independent test.**
6. Run *diagnostic-only* controlled mixing ablations; optionally run a fixed 900-step, checkpoint-initialized **fresh-optimizer** fitting experiment if you decide it is justified. Results go to a **new** Stage 34 evidence directory.

**Scientific limits:** Stage 33 is a *learned eigenspectrum statistical hybrid*, **not** the original NASA c-LFM VAE + latent flow, and a PhysicsNeMo base class by itself is not a PhysicsNeMo-specific acceleration. These analyses reuse the original-training development cohort and are not blind evaluation. The fixed-energy shell is a synthetic reference artifact, not proof of real turbulence fidelity. A model that matches training moments but fails covariance is **not** an accepted generator.

**Source provenance:** Built from the user-provided `WIND_VISION_STAGE33_IMPLEMENTATION(1).zip`. Its full manifest passed SHA-256 verification locally; the original Stage 33 script SHA-256 is `9aa96d5601f21883416408a964972d18413acfc3cf62529d00777534a54a3906`. This notebook copies the needed exact architecture / feature formula / generator / metric formula as visible, auditable source. Its new Stage 34 analysis is not represented as upstream NASA code.

**How to run:** Runtime → Change runtime type → GPU (L4 is convenient, not required for the default diagnostics); run cells from top to bottom. No package installations or training occur by default. At the final decision cell, set `RUN_FIXED_900_STEP_EXTENSION=True` only if the diagnostic shows large train-only eigenvalue-allocation mismatch and you want that specific optimization ablation.


In [ ]:
# 1 — Runtime, configuration, mount; no training or data reads yet.
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, re, uuid, warnings, time
import numpy as np
import torch
from torch import nn

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    print('Not in Colab: Drive must already be mounted at /content/drive/MyDrive')

PROJECT = Path('/content/drive/MyDrive/CLFM_Research_Results/physicsnemo_proposal_reproduction')
EVIDENCE = PROJECT / 'evidence' / 'fullscale_physicsnemo_training'
STAGE33 = EVIDENCE / 'stage33_learned_eigenspectrum_5000_v2'
STAGE33_LOG = EVIDENCE / 'stage33_training_logs_v2' / 'stage33_5000.log'
EXPECTED_STAGE33_SOURCE_SHA = '9aa96d5601f21883416408a964972d18413acfc3cf62529d00777534a54a3906'
EXPECTED_STAGE33_CHECKPOINT_SHA = '907baee28d40192d3d0a7861e4c673f7e0e89151502931fad400c4e311d6d9f0'
EXPECTED_PARENT_TRAIN_SHA = '3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6'
EXPECTED_SOURCE_ZIP_SHA = '1b9a4490e3a4588edf36e6d66fcda12e631ce4bf0fe7611ac3bc2cb03b5d8f63'

# Leave blank to adopt report.json's original training-file path.
# If Drive paths changed, manually enter the EXACT ORIGINAL *TRAIN* HDF5 path here.
TRAIN_DATA_OVERRIDE = ''
REPLAY_FULL_256 = True          # Replay official same-seed 256-field cohorts; may take CPU time.
READ_INTERNAL_DEV_IF_FOUND = True   # Only 500 internal original-training development fields, with parent hash verified.
RUN_FIXED_900_STEP_EXTENSION = False  # Optional controlled optimization ablation. Fresh AdamW state, NOT exact optimizer resume.
WRITE_NEW_STAGE34_EVIDENCE = True

assert STAGE33.is_dir(), f'Missing Stage33 output directory: {STAGE33}'
print('STAGE34 CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('STAGE33_DIRECTORY:', STAGE33)
print('INDEPENDENT_TEST_ACCESS: DISABLED BY DESIGN')
print('RUN_FIXED_900_STEP_EXTENSION:', RUN_FIXED_900_STEP_EXTENSION)


Mounted at /content/drive
STAGE34 CUDA: True NVIDIA L4
STAGE33_DIRECTORY: /content/drive/MyDrive/CLFM_Research_Results/physicsnemo_proposal_reproduction/evidence/fullscale_physicsnemo_training/stage33_learned_eigenspectrum_5000_v2
INDEPENDENT_TEST_ACCESS: DISABLED BY DESIGN
RUN_FIXED_900_STEP_EXTENSION: False


In [ ]:
# 2 — Verify provenance, run report and saved artifact fingerprints BEFORE model deserialization.
def file_sha256(p, chunk_bytes=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(p).open('rb') as stream:
        for block in iter(lambda: stream.read(chunk_bytes), b''):
            h.update(block)
    return h.hexdigest()

source_file = STAGE33 / 'train_learned_eigenspectrum.py'
checkpoint_path = STAGE33 / 'learned_spectral_model.pt'
report_file = STAGE33 / 'report.json'
preview_file = STAGE33 / 'preview_8_fields.npz'
manifest_file = STAGE33 / 'SHA256SUMS.json'

for p in (source_file, checkpoint_path, report_file, preview_file, manifest_file):
    assert p.is_file(), f'Required Stage33 evidence missing: {p}'
assert file_sha256(source_file) == EXPECTED_STAGE33_SOURCE_SHA, 'Stage33 source snapshot changed'
assert file_sha256(checkpoint_path) == EXPECTED_STAGE33_CHECKPOINT_SHA, 'Stage33 checkpoint changed'
report = json.loads(report_file.read_text())
manifest = json.loads(manifest_file.read_text())
assert report.get('development_screening_all_passed') is False, 'Unexpected Stage33 gate state'
assert report.get('no_final_test_access') is True, 'Stage33 report does not assert sealed independent test'
assert report.get('source_dataset_sha256') == EXPECTED_PARENT_TRAIN_SHA, 'Parent train corpus identity mismatch'
assert report.get('model_checkpoint_sha256') == EXPECTED_STAGE33_CHECKPOINT_SHA
assert report.get('source_script_sha256') == EXPECTED_STAGE33_SOURCE_SHA
assert report.get('train_count') == 4500 and report.get('dev_count') == 500
for filename, expected in manifest.items():
    p = STAGE33 / filename
    assert p.is_file() and file_sha256(p) == expected, f'Manifest failure: {filename}'
assert report['backend_executed'] == 'PhysicsNeMo', 'Stage33 backend was not native PhysicsNeMo'
print('STAGE33_SOURCE_SHA256_VERIFIED:', EXPECTED_STAGE33_SOURCE_SHA)
print('STAGE33_CHECKPOINT_SHA256_VERIFIED:', EXPECTED_STAGE33_CHECKPOINT_SHA)
print('ALL_STAGE33_MANIFEST_ENTRIES_MATCH:', len(manifest))
print('BACKEND_RECORDED_STAGE33:', report['backend_executed'])
print('STAGE33_LEARNED_GATE:', report['development_screening_all_passed'])
print('STAGE33_METRICS:',json.dumps(report['development_learned'],indent=2))
print('STAGE33_BASELINE:',json.dumps(report['development_baseline'],indent=2))
if STAGE33_LOG.is_file():
    print('LOG_TAIL:\n','\n'.join(STAGE33_LOG.read_text(errors='replace').splitlines()[-15:]))
else:
    print('Additional training log not present at expected path; report remains source of record')


STAGE33_SOURCE_SHA256_VERIFIED: 9aa96d5601f21883416408a964972d18413acfc3cf62529d00777534a54a3906
STAGE33_CHECKPOINT_SHA256_VERIFIED: 907baee28d40192d3d0a7861e4c673f7e0e89151502931fad400c4e311d6d9f0
ALL_STAGE33_MANIFEST_ENTRIES_MATCH: 4
BACKEND_RECORDED_STAGE33: PhysicsNeMo
STAGE33_LEARNED_GATE: False
STAGE33_METRICS: {
  "band_power_ratios": {
    "high": 1.008999090121558,
    "low": 1.0115926844629974,
    "middle": 0.9903194448389756,
    "tail": 0.991568044522554
  },
  "between_field_temporal_energy_cv": 2.4421044482435883e-16,
  "mean_profile_rmse_mps": 0.0805193758352065,
  "spatial_corr_offdiag_mae": 0.5202661702499826,
  "step_rms_ratio": 0.9999782351363181,
  "temporal_variance_ratio": 1.0000000186438147
}
STAGE33_BASELINE: {
  "band_power_ratios": {
    "high": 1.0101005461574073,
    "low": 0.9974433269735777,
    "middle": 0.9921946708341974,
    "tail": 0.9936756719529033
  },
  "between_field_temporal_energy_cv": 3.391424139120893e-16,
  "mean_profile_rmse_mps": 0.080519

## Exact Stage 33 neural model and spectral generator (visible source reproduction)
These definitions reproduce the relevant formulas from the verified Stage 33 script rather than importing and re-executing the original training program. The checkpoint records that Stage 33 used PhysicsNeMo; *this diagnostic notebook uses PyTorch classes to reload compatible weights* and does **not** claim to benchmark PhysicsNeMo.


In [ ]:
# 3 — Verified source formulas, no HDF5 and no training.
class LearnedEigenShares(nn.Module):
    def __init__(self, width=96):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(),
            nn.Linear(width, width), nn.GELU(),
            nn.Linear(width, 1),
        )
    def forward(self, features):
        return torch.softmax(self.network(features).squeeze(-1), dim=-1)

def make_features(nfreq, sensors, device):
    freq = torch.arange(1, nfreq + 1, device=device).float() / (nfreq + 1)
    mode = torch.arange(sensors, device=device).float() / max(sensors - 1, 1)
    f, m = torch.meshgrid(freq, mode, indexing='ij')
    return torch.stack((
        f, torch.log1p(f * nfreq) / np.log1p(nfreq),
        m, torch.log1p(m * sensors) / np.log1p(sensors),
        torch.sin(np.pi * f), torch.sqrt(m + 1e-6),
    ), dim=-1)

def generate(n, ntime, profile, mean_cov, vectors, eigenvalues, seed, batch_size=64):
    """Exact Stage33 sampling logic. Outputs [N,S,T]. No reference data accessed."""
    rng = np.random.default_rng(seed)
    nfreq, sensors, _ = vectors.shape
    assert nfreq == ntime // 2 + 1 and eigenvalues.shape == (nfreq, sensors)
    dc_val, dc_vec = np.linalg.eigh(mean_cov)
    dc_factor = dc_vec @ np.diag(np.sqrt(np.clip(dc_val, 0, None)))
    outs = []
    for a in range(0, n, batch_size):
        b = min(batch_size, n-a)
        coefficients = np.zeros((b, sensors, nfreq), np.complex128)
        dc_means = profile + rng.standard_normal((b, sensors)) @ dc_factor.T
        coefficients[:, :, 0] = np.sqrt(ntime) * dc_means
        phases = rng.uniform(0, 2*np.pi, (b, nfreq-2, sensors))
        innovations = np.exp(1j * phases) * np.sqrt(eigenvalues[1:-1])[None, :, :]
        coefficients[:, :, 1:-1] = np.einsum('fsk,bfk->bsf', vectors[1:-1], innovations, optimize=True)
        signs = rng.integers(0, 2, (b, sensors))*2-1
        nyquist_basis = vectors[-1].real
        coefficients[:, :, -1] = (signs*np.sqrt(eigenvalues[-1])[None, :]) @ nyquist_basis.T
        generated = np.fft.irfft(coefficients, n=ntime, axis=-1, norm='ortho')
        assert np.isfinite(generated).all()
        outs.append(generated)
    return np.concatenate(outs)

def metrics(samples, reference):
    def centered(x): return x - x.mean(axis=-1,keepdims=True)
    x, y = centered(samples), centered(reference)
    refvar = float(np.mean(y*y))
    rv = float(np.mean(x*x) / refvar)
    refstep = float(np.mean(np.diff(y,axis=-1)**2))
    step = float(np.sqrt(np.mean(np.diff(x,axis=-1)**2) / refstep))
    window = np.hanning(x.shape[-1])
    xf = np.abs(np.fft.rfft(x*window,axis=-1))**2
    yf = np.abs(np.fft.rfft(y*window,axis=-1))**2
    bands = {name:float(xf[...,lo:hi].mean()/yf[...,lo:hi].mean())
             for name,lo,hi in [('low',1,7),('middle',7,17),('high',17,65),('tail',65,129)]}
    xc = x.transpose(1,0,2).reshape(x.shape[1],-1)
    yc = y.transpose(1,0,2).reshape(y.shape[1],-1)
    xi,yi = np.corrcoef(xc),np.corrcoef(yc)
    off = np.triu_indices(x.shape[1],k=1)
    corr_mae = float(np.abs(xi[off]-yi[off]).mean())
    en = np.mean(x*x,axis=(1,2))
    return {'temporal_variance_ratio':rv,'step_rms_ratio':step,
            'band_power_ratios':bands,
            'spatial_corr_offdiag_mae':corr_mae,
            'between_field_temporal_energy_cv':float(en.std()/en.mean()),
            'mean_profile_rmse_mps':float(np.sqrt(np.mean((samples.mean((0,2))-reference.mean((0,2)))**2)))}

def model_implied_covariance(eigenvectors, eigenvalues, ntime=256):
    """Time-centered spatial covariance implied by real irfft with norm=ortho.
    For f=1..F-2, both positive and negative frequencies contribute 1/T each.
    Nyquist contributes once. DC is excluded because each field is time-centered.
    """
    fvec = eigenvectors[1:-1]
    lam = eigenvalues[1:-1]
    interior = (2.0/ntime)*np.einsum('fsk,fk,ftk->st',fvec,lam,fvec.conj(),optimize=True).real
    vn = eigenvectors[-1].real
    nyquist = (vn*eigenvalues[-1][None,:]) @ vn.T / ntime
    cov = interior + nyquist
    return (cov+cov.T)/2

def cov_to_corr(c):
    sd=np.sqrt(np.clip(np.diag(c).real,1e-30,None))
    return (c / (sd[:,None]*sd[None,:])).real

def corr_mae(c1,c2):
    off=np.triu_indices(c1.shape[0],1)
    return float(np.abs(c1[off]-c2[off]).mean())

# Tiny mathematical smoke test; no NASA files touched.
sensors=4; ntime=16; nfreq=ntime//2+1
vec0=np.broadcast_to(np.eye(sensors,dtype=complex), (nfreq,sensors,sensors)).copy()
eig0=np.ones((nfreq,sensors))
c0=model_implied_covariance(vec0,eig0,ntime)
assert np.max(np.abs(c0-np.eye(sensors)*((2*(nfreq-2)+1)/ntime)))<1e-12
print('STAGE34_SPECTRAL_COVARIANCE_SMOKE_PASS')


STAGE34_SPECTRAL_COVARIANCE_SMOKE_PASS


In [ ]:
# ================================================================
# CELL 4 — STAGE 34 CHECKPOINT RESTORATION (FIXED)
# ================================================================
# Fix: use float32-appropriate tolerance for spectral power.
# Preserves all scientific checks and existing evidence.
# No training, checkpoint modification, or independent-test access.

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# 1. Verify checkpoint provenance
assert ckpt["format"] == "wind_vision_stage33_learned_eigenspectrum_v1"
assert ckpt["source_dataset_sha256"] == EXPECTED_PARENT_TRAIN_SHA
assert ckpt["train_ids_sha256"] == report["train_indices_sha256"]
assert ckpt["dev_ids_sha256"] == report["dev_indices_sha256"]
assert ckpt["seed"] == 331001

# 2. Restore saved spectral data
vectors = ckpt["train_only_eigenvectors"].numpy().astype(
    np.complex128, copy=False
)
values = ckpt["train_only_eigenvalues"].numpy().astype(
    np.float64, copy=False
)
learned_values = ckpt["train_only_learned_eigenvalues"].numpy().astype(
    np.float64, copy=False
)
profile = ckpt["train_only_profile"].numpy().astype(
    np.float64, copy=False
)
mean_cov = ckpt["train_only_mean_cov"].numpy().astype(
    np.float64, copy=False
)

# 3. Validate shapes and numerical integrity
assert vectors.shape == (129, 100, 100)
assert values.shape == (129, 100)
assert learned_values.shape == values.shape
assert profile.shape == (100,)
assert mean_cov.shape == (100, 100)

assert np.isfinite(vectors).all()
assert np.isfinite(values).all()
assert np.isfinite(learned_values).all()
assert (values >= 0).all()
assert (learned_values >= 0).all()

assert np.allclose(
    vectors.conj().swapaxes(-1, -2) @ vectors,
    np.eye(100),
    atol=1e-8
)

# 4. Restore learned PhysicsNeMo model
model = LearnedEigenShares(
    width=ckpt["model_config"]["width"]
).eval()

model.load_state_dict(
    ckpt["model_state"],
    strict=True
)

features = make_features(127, 100, "cpu")

with torch.no_grad():
    nn_shares = model(features).numpy().astype(np.float64)

# 5. Verify learned eigenvalue allocations
total_power = values[1:-1].sum(axis=-1)

emp_shares = values[1:-1] / total_power[:, None]
saved_shares = learned_values[1:-1] / total_power[:, None]

assert np.allclose(
    nn_shares,
    saved_shares,
    rtol=3e-6,
    atol=3e-6
), "Saved model does not match learned eigenvalues"

# Verify DC and Nyquist components remain unchanged
assert np.allclose(learned_values[0], values[0])
assert np.allclose(learned_values[-1], values[-1])

# 6. FIXED — Verify spectral power conservation
# Stage 33's softmax uses float32.
# Differences around 1e-7 are normal numerical rounding.

ref_power = values.sum(axis=-1, dtype=np.float64)
learned_power = learned_values.sum(axis=-1, dtype=np.float64)

power_rel_err = (
    np.abs(ref_power - learned_power)
    / np.maximum(np.abs(ref_power), 1e-30)
)

max_power_rel_err = float(np.max(power_rel_err))

max_interior_rel_err = float(
    np.max(power_rel_err[1:-1])
)

softmax_max_sum_error = float(
    np.max(np.abs(nn_shares.sum(axis=-1) - 1.0))
)

print("SOFTMAX_MAX_SHARE_SUM_ERROR:", softmax_max_sum_error)
print("POWER_MAX_RELATIVE_ERROR_ALL_FREQUENCIES:", max_power_rel_err)
print("POWER_MAX_RELATIVE_ERROR_INTERIOR:", max_interior_rel_err)

# Appropriate numerical tolerance without disabling verification
assert np.all(np.isfinite(power_rel_err)), (
    "Nonfinite spectral power residual"
)

assert np.allclose(
    ref_power,
    learned_power,
    atol=1e-10,
    rtol=1e-6
), (
    "Substantive spectral power conservation failure",
    max_power_rel_err,
    max_interior_rel_err
)

assert softmax_max_sum_error < 1e-6, (
    "Softmax normalization mismatch",
    softmax_max_sum_error
)

# 7. Final verification
print()
print("=" * 70)
print("STAGE 34 — CHECKPOINT VERIFICATION")
print("=" * 70)
print("CHECKPOINT_RESTORED_AND_LEARNED_SPECTRUM_MATCHED")
print("SOURCE_DATASET_SHA256:", ckpt["source_dataset_sha256"])
print("MODEL_BACKEND:", ckpt["source_backend"])
print("MAX_POWER_RELATIVE_ERROR:", max_power_rel_err)
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("RESULT: PASS")
print("=" * 70)

SOFTMAX_MAX_SHARE_SUM_ERROR: 1.6695099702701555e-07
POWER_MAX_RELATIVE_ERROR_ALL_FREQUENCIES: 1.22056007898497e-07
POWER_MAX_RELATIVE_ERROR_INTERIOR: 1.22056007898497e-07

STAGE 34 — CHECKPOINT VERIFICATION
CHECKPOINT_RESTORED_AND_LEARNED_SPECTRUM_MATCHED
SOURCE_DATASET_SHA256: 3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6
MODEL_BACKEND: PhysicsNeMo
MAX_POWER_RELATIVE_ERROR: 1.22056007898497e-07
INDEPENDENT_TEST_ACCESSED: FALSE
RESULT: PASS


In [ ]:
# 5 — Training-only mode allocation and analytical covariance attribution.
EPS = 1e-20
H_target = -np.sum(emp_shares*np.log(np.maximum(emp_shares,EPS)),axis=-1)
H_learned = -np.sum(nn_shares*np.log(np.maximum(nn_shares,EPS)),axis=-1)
CE = -np.sum(emp_shares*np.log(np.maximum(nn_shares,EPS)),axis=-1)
KL=CE-H_target
TV=0.5*np.sum(np.abs(emp_shares-nn_shares),axis=-1)
mode_allocation_summary={
    'median_target_entropy_nats':float(np.median(H_target)),
    'median_learned_entropy_nats':float(np.median(H_learned)),
    'mean_training_only_KL_target_to_learned_nats':float(np.mean(KL)),
    'maximum_training_only_KL_nats':float(np.max(KL)),
    'mean_total_variation_share':float(np.mean(TV)),
    'median_top1_share_training_target':float(np.median(emp_shares.max(-1))),
    'median_top1_share_stage33_learned':float(np.median(nn_shares.max(-1))),
    'median_top5_share_training_target':float(np.median(np.sort(emp_shares,axis=-1)[:,-5:].sum(-1))),
    'median_top5_share_stage33_learned':float(np.median(np.sort(nn_shares,axis=-1)[:,-5:].sum(-1))),
}
C_emp=model_implied_covariance(vectors,values)
C_net=model_implied_covariance(vectors,learned_values)
R_emp=cov_to_corr(C_emp)
R_net=cov_to_corr(C_net)
analytic_corr_error_vs_baseline=corr_mae(R_net,R_emp)
mode_allocation_summary['analytical_corr_mae_stage33_vs_training_baseline']=analytic_corr_error_vs_baseline
mode_allocation_summary['mean_trace_rel_error']=float(np.mean(np.abs(total_power-learned_values[1:-1].sum(-1))/total_power))
assert mode_allocation_summary['mean_trace_rel_error']<1e-5
print('=== STAGE34 TRAINING-ONLY SPECTRAL FAILURE ATTRIBUTION ===')
print(json.dumps(mode_allocation_summary,indent=2))
print('FREQUENCY_DETAIL: freq_idx, KL, target_top1, learned_top1')
for fi in [0,1,2,5,10,20,40,70,100,126]:
    print(fi+1, float(KL[fi]),float(emp_shares[fi].max()),float(nn_shares[fi].max()))
if mode_allocation_summary['mean_training_only_KL_target_to_learned_nats']>.05:
    print('EVIDENCE: Learned allocations remain materially different from train-only targets; investigate incomplete optimization and feature capacity.')
else:
    print('EVIDENCE: Average KL is modest; prioritize sensitive frequencies, synthesis and finite-sampling effects, not blindly more epochs.')
print('NOTE: This is analytic attribution, not an independent development/test result.')


=== STAGE34 TRAINING-ONLY SPECTRAL FAILURE ATTRIBUTION ===
{
  "median_target_entropy_nats": 2.2639436257384933,
  "median_learned_entropy_nats": 2.4951156026086756,
  "mean_training_only_KL_target_to_learned_nats": 0.23488093422475403,
  "maximum_training_only_KL_nats": 0.8795824324893062,
  "mean_total_variation_share": 0.24859109954718284,
  "median_top1_share_training_target": 0.17217843324577378,
  "median_top1_share_stage33_learned": 0.2683267593383789,
  "median_top5_share_training_target": 0.665141497900067,
  "median_top5_share_stage33_learned": 0.6957877799868584,
  "analytical_corr_mae_stage33_vs_training_baseline": 0.5166931495191308,
  "mean_trace_rel_error": 3.50289359891505e-08
}
FREQUENCY_DETAIL: freq_idx, KL, target_top1, learned_top1
1 0.8795824324893062 0.914053268539033 0.29391804337501526
2 0.7306400257034525 0.8634202004018954 0.29754018783569336
3 0.613634042968331 0.8144469460625237 0.29949480295181274
6 0.39311084835447296 0.6906700355527421 0.30097833275794983

In [ ]:
# 6 — Matched innovation seeds, deterministic replay of official Stage33 preview and spatial correlations.
# This re-generates data solely from train-only moments in the checkpoint. No HDF5 is accessed.
with np.load(preview_file, allow_pickle=False) as preview:
    expected_8_learned=np.array(preview['learned_fields'])
    expected_8_baseline=np.array(preview['empirical_baseline_fields'])
    preview_train_ids=np.array(preview['train_indices'])
    preview_dev_ids=np.array(preview['development_indices'])
assert expected_8_learned.shape==(8,100,256)
assert expected_8_baseline.shape==(8,100,256)
assert len(preview_train_ids)==4500 and len(preview_dev_ids)==500
assert set(preview_train_ids.tolist()).isdisjoint(set(preview_dev_ids.tolist()))

n_generate=256 if REPLAY_FULL_256 else 8
start_replay=time.monotonic()
baseline_generated=generate(n_generate,256,profile,mean_cov,vectors,values,331001+101)
learned_generated=generate(n_generate,256,profile,mean_cov,vectors,learned_values,331001+101)
assert np.array_equal(baseline_generated[:8].astype(np.float32),expected_8_baseline), 'Baseline preview replay mismatch'
assert np.array_equal(learned_generated[:8].astype(np.float32),expected_8_learned), 'Learned preview replay mismatch'
assert np.isfinite(baseline_generated).all() and np.isfinite(learned_generated).all()

def empirical_corr(samples):
    centered=samples-samples.mean(-1,keepdims=True)
    flat=centered.transpose(1,0,2).reshape(samples.shape[1],-1)
    return np.corrcoef(flat)

R_baseline_sample=empirical_corr(baseline_generated)
R_learned_sample=empirical_corr(learned_generated)
stochastic_summary={
    'generated_fields_per_arm':int(n_generate),
    'baseline_sample_vs_analytic_mae':corr_mae(R_baseline_sample,R_emp),
    'learned_sample_vs_analytic_mae':corr_mae(R_learned_sample,R_net),
    'learned_vs_baseline_sample_mae':corr_mae(R_learned_sample,R_baseline_sample),
    'stage33_first_eight_fields_exact_replay':True,
    'elapsed_seconds':time.monotonic()-start_replay,
}
print('MATCHED_SEED_PREVIEW_EXACT_REPLAY_PASS')
print(json.dumps(stochastic_summary,indent=2))
print('Simulation matches the verified Stage33 generator (first 8 float32 fields bitwise).')


MATCHED_SEED_PREVIEW_EXACT_REPLAY_PASS
{
  "generated_fields_per_arm": 256,
  "baseline_sample_vs_analytic_mae": 0.002509268431464805,
  "learned_sample_vs_analytic_mae": 0.015877612062220645,
  "learned_vs_baseline_sample_mae": 0.520062251108218,
  "stage33_first_eight_fields_exact_replay": true,
  "elapsed_seconds": 0.9564417130000038
}
Simulation matches the verified Stage33 generator (first 8 float32 fields bitwise).


## Optional exact Stage 33 development replay

This is the historical **500-field internal development partition**, not an independent dataset. Its file must have the original 5,000-training-field SHA-256, **not** the separate independent 1,000-field test checksum. The notebook derives and verifies the exact shuffled IDs and hashes from Stage 33 before reading any field. If the original training HDF5 is absent from Drive, the previous analytical and stochastic diagnostics still run and this cell skips safely.


In [ ]:
# 7 — Read ONLY the verified 5000-field original TRAIN HDF5 (optional); repeat exact Stage33 development metrics.
import h5py

def source_guard(p):
    """Prevent accidental access to independent-test-named files."""
    p=Path(p)
    lower=p.name.lower()
    assert p.suffix.lower() in ('.h5','.hdf5'), f'Not an HDF5: {p}'
    assert ('train' in lower and 'test' not in lower and 'independent' not in lower), f'Not an original training file: {p}'
    assert p.is_file(), f'Training HDF5 does not exist: {p}'
    return p

ids=np.random.default_rng(331001).permutation(5000)
dev_ids=ids[:500].copy()
train_ids=ids[500:].copy()
def ids_sha(arr):
    return hashlib.sha256(np.asarray(arr,dtype='<i8').tobytes()).hexdigest()
assert ids_sha(train_ids)==ckpt['train_ids_sha256']==report['train_indices_sha256']
assert ids_sha(dev_ids)==ckpt['dev_ids_sha256']==report['dev_indices_sha256']
assert np.array_equal(preview_train_ids,train_ids)
assert np.array_equal(preview_dev_ids,dev_ids)
print('EXACT_4500_500_SPLIT_IDENTITIES_PASS')

candidate_train=str(TRAIN_DATA_OVERRIDE).strip() or str(report['configuration']['input'])
train_path=Path(candidate_train)
dev=None; dev_replay_summary={'status':'skipped_train_file_missing'}
if not READ_INTERNAL_DEV_IF_FOUND:
    print('Internal development read disabled by configuration')
elif not train_path.is_file():
    print('Original TRAIN HDF5 not found at path:',train_path)
    print('If you want internal development replay, set TRAIN_DATA_OVERRIDE in Cell 1 to the verified 5000-field train HDF5 path; rerun cells 1 and 7 onward.')
else:
    source_guard(train_path)
    got_hash=file_sha256(train_path)
    assert got_hash==EXPECTED_PARENT_TRAIN_SHA, ('Wrong original training file; refusing access',got_hash)
    with h5py.File(train_path,'r') as h:
        g=h['wind_samples']
        assert len(g)>=5000 and g['0/v1'].shape==(100,256)
        dev=np.stack([g[str(int(i))]['v1'][:] for i in dev_ids]).astype(np.float64)
    assert dev.shape==(500,100,256) and np.isfinite(dev).all()
    # Exact published 256-sample comparison only if official count used.
    if not REPLAY_FULL_256:
        print('Internal reference loaded; REPLAY_FULL_256=False so official Stage33 metric replay is skipped')
        dev_replay_summary={'status':'dev_loaded_but_not_official_256_replay'}
    else:
        baseline_dev_metrics=metrics(baseline_generated,dev)
        learned_dev_metrics=metrics(learned_generated,dev)
        def compare_original(got,ref):
            for key in ['temporal_variance_ratio','step_rms_ratio','spatial_corr_offdiag_mae',
                        'between_field_temporal_energy_cv','mean_profile_rmse_mps']:
                assert np.isclose(got[key],ref[key],rtol=2e-7,atol=2e-8), (key,got[key],ref[key])
            for key in ['low','middle','high','tail']:
                assert np.isclose(got['band_power_ratios'][key],ref['band_power_ratios'][key],rtol=2e-7,atol=2e-8)
        compare_original(baseline_dev_metrics,report['development_baseline'])
        compare_original(learned_dev_metrics,report['development_learned'])
        dev_replay_summary={
            'status':'full_stage33_metrics_exact_numerical_replay_pass',
            'baseline_development':baseline_dev_metrics,
            'learned_development':learned_dev_metrics,
            'no_independent_test_access':True,
        }
        print('DEVELOPMENT_METRIC_REPLAY_PASS')
        print('LEARNED_SPATIAL_CORR_MAE:',learned_dev_metrics['spatial_corr_offdiag_mae'])
        print('BASELINE_SPATIAL_CORR_MAE:',baseline_dev_metrics['spatial_corr_offdiag_mae'])

# Analytic population-vs-dev covariance clarifies model error vs finite-sample error.
if dev is not None:
    R_dev=empirical_corr(dev)
    analytic_dev_summary={
        'empirical_baseline_analytic_to_internal_dev_corr_mae':corr_mae(R_emp,R_dev),
        'learned_analytic_to_internal_dev_corr_mae':corr_mae(R_net,R_dev),
        'sampled_baseline_to_internal_dev_corr_mae':corr_mae(R_baseline_sample,R_dev),
        'sampled_learned_to_internal_dev_corr_mae':corr_mae(R_learned_sample,R_dev),
    }
    print('=== ANALYTIC VERSUS INTERNAL DEVELOPMENT ===')
    print(json.dumps(analytic_dev_summary,indent=2))
else:
    R_dev=None; analytic_dev_summary={'status':'no_original_training_file_loaded'}


EXACT_4500_500_SPLIT_IDENTITIES_PASS
Original TRAIN HDF5 not found at path: /content/wind_train_data.hdf5
If you want internal development replay, set TRAIN_DATA_OVERRIDE in Cell 1 to the verified 5000-field train HDF5 path; rerun cells 1 and 7 onward.


In [ ]:
# 8 — Diagnostic covariance ablation: interpolate train-only eigenspectra (not a candidate test pass).
# alpha=0 is the empirical baseline; alpha=1 is Stage33 learned model.
# Interpolation is a forensic causal attribution, not validated new neural research.
alpha_rows=[]
reference_corr=R_dev if R_dev is not None else R_emp
reference_label='internal_development' if R_dev is not None else 'training_only_empirical_baseline'
for alpha in (0.0,0.25,0.50,0.75,1.0):
    candidate=values.copy()
    candidate[1:-1]=(1-alpha)*values[1:-1]+alpha*learned_values[1:-1]
    rc=cov_to_corr(model_implied_covariance(vectors,candidate))
    alpha_rows.append({'alpha_learned':alpha,
                       f'analytic_spatial_corr_mae_vs_{reference_label}':corr_mae(rc,reference_corr),
                       'analytic_spatial_corr_mae_vs_empirical_baseline':corr_mae(rc,R_emp)})
print('=== FROZEN-EIGENVECTOR / FROZEN-POWER / INTERPOLATED-EIGENVALUE ABLATION ===')
for row in alpha_rows: print(row)
print('NO TEST DATA ACCESSED; mixing alpha is diagnostic and must not be tuned on dev then promoted as blind.')


=== FROZEN-EIGENVECTOR / FROZEN-POWER / INTERPOLATED-EIGENVALUE ABLATION ===
{'alpha_learned': 0.0, 'analytic_spatial_corr_mae_vs_training_only_empirical_baseline': 0.0, 'analytic_spatial_corr_mae_vs_empirical_baseline': 0.0}
{'alpha_learned': 0.25, 'analytic_spatial_corr_mae_vs_training_only_empirical_baseline': 0.12115454857668012, 'analytic_spatial_corr_mae_vs_empirical_baseline': 0.12115454857668012}
{'alpha_learned': 0.5, 'analytic_spatial_corr_mae_vs_training_only_empirical_baseline': 0.2468406844444004, 'analytic_spatial_corr_mae_vs_empirical_baseline': 0.2468406844444004}
{'alpha_learned': 0.75, 'analytic_spatial_corr_mae_vs_training_only_empirical_baseline': 0.3782329040247308, 'analytic_spatial_corr_mae_vs_empirical_baseline': 0.3782329040247308}
{'alpha_learned': 1.0, 'analytic_spatial_corr_mae_vs_training_only_empirical_baseline': 0.5166931495191308, 'analytic_spatial_corr_mae_vs_empirical_baseline': 0.5166931495191308}
NO TEST DATA ACCESSED; mixing alpha is diagnostic and 

## Optional Stage 34 hypothesis test: is 100-step fitting insufficient?

The Stage 33 source shows the MLP's training cross-entropy was still decreasing at step 100. If the preceding cells show a substantial **training-only** KL mismatch between the target and learned eigenvalue shares, an informative controlled experiment is to fit that *same model architecture* for an **additional fixed 900 steps**.

**Important:** Stage 33 did not save AdamW optimizer moments. This is **checkpoint-initialized continued fitting with a freshly constructed AdamW**, not an exact resume of the Stage 33 optimization trajectory. The training target is the saved 4,500-field empirical covariance. No dev data enter training or checkpoint selection. This ablation tests whether simple optimization time is the blocker, not whether the model learns unseen physical dynamics.

To enable, change `RUN_FIXED_900_STEP_EXTENSION=True` in Cell 1 and rerun this cell and the final saving cell. Its fixed step count is a predeclared diagnostic; we do not run an open-ended hyperparameter search against reused development data.


In [ ]:
# 9 — OPTIONAL additional fitting. Default: disabled. Uses saved TRAIN-only eigenvalues, no HDF5.
extension_summary={'status':'not_run'}
extension_state=None
extension_learned_values=None
if RUN_FIXED_900_STEP_EXTENSION:
    if mode_allocation_summary['mean_training_only_KL_target_to_learned_nats']<1e-4:
        raise RuntimeError('Target KL already ~0; optimization-extension premise not supported. Do not run.')
    devicename='cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(331001+34)
    if devicename=='cuda': torch.cuda.manual_seed_all(331001+34)
    torch.backends.cuda.matmul.allow_tf32=False
    extended=LearnedEigenShares(width=96).to(devicename)
    extended.load_state_dict(ckpt['model_state'],strict=True)
    target=torch.tensor(emp_shares,dtype=torch.float32,device=devicename)
    features_device=make_features(127,100,devicename)
    optimizer=torch.optim.AdamW(extended.parameters(),lr=5e-4,weight_decay=1e-5) # fresh state; explicitly not exact resumed optimizer
    losses=[]
    start_fit=time.monotonic()
    for step in range(900):
        extended.train()
        optimizer.zero_grad(set_to_none=True)
        prediction=extended(features_device)
        loss=-(target*(prediction+1e-12).log()).sum(-1).mean()
        assert torch.isfinite(loss).item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(extended.parameters(),max_norm=10.0)
        optimizer.step()
        if (step+1)%100==0:
            losses.append({'additional_steps':step+1,'train_cross_entropy':float(loss.detach().cpu())})
            print('STAGE34_ADDITIONAL_STEP',step+1,'TRAIN_CE',losses[-1]['train_cross_entropy'])
    extended.eval()
    with torch.no_grad():
        new_shares=extended(features_device).cpu().numpy().astype(np.float64)
    extension_learned_values=values.copy()
    extension_learned_values[1:-1]=new_shares*total_power[:,None]
    extended_corr=cov_to_corr(model_implied_covariance(vectors,extension_learned_values))
    new_kl=-np.sum(emp_shares*np.log(np.maximum(new_shares,EPS)),axis=-1)-H_target
    new_analytic_err=corr_mae(extended_corr,R_dev if R_dev is not None else R_emp)
    old_analytic_err=corr_mae(R_net,R_dev if R_dev is not None else R_emp)
    extension_summary={
        'status':'completed_fixed_900_step_ablation',
        'optimizer':'fresh_AdamW_not_exact_resume',
        'optimizer_additional_steps':900,
        'initial_training_only_KL_nats':float(np.mean(KL)),
        'final_training_only_KL_nats':float(np.mean(new_kl)),
        'stage33_analytic_corr_mae_vs_reference':old_analytic_err,
        'extension_analytic_corr_mae_vs_reference':new_analytic_err,
        'reference_type':reference_label,
        'training_loss_trace':losses,
        'elapsed_seconds':time.monotonic()-start_fit,
        'scientific_status':'development-analytical ablation only; NOT final independent test',
    }
    extension_state={k:v.detach().cpu() for k,v in extended.state_dict().items()}
    print('=== STAGE34 OPTIMIZATION ABLATION ===')
    print(json.dumps(extension_summary,indent=2))
else:
    print('OPTIONAL_TRAINING_NOT_RUN; Stage34 forensic diagnostics are complete without training.')


STAGE34_ADDITIONAL_STEP 100 TRAIN_CE 2.1781132221221924
STAGE34_ADDITIONAL_STEP 200 TRAIN_CE 2.1700332164764404
STAGE34_ADDITIONAL_STEP 300 TRAIN_CE 2.156536817550659
STAGE34_ADDITIONAL_STEP 400 TRAIN_CE 2.1460182666778564
STAGE34_ADDITIONAL_STEP 500 TRAIN_CE 2.136712074279785
STAGE34_ADDITIONAL_STEP 600 TRAIN_CE 2.1230762004852295
STAGE34_ADDITIONAL_STEP 700 TRAIN_CE 2.11393666267395
STAGE34_ADDITIONAL_STEP 800 TRAIN_CE 2.1086695194244385
STAGE34_ADDITIONAL_STEP 900 TRAIN_CE 2.1052541732788086
=== STAGE34 OPTIMIZATION ABLATION ===
{
  "status": "completed_fixed_900_step_ablation",
  "optimizer": "fresh_AdamW_not_exact_resume",
  "optimizer_additional_steps": 900,
  "initial_training_only_KL_nats": 0.23488093422475403,
  "final_training_only_KL_nats": 0.00793769301661986,
  "stage33_analytic_corr_mae_vs_reference": 0.5166931495191308,
  "extension_analytic_corr_mae_vs_reference": 0.004523219738530946,
  "reference_type": "training_only_empirical_baseline",
  "training_loss_trace": [
  

In [ ]:
# 10 — Save a non-overwriting Stage34 evidence bundle; never modify Stage33 or prior releases.
from datetime import datetime,timezone
stage34_summary={
    'format':'WIND_VISION_STAGE34_EIGENSPECTRUM_FORENSIC_V1',
    'stage33_source_sha256':EXPECTED_STAGE33_SOURCE_SHA,
    'stage33_checkpoint_sha256':EXPECTED_STAGE33_CHECKPOINT_SHA,
    'original_5000_training_sha256':EXPECTED_PARENT_TRAIN_SHA,
    'stage33_report_metrics':{
        'learned':report['development_learned'],
        'baseline':report['development_baseline'],
        'gate':report['development_screening_all_passed'],
    },
    'identity_and_integrity':{
        'parent_train_count':5000,'train_count':4500,'internal_dev_count':500,
        'train_ids_sha256':report['train_indices_sha256'],
        'internal_dev_ids_sha256':report['dev_indices_sha256'],
        'stage33_manifest_verified':True,
        'stage33_preview_replayed_exactly':True,
    },
    'training_only_mode_allocation':mode_allocation_summary,
    'matched_seed_stochastic_replay':stochastic_summary,
    'internal_development_replay':dev_replay_summary,
    'analytic_vs_internal_development':analytic_dev_summary,
    'diagnostic_alpha_ablation':alpha_rows,
    'optional_training_extension':extension_summary,
    'independent_test_accessed':False,
    'classification':'Stage33 learned-eigenspectrum statistical hybrid; NOT original NASA c-LFM latent-flow implementation',
    'note':'Diagnostic reuse of historically examined internal development; not fresh blind validation.',
    'environment':{'python':platform.python_version(),'torch':torch.__version__,
                   'cuda_available':torch.cuda.is_available(),
                   'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
}
if WRITE_NEW_STAGE34_EVIDENCE:
    assert EVIDENCE.is_dir(), 'Drive evidence folder missing; refuse to create elsewhere accidentally'
    tag=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
    STAGE34_OUT=EVIDENCE / ('stage34_eigenspectrum_forensic_'+tag)
    STAGE34_OUT.mkdir(parents=False,exist_ok=False)
    (STAGE34_OUT/'stage34_diagnostic_report.json').write_text(json.dumps(stage34_summary,indent=2,sort_keys=True,allow_nan=False)+'\n')
    np.savez_compressed(STAGE34_OUT/'stage34_training_only_mode_shares.npz',
                        training_target_shares=emp_shares.astype(np.float32),
                        stage33_learned_shares=nn_shares.astype(np.float32),
                        training_only_kl_by_frequency=KL.astype(np.float32),
                        training_only_tv_by_frequency=TV.astype(np.float32))
    if extension_state is not None:
        # Save only the newly trained weights. Parent Stage33 covariance lives in the verified old checkpoint.
        torch.save({'format':'stage34_fixed_900_step_extension',
                    'parent_stage33_checkpoint_sha256':EXPECTED_STAGE33_CHECKPOINT_SHA,
                    'original_train_sha256':EXPECTED_PARENT_TRAIN_SHA,
                    'state_dict':extension_state,'new_eigenvalues':torch.from_numpy(extension_learned_values),
                    'optimizer_reset':True,'optimizer_additional_steps':900},STAGE34_OUT/'stage34_extension.pt')
    files_to_hash=sorted(p for p in STAGE34_OUT.iterdir() if p.is_file())
    hashes={p.name:file_sha256(p) for p in files_to_hash}
    (STAGE34_OUT/'SHA256SUMS.json').write_text(json.dumps(hashes,indent=2,sort_keys=True)+'\n')
    print('STAGE34_EVIDENCE_SAVED_TO:',STAGE34_OUT)
    print('STAGE34_EVIDENCE_SHA256_MANIFEST:',file_sha256(STAGE34_OUT/'SHA256SUMS.json'))
else:
    STAGE34_OUT=None
    print('Evidence saving disabled; results remain in notebook memory only')
print('INDEPENDENT_TEST_ACCESS: NONE')
print('STAGE33_GATE_REMAINS:',report['development_screening_all_passed'])


STAGE34_EVIDENCE_SAVED_TO: /content/drive/MyDrive/CLFM_Research_Results/physicsnemo_proposal_reproduction/evidence/fullscale_physicsnemo_training/stage34_eigenspectrum_forensic_20261110T151857Z_2a78e6bc
STAGE34_EVIDENCE_SHA256_MANIFEST: daf2f85e0bef715c40dcd88eca02ec1a5c4632f3db49a5d50c4740117c27fb98
INDEPENDENT_TEST_ACCESS: NONE
STAGE33_GATE_REMAINS: False


## Stage 34 decision and exact stop rule

After running, read **Cell 5**, **Cell 6**, **Cell 7** (if training HDF5 was available), and **Cell 8**:

- **Large train-only KL / too-flat learned top-mode shares + analytic correlation failure:** the 100-step neural fit is inaccurate; a fixed, saved-weight 900-step fitting ablation is scientifically informative. It does not constitute a new wind model success.
- **Tiny train-only KL but large analytic covariance error:** investigate the numerical mapping to covariance, eigensystem indexing, and mode/sample covariance assumptions before training any model.
- **Analytic correlations are correct but sampled correlations fail:** audit fixed-phase innovation sampling, finite-sample Monte Carlo, Nyquist special case, and the real inverse FFT.
- **Baseline and learned both differ from development:** investigate distribution shift between the 4,500 train fields and 500 internal development fields, finite-sample error and the reference statistic.
- **Any source / checkpoint hash, preview replay, split identity, or development replay mismatch:** **STOP**; do not use the results to justify new research training.

**Do not** access the independent test, tune mixing `alpha` using development and promote that as blind performance, or call a restored PhysicsNeMo-compatible base-class model a full native c-LFM implementation. The separate NASA original VAE + flow training-from-scratch task remains outstanding. Follow-up scientific research should only be selected after these source-backed diagnostics are interpreted.


In [ ]:
# STAGE 34 — Enable controlled training extension
RUN_FIXED_900_STEP_EXTENSION = True

assert mode_allocation_summary[
    "mean_training_only_KL_target_to_learned_nats"
] > 1e-4

print("900-STEP OPTIMIZATION ABLATION ENABLED")
print("Original Stage 33 checkpoint preserved")
print("Independent test remains untouched")

900-STEP OPTIMIZATION ABLATION ENABLED
Original Stage 33 checkpoint preserved
Independent test remains untouched


In [ ]:
# ================================================================
# STAGE 35 — SAVED EXTENSION + DEVELOPMENT VALIDATION
# ================================================================
# Run as a NEW cell after Stage 34 Cell 10.
# No retraining. No independent-test access.
# Previous checkpoints and evidence remain unchanged.

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, uuid
import numpy as np
import torch

STAGE34_SAVED = EVIDENCE / (
    "stage34_eigenspectrum_forensic_20261110T151857Z_2a78e6bc"
)

EXPECTED_MANIFEST_SHA = (
    "daf2f85e0bef715c40dcd88eca02ec1a5c4632f3db49a5d50c4740117c27fb98"
)

# ------------------------------------------------
# 1. Verify saved Stage 34 evidence
# ------------------------------------------------

assert STAGE34_SAVED.is_dir()

manifest_path = STAGE34_SAVED / "SHA256SUMS.json"

assert file_sha256(manifest_path) == EXPECTED_MANIFEST_SHA

saved_manifest = json.loads(manifest_path.read_text())

for filename, expected_hash in saved_manifest.items():
    actual_hash = file_sha256(STAGE34_SAVED / filename)
    assert actual_hash == expected_hash, filename

saved_extension = torch.load(
    STAGE34_SAVED / "stage34_extension.pt",
    map_location="cpu",
    weights_only=True,
)

assert (
    saved_extension["parent_stage33_checkpoint_sha256"]
    == EXPECTED_STAGE33_CHECKPOINT_SHA
)

assert (
    saved_extension["original_train_sha256"]
    == EXPECTED_PARENT_TRAIN_SHA
)

assert saved_extension["optimizer_additional_steps"] == 900

stage34_eigs = (
    saved_extension["new_eigenvalues"]
    .numpy()
    .astype(np.float64)
)

assert stage34_eigs.shape == values.shape
assert np.isfinite(stage34_eigs).all()
assert (stage34_eigs >= 0).all()

assert np.allclose(
    stage34_eigs.sum(-1),
    values.sum(-1),
    rtol=1e-6,
    atol=1e-10,
)

# Restore actual neural weights and verify the saved spectrum.
restored_net = LearnedEigenShares(
    width=ckpt["model_config"]["width"]
).eval()

restored_net.load_state_dict(
    saved_extension["state_dict"],
    strict=True,
)

with torch.no_grad():
    restored_shares = restored_net(
        make_features(127, 100, "cpu")
    ).numpy().astype(np.float64)

assert np.allclose(
    restored_shares * total_power[:, None],
    stage34_eigs[1:-1],
    rtol=3e-6,
    atol=3e-6,
), "Saved model weights and eigenspectrum disagree"

print("STAGE35_SAVED_CHECKPOINT_VERIFICATION: PASS")

# ------------------------------------------------
# 2. Matched-seed independent sample generation
# ------------------------------------------------

GEN_SEED = 331102
N_FIELDS = 256

baseline_fields = generate(
    N_FIELDS, 256,
    profile, mean_cov,
    vectors, values,
    GEN_SEED,
)

stage34_fields = generate(
    N_FIELDS, 256,
    profile, mean_cov,
    vectors, stage34_eigs,
    GEN_SEED,
)

assert baseline_fields.shape == (256, 100, 256)
assert stage34_fields.shape == (256, 100, 256)
assert np.isfinite(stage34_fields).all()

R_stage34_analytic = cov_to_corr(
    model_implied_covariance(vectors, stage34_eigs)
)

R_baseline_sample = empirical_corr(baseline_fields)
R_stage34_sample = empirical_corr(stage34_fields)

paired_results = {
    "fields_per_model": N_FIELDS,
    "seed": GEN_SEED,
    "analytic_stage34_vs_training_baseline": corr_mae(
        R_stage34_analytic, R_emp
    ),
    "sampled_baseline_vs_training_baseline": corr_mae(
        R_baseline_sample, R_emp
    ),
    "sampled_stage34_vs_training_baseline": corr_mae(
        R_stage34_sample, R_emp
    ),
    "sampled_stage34_vs_own_analytic": corr_mae(
        R_stage34_sample, R_stage34_analytic
    ),
    "sampled_stage34_vs_sampled_baseline": corr_mae(
        R_stage34_sample, R_baseline_sample
    ),
}

print("\n=== STAGE 35 PAIRED GENERATION ===")
print(json.dumps(paired_results, indent=2))

# ------------------------------------------------
# 3. Locate the ORIGINAL 5000-field training HDF5
# ------------------------------------------------
# Never substitute the independent test dataset.

candidates = [
    Path(str(TRAIN_DATA_OVERRIDE).strip())
        if str(TRAIN_DATA_OVERRIDE).strip() else None,
    Path(report["configuration"]["input"]),
    Path("/content/wind_train_data.hdf5"),
    Path(
        "/content/nasa_clfm_reference/data/wind/"
        "wind_train_data.hdf5"
    ),
    Path("/content/drive/MyDrive/wind_train_data.hdf5"),
    PROJECT / "data" / "wind_train_data.hdf5",
    PROJECT / "data" / "wind" / "wind_train_data.hdf5",
]

# Search only known research directories.
for root in (
    Path("/content/nasa_clfm_reference"),
    PROJECT.parent,
):
    if root.is_dir():
        candidates.extend(
            root.rglob("wind_train_data.hdf5")
        )

verified_train = None
checked = set()

for candidate in candidates:
    if candidate is None or candidate in checked:
        continue

    checked.add(candidate)

    if candidate.name != "wind_train_data.hdf5":
        continue

    if not candidate.is_file():
        continue

    actual_sha = file_sha256(candidate)

    print("TRAIN_HDF5_CANDIDATE:", candidate)
    print("SHA256:", actual_sha)

    if actual_sha == EXPECTED_PARENT_TRAIN_SHA:
        verified_train = candidate
        break

# ------------------------------------------------
# 4. Evaluate original 500-field development split
# ------------------------------------------------

development_results = {
    "status": "not_evaluated_training_HDF5_unavailable"
}

if verified_train is not None:
    import h5py

    assert ids_sha(dev_ids) == ckpt["dev_ids_sha256"]
    assert ids_sha(train_ids) == ckpt["train_ids_sha256"]

    with h5py.File(verified_train, "r") as h:
        group = h["wind_samples"]

        assert len(group) >= 5000
        assert group["0/v1"].shape == (100, 256)

        development_fields = np.stack([
            group[str(int(i))]["v1"][:]
            for i in dev_ids
        ]).astype(np.float64)

    assert development_fields.shape == (500, 100, 256)
    assert np.isfinite(development_fields).all()

    baseline_metrics = metrics(
        baseline_fields, development_fields
    )

    improved_metrics = metrics(
        stage34_fields, development_fields
    )

    # Verify historical Stage 33 baseline replay
    # before interpreting any newly computed result.

    original_metrics = report["development_baseline"]

    for key in (
        "temporal_variance_ratio",
        "step_rms_ratio",
        "spatial_corr_offdiag_mae",
        "between_field_temporal_energy_cv",
        "mean_profile_rmse_mps",
    ):
        assert np.isclose(
            baseline_metrics[key],
            original_metrics[key],
            rtol=2e-7,
            atol=2e-8,
        ), f"Historical baseline mismatch: {key}"

    for key in ("low", "middle", "high", "tail"):
        assert np.isclose(
            baseline_metrics["band_power_ratios"][key],
            original_metrics["band_power_ratios"][key],
            rtol=2e-7,
            atol=2e-8,
        ), f"Historical baseline band mismatch: {key}"

    print("HISTORICAL_BASELINE_REPLAY: PASS")

    # Preserve original Stage 33 screening thresholds.

    screening = {
        "temporal_variance_within_10pct":
            abs(improved_metrics["temporal_variance_ratio"] - 1) <= 0.10,

        "step_rms_within_10pct":
            abs(improved_metrics["step_rms_ratio"] - 1) <= 0.10,

        "four_bands_within_20pct":
            all(
                abs(v - 1) <= 0.20
                for v in improved_metrics["band_power_ratios"].values()
            ),

        "spatial_corr_mae_le_0p02":
            improved_metrics["spatial_corr_offdiag_mae"] <= 0.02,

        "between_field_energy_cv_le_1e_minus_6":
            improved_metrics["between_field_temporal_energy_cv"] <= 1e-6,
    }

    development_results = {
        "status": "historical_internal_development_evaluated",
        "training_HDF5_hash_verified": True,
        "historical_baseline_replay_verified": True,
        "stage33_original_learned":
            report["development_learned"],
        "empirical_baseline":
            baseline_metrics,
        "stage34_extension":
            improved_metrics,
        "screening":
            screening,
        "screen_all_passed":
            bool(all(screening.values())),
        "independent_test": False,
    }

    print("\n=== STAGE 35 DEVELOPMENT RESULTS ===")
    print(json.dumps(development_results, indent=2))

else:
    print("\nDEVELOPMENT_VALIDATION: NOT RUN")
    print("Original training HDF5 not found.")
    print("Required SHA256:", EXPECTED_PARENT_TRAIN_SHA)
    print("Independent test data will NOT be substituted.")

# ------------------------------------------------
# 5. Save immutable Stage 35 evidence
# ------------------------------------------------

tag = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:8]
)

STAGE35_OUT = EVIDENCE / (
    "stage35_saved_extension_evaluation_" + tag
)

STAGE35_OUT.mkdir(parents=False, exist_ok=False)

stage35_report = {
    "format": "wind_vision_stage35_v1",
    "parent_stage34": str(STAGE34_SAVED),
    "stage34_manifest_sha256": EXPECTED_MANIFEST_SHA,
    "parent_stage33_checkpoint_sha256":
        EXPECTED_STAGE33_CHECKPOINT_SHA,
    "training_corpus_sha256": EXPECTED_PARENT_TRAIN_SHA,
    "paired_generation": paired_results,
    "internal_development": development_results,
    "independent_test_accessed": False,
    "stage33_original_gate": bool(
        report["development_screening_all_passed"]
    ),
    "scientific_scope":
        "Learned spectral-statistical hybrid, not NASA c-LFM.",
}

result_file = STAGE35_OUT / "stage35_report.json"

result_file.write_text(
    json.dumps(
        stage35_report,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    ) + "\n"
)

manifest = {
    result_file.name: file_sha256(result_file)
}

manifest_file = STAGE35_OUT / "SHA256SUMS.json"

manifest_file.write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n"
)

print("\n" + "=" * 70)
print("STAGE 35 COMPLETE")
print("=" * 70)
print("EVIDENCE:", STAGE35_OUT)
print("MANIFEST_SHA256:", file_sha256(manifest_file))
print("DEVELOPMENT_STATUS:", development_results["status"])
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("STAGE33_ORIGINAL_GATE:", report["development_screening_all_passed"])
print("=" * 70)

STAGE35_SAVED_CHECKPOINT_VERIFICATION: PASS

=== STAGE 35 PAIRED GENERATION ===
{
  "fields_per_model": 256,
  "seed": 331102,
  "analytic_stage34_vs_training_baseline": 0.004523219738530946,
  "sampled_baseline_vs_training_baseline": 0.002509268431464805,
  "sampled_stage34_vs_training_baseline": 0.005269331512533739,
  "sampled_stage34_vs_own_analytic": 0.0025010544958824436,
  "sampled_stage34_vs_sampled_baseline": 0.004485680933710681
}

DEVELOPMENT_VALIDATION: NOT RUN
Original training HDF5 not found.
Required SHA256: 3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6
Independent test data will NOT be substituted.

STAGE 35 COMPLETE
EVIDENCE: /content/drive/MyDrive/CLFM_Research_Results/physicsnemo_proposal_reproduction/evidence/fullscale_physicsnemo_training/stage35_saved_extension_evaluation_20261110T155709Z_684d9aa8
MANIFEST_SHA256: 3574ea7f25964e188dad4b93a8df6ec5b7bdfdec5db6500ae0bd5c030a392b39
DEVELOPMENT_STATUS: not_evaluated_training_HDF5_unavailable
INDEPEND

In [ ]:
# ================================================================
# STAGE 36 — DERIVED DEVELOPMENT SCREEN (CORRECTED)
# ================================================================
# Exact ratio identities + conservative spatial-correlation bound.
#
# NO TRAINING
# NO HDF5 ACCESS
# NO INDEPENDENT TEST ACCESS
# NO OVERWRITING PREVIOUS EVIDENCE
#
# Requires completed Stage 34 and Stage 35 notebook cells.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import uuid
import numpy as np

print("=" * 70)
print("WIND VISION — STAGE 36")
print("DERIVED HISTORICAL DEVELOPMENT VALIDATION")
print("=" * 70)

# ================================================================
# 1 — VERIFY ORIGINAL STAGE 33 PROVENANCE
# ================================================================

assert STAGE33.is_dir(), "Stage 33 directory missing"

assert file_sha256(source_file) == EXPECTED_STAGE33_SOURCE_SHA, (
    "Stage 33 source integrity failed"
)

assert file_sha256(checkpoint_path) == EXPECTED_STAGE33_CHECKPOINT_SHA, (
    "Stage 33 checkpoint integrity failed"
)

# FIX: Read the original manifest from disk.
# Never use the global variable 'manifest', since Stage 35
# replaced it with a different evidence manifest.

s36_stage33_manifest_path = STAGE33 / "SHA256SUMS.json"

assert s36_stage33_manifest_path.is_file()

s36_stage33_manifest = json.loads(
    s36_stage33_manifest_path.read_text()
)

assert isinstance(s36_stage33_manifest, dict)

for s36_filename, s36_expected_hash in s36_stage33_manifest.items():

    s36_path = STAGE33 / s36_filename

    assert s36_path.is_file(), (
        f"Missing Stage 33 artifact: {s36_filename}"
    )

    assert file_sha256(s36_path) == s36_expected_hash, (
        f"Stage 33 integrity mismatch: {s36_filename}"
    )

assert report_file.name in s36_stage33_manifest

assert (
    file_sha256(report_file)
    == s36_stage33_manifest[report_file.name]
)

assert report["development_screening_all_passed"] is False

assert report["source_dataset_sha256"] == EXPECTED_PARENT_TRAIN_SHA

assert report["train_indices_sha256"] == ckpt["train_ids_sha256"]

assert report["dev_indices_sha256"] == ckpt["dev_ids_sha256"]

print("STAGE33_SOURCE_SHA256: PASS")
print("STAGE33_CHECKPOINT_SHA256: PASS")
print(
    "STAGE33_MANIFEST_FILES_VERIFIED:",
    len(s36_stage33_manifest)
)
print("STAGE33_ORIGINAL_GATE: False")


# ================================================================
# 2 — VERIFY SAVED STAGE 34 EXTENSION
# ================================================================

assert STAGE34_SAVED.is_dir(), (
    "Saved Stage 34 extension directory missing"
)

s36_stage34_manifest_path = (
    STAGE34_SAVED / "SHA256SUMS.json"
)

assert (
    file_sha256(s36_stage34_manifest_path)
    == EXPECTED_MANIFEST_SHA
), "Stage 34 manifest integrity failed"

s36_stage34_manifest = json.loads(
    s36_stage34_manifest_path.read_text()
)

assert isinstance(s36_stage34_manifest, dict)

for s36_filename, s36_expected_hash in s36_stage34_manifest.items():

    s36_path = STAGE34_SAVED / s36_filename

    assert s36_path.is_file(), (
        f"Missing Stage 34 artifact: {s36_filename}"
    )

    assert file_sha256(s36_path) == s36_expected_hash, (
        f"Stage 34 integrity mismatch: {s36_filename}"
    )

assert "stage34_extension.pt" in s36_stage34_manifest

assert stage34_eigs.shape == values.shape

assert np.isfinite(stage34_eigs).all()

assert np.allclose(
    stage34_eigs.sum(axis=-1),
    values.sum(axis=-1),
    rtol=1e-6,
    atol=1e-10
)

print("STAGE34_EXTENSION_INTEGRITY: PASS")
print(
    "STAGE34_MANIFEST_FILES_VERIFIED:",
    len(s36_stage34_manifest)
)


# ================================================================
# 3 — VERIFY EXACT GENERATOR LINEAGE
# ================================================================

assert report["configuration"]["generated_count"] == 256

assert report["configuration"]["seed"] == 331001

assert GEN_SEED == report["configuration"]["seed"] + 101

assert GEN_SEED == 331102

assert baseline_fields.shape == (256, 100, 256)

assert stage34_fields.shape == (256, 100, 256)

assert np.isfinite(baseline_fields).all()

assert np.isfinite(stage34_fields).all()

# Verify official Stage 33 saved preview.
with np.load(preview_file, allow_pickle=False) as s36_preview:

    s36_expected_baseline_8 = np.array(
        s36_preview["empirical_baseline_fields"]
    )

assert np.array_equal(
    baseline_fields[:8].astype(np.float32),
    s36_expected_baseline_8
), "Original Stage 33 baseline preview mismatch"

# Deterministic replay of all 256 generated fields.
# Uses saved training moments, NOT the original HDF5.

s36_baseline_replay = generate(
    256,
    256,
    profile,
    mean_cov,
    vectors,
    values,
    GEN_SEED
)

s36_extension_replay = generate(
    256,
    256,
    profile,
    mean_cov,
    vectors,
    stage34_eigs,
    GEN_SEED
)

assert np.array_equal(
    baseline_fields,
    s36_baseline_replay
), "Baseline generation replay failed"

assert np.array_equal(
    stage34_fields,
    s36_extension_replay
), "Stage 34 generation replay failed"

del s36_baseline_replay
del s36_extension_replay

print("STAGE36_GENERATION_LINEAGE: PASS")
print("GENERATED_FIELDS_PER_ARM: 256")
print("GENERATION_SEED:", GEN_SEED)


# ================================================================
# 4 — CALCULATE STAGE 34 VERSUS BASELINE METRICS
# ================================================================

# These are paired generated cohorts using identical seeds.
# No original development HDF5 is needed.

s36_improved_vs_baseline = metrics(
    stage34_fields,
    baseline_fields
)

s36_historical_baseline = report["development_baseline"]

# Independently verify the spatial correlation result
# already reported in Stage 35.

s36_paired_corr_mae = (
    s36_improved_vs_baseline["spatial_corr_offdiag_mae"]
)

assert np.isclose(
    s36_paired_corr_mae,
    paired_results["sampled_stage34_vs_sampled_baseline"],
    rtol=1e-10,
    atol=1e-12
), "Stage 35 spatial correlation replay mismatch"

print("STAGE36_PAIRED_METRICS: PASS")


# ================================================================
# 5 — DERIVE HISTORICAL DEVELOPMENT RATIOS
# ================================================================

# Mathematical identity for variance and power ratios:
#
#   Stage34 / Dev
#       = (Stage34 / Baseline) * (Baseline / Dev)
#
# For step RMS, the same identity holds because the
# metric is a ratio of root-mean-square differences.
#
# Historical Baseline / Dev ratios come from the
# hash-verified original Stage 33 report.

s36_derived = {

    "temporal_variance_ratio": float(
        s36_improved_vs_baseline["temporal_variance_ratio"]
        * s36_historical_baseline["temporal_variance_ratio"]
    ),

    "step_rms_ratio": float(
        s36_improved_vs_baseline["step_rms_ratio"]
        * s36_historical_baseline["step_rms_ratio"]
    ),

    "band_power_ratios": {

        s36_band: float(
            s36_improved_vs_baseline["band_power_ratios"][s36_band]
            * s36_historical_baseline["band_power_ratios"][s36_band]
        )

        for s36_band in (
            "low",
            "middle",
            "high",
            "tail"
        )
    },

    # This metric depends only on the generated cohort.
    # It is not a generated/reference ratio.

    "between_field_temporal_energy_cv": float(
        s36_improved_vs_baseline[
            "between_field_temporal_energy_cv"
        ]
    )
}

assert all(
    np.isfinite(s36_value)
    for s36_value in (
        s36_derived["temporal_variance_ratio"],
        s36_derived["step_rms_ratio"],
        s36_derived["between_field_temporal_energy_cv"],
        *s36_derived["band_power_ratios"].values()
    )
)

print("\n=== STAGE 36 DERIVED DEVELOPMENT METRICS ===")
print(json.dumps(s36_derived, indent=2))


# ================================================================
# 6 — RIGOROUS SPATIAL CORRELATION BOUNDS
# ================================================================

# Correlation-matrix MAE is a metric on the off-diagonal
# correlation vectors.
#
# Triangle inequality:
#
#   MAE(Stage34, Dev)
#       <= MAE(Stage34, Baseline)
#        + MAE(Baseline, Dev)
#
# Reverse triangle inequality:
#
#   MAE(Stage34, Dev)
#       >= |MAE(Stage34, Baseline)
#          - MAE(Baseline, Dev)|
#
# This gives bounds without reloading raw development data.

s36_model_vs_baseline_mae = float(
    s36_improved_vs_baseline[
        "spatial_corr_offdiag_mae"
    ]
)

s36_baseline_vs_dev_mae = float(
    s36_historical_baseline[
        "spatial_corr_offdiag_mae"
    ]
)

s36_spatial_lower = float(
    abs(
        s36_model_vs_baseline_mae
        - s36_baseline_vs_dev_mae
    )
)

s36_spatial_upper = float(
    s36_model_vs_baseline_mae
    + s36_baseline_vs_dev_mae
)

assert s36_spatial_lower >= 0.0
assert s36_spatial_upper >= s36_spatial_lower

print("\n=== SPATIAL CORRELATION DEVELOPMENT BOUNDS ===")
print("STAGE34_VS_BASELINE_MAE:", s36_model_vs_baseline_mae)
print("BASELINE_VS_DEVELOPMENT_MAE:", s36_baseline_vs_dev_mae)
print("DERIVED_DEVELOPMENT_LOWER:", s36_spatial_lower)
print("DERIVED_DEVELOPMENT_UPPER:", s36_spatial_upper)
print("SPATIAL_ACCEPTANCE_THRESHOLD:", 0.02)


# ================================================================
# 7 — OPTIONAL MEAN PROFILE RMSE BOUNDS
# ================================================================

# Mean-profile RMSE is not one of Stage 33's five gates,
# but we preserve a conservative bound for completeness.

s36_profile_model_baseline = float(
    s36_improved_vs_baseline[
        "mean_profile_rmse_mps"
    ]
)

s36_profile_baseline_dev = float(
    s36_historical_baseline[
        "mean_profile_rmse_mps"
    ]
)

s36_profile_bounds = {

    "lower_mps": float(
        max(
            0.0,
            s36_profile_baseline_dev
            - s36_profile_model_baseline
        )
    ),

    "upper_mps": float(
        s36_profile_baseline_dev
        + s36_profile_model_baseline
    )
}

print("\nMEAN_PROFILE_RMSE_BOUNDS:")
print(json.dumps(s36_profile_bounds, indent=2))


# ================================================================
# 8 — APPLY UNCHANGED STAGE 33 DEVELOPMENT GATES
# ================================================================

s36_screen = {

    "temporal_variance_within_10pct": bool(
        abs(
            s36_derived["temporal_variance_ratio"] - 1.0
        ) <= 0.10
    ),

    "adjacent_step_within_10pct": bool(
        abs(
            s36_derived["step_rms_ratio"] - 1.0
        ) <= 0.10
    ),

    "four_bands_within_20pct": bool(
        all(
            abs(s36_value - 1.0) <= 0.20
            for s36_value in
            s36_derived["band_power_ratios"].values()
        )
    ),

    # Conservative: entire provable error interval
    # must be below the original threshold.

    "spatial_correlation_mae_le_0p02": bool(
        s36_spatial_upper <= 0.02
    ),

    "between_field_energy_cv_le_1e_minus_6": bool(
        s36_derived[
            "between_field_temporal_energy_cv"
        ] <= 1e-6
    )
}

s36_screen_passed = bool(
    all(s36_screen.values())
)

print("\n" + "=" * 70)
print("STAGE 36 — FIVE-PART DEVELOPMENT SCREEN")
print("=" * 70)

for s36_name, s36_passed in s36_screen.items():

    print(
        f"{s36_name}: "
        f"{'PASS' if s36_passed else 'FAIL'}"
    )

print()
print("SCREEN_CERTIFIED_BY_BOUND:", s36_screen_passed)
print("ORIGINAL_STAGE33_GATE:", report["development_screening_all_passed"])
print("DIRECT_DEVELOPMENT_HDF5_ACCESSED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")


# ================================================================
# 9 — BUILD SCIENTIFIC EVIDENCE RECORD
# ================================================================

s36_outcome = {

    "format": (
        "wind_vision_stage36_"
        "derived_development_screen_v2"
    ),

    "method": (
        "exact_ratio_transfer_and_spatial_"
        "correlation_MAE_triangle_inequality"
    ),

    "scientific_scope": (
        "Training-derived spectral-statistical hybrid; "
        "not the original NASA c-LFM latent flow."
    ),

    "evaluation_scope": (
        "Derived historical internal development comparison; "
        "not new blind validation."
    ),

    "direct_development_hdf5_accessed": False,

    "independent_test_accessed": False,

    "stage33_source_sha256": (
        EXPECTED_STAGE33_SOURCE_SHA
    ),

    "stage33_checkpoint_sha256": (
        EXPECTED_STAGE33_CHECKPOINT_SHA
    ),

    "stage34_extension_manifest_sha256": (
        EXPECTED_MANIFEST_SHA
    ),

    "generation_seed": int(GEN_SEED),

    "generated_fields_per_arm": 256,

    "baseline_preview_verified": True,

    "full_generation_deterministic_replay": True,

    "historical_baseline_development_metrics": (
        s36_historical_baseline
    ),

    "stage34_vs_baseline_metrics": (
        s36_improved_vs_baseline
    ),

    "derived_stage34_development_metrics": (
        s36_derived
    ),

    "spatial_corr_mae_development_bounds": {

        "lower": s36_spatial_lower,

        "upper": s36_spatial_upper,

        "threshold": 0.02
    },

    "mean_profile_rmse_bounds_mps": (
        s36_profile_bounds
    ),

    "five_part_screen": (
        s36_screen
    ),

    "historical_screen_certified_by_bound": (
        s36_screen_passed
    ),

    "original_stage33_gate": bool(
        report["development_screening_all_passed"]
    ),

    "limitations": [

        "No original development HDF5 was read.",

        "Historical development metrics were taken from "
        "the verified original Stage33 report.",

        "Spatial development error is bounded rather "
        "than measured directly.",

        "This is not an independent test evaluation.",

        "Passing the historical statistical screen "
        "does not prove physical wind realism.",

        "Near-zero between-field energy CV is a "
        "fixed-energy shell artifact, not learned "
        "atmospheric variability.",

        "The model learns eigenvalue allocation using "
        "fixed training-derived eigenvectors and "
        "externally sampled phases."
    ]
}


# ================================================================
# 10 — SAVE NON-OVERWRITING STAGE 36 EVIDENCE
# ================================================================

assert EVIDENCE.is_dir(), (
    "Expected evidence directory does not exist"
)

s36_tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

s36_out = EVIDENCE / (
    "stage36_historical_dev_screen_" + s36_tag
)

s36_out.mkdir(
    parents=False,
    exist_ok=False
)

s36_result_path = (
    s36_out / "stage36_derived_development_screen.json"
)

s36_result_path.write_text(
    json.dumps(
        s36_outcome,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
)

s36_new_manifest = {

    s36_result_path.name: file_sha256(
        s36_result_path
    )
}

s36_new_manifest_path = (
    s36_out / "SHA256SUMS.json"
)

s36_new_manifest_path.write_text(
    json.dumps(
        s36_new_manifest,
        indent=2,
        sort_keys=True
    ) + "\n"
)

# Verify newly written evidence

assert (
    file_sha256(s36_result_path)
    == s36_new_manifest[s36_result_path.name]
)

s36_manifest_sha256 = file_sha256(
    s36_new_manifest_path
)

print("\n" + "=" * 70)
print("STAGE 36 — COMPLETE")
print("=" * 70)

print("STAGE36_EVIDENCE:", s36_out)

print(
    "STAGE36_MANIFEST_SHA256:",
    s36_manifest_sha256
)

print(
    "SCREEN_CERTIFIED_BY_BOUND:",
    s36_screen_passed
)

print(
    "STAGE33_ORIGINAL_GATE:",
    report["development_screening_all_passed"]
)

print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("=" * 70)

WIND VISION — STAGE 36
DERIVED HISTORICAL DEVELOPMENT VALIDATION
STAGE33_SOURCE_SHA256: PASS
STAGE33_CHECKPOINT_SHA256: PASS
STAGE33_MANIFEST_FILES_VERIFIED: 4
STAGE33_ORIGINAL_GATE: False
STAGE34_EXTENSION_INTEGRITY: PASS
STAGE34_MANIFEST_FILES_VERIFIED: 3
STAGE36_GENERATION_LINEAGE: PASS
GENERATED_FIELDS_PER_ARM: 256
GENERATION_SEED: 331102
STAGE36_PAIRED_METRICS: PASS

=== STAGE 36 DERIVED DEVELOPMENT METRICS ===
{
  "temporal_variance_ratio": 0.999999994708725,
  "step_rms_ratio": 0.9999465991666956,
  "band_power_ratios": {
    "low": 0.9976441217853165,
    "middle": 0.9921282825067361,
    "high": 1.0101758708447175,
    "tail": 0.9937558274457955
  },
  "between_field_temporal_energy_cv": 2.5833814026999673e-16
}

=== SPATIAL CORRELATION DEVELOPMENT BOUNDS ===
STAGE34_VS_BASELINE_MAE: 0.004485680933710681
BASELINE_VS_DEVELOPMENT_MAE: 0.0026818699609928854
DERIVED_DEVELOPMENT_LOWER: 0.001803810972717796
DERIVED_DEVELOPMENT_UPPER: 0.007167550894703566
SPATIAL_ACCEPTANCE_THRESHOLD

In [ ]:

# ================================================================
# STAGE 37 — FOUR-ARM SPECTRAL-INNOVATION ABLATION
# ================================================================
# Requires completed Stage 36.
# No training, HDF5 reads, or independent-test access.
# Original model, checkpoints, and gates remain unchanged.

from pathlib import Path
from datetime import datetime, timezone
import json, uuid
import numpy as np

# 1. Verify research lineage
assert s36_screen_passed is True
assert file_sha256(checkpoint_path) == EXPECTED_STAGE33_CHECKPOINT_SHA
assert file_sha256(
    STAGE34_SAVED / "SHA256SUMS.json"
) == EXPECTED_MANIFEST_SHA
assert file_sha256(
    s36_out / "SHA256SUMS.json"
) == s36_manifest_sha256

assert np.allclose(
    stage34_eigs.sum(-1),
    values.sum(-1),
    rtol=1e-6,
    atol=1e-10
)

S37_SEEDS = (371101, 371102, 371103, 371104)
S37_FIELDS_PER_SEED = 256
S37_BATCH_SIZE = 64
S37_TIME_POINTS = 256

# Training-only analytical reference covariances
C_train = model_implied_covariance(
    vectors, values, S37_TIME_POINTS
)
C_learned = model_implied_covariance(
    vectors, stage34_eigs, S37_TIME_POINTS
)

R_train = cov_to_corr(C_train)
R_learned = cov_to_corr(C_learned)

expected_avg_temporal_variance = float(
    np.trace(C_train) / 100.0
)
assert expected_avg_temporal_variance > 0


# 2. Controlled four-arm generator
#
# z = (X + iY)/sqrt(2), with X,Y ~ N(0,1).
#
# Fixed shell uses z/|z|.
# Gaussian uses z.
#
# Both have E[|innovation|^2] = 1.
# The same underlying random draws are used for all arms.

def s37_four_arm_sample(seed, n_fields=S37_FIELDS_PER_SEED):
    rng = np.random.default_rng(seed)

    nfreq, nsensors, _ = vectors.shape
    assert nfreq == S37_TIME_POINTS // 2 + 1
    assert nsensors == 100

    dc_val, dc_vec = np.linalg.eigh(mean_cov)
    dc_factor = dc_vec @ np.diag(
        np.sqrt(np.clip(dc_val, 0, None))
    )

    nyq_basis = vectors[-1].real

    specs = {
        "empirical": values,
        "learned": stage34_eigs,
    }

    out = {
        f"{law}_{name}": []
        for law in ("shell", "gaussian")
        for name in specs
    }

    for offset in range(0, n_fields, S37_BATCH_SIZE):
        b = min(S37_BATCH_SIZE, n_fields - offset)

        dc_means = (
            profile
            + rng.standard_normal((b, nsensors)) @ dc_factor.T
        )

        z = (
            rng.standard_normal((b, nfreq - 2, nsensors))
            + 1j * rng.standard_normal((b, nfreq - 2, nsensors))
        ) / np.sqrt(2.0)

        phase = z / np.maximum(np.abs(z), 1e-300)

        nyq_gaussian = rng.standard_normal((b, nsensors))
        nyq_shell = np.where(
            nyq_gaussian >= 0, 1.0, -1.0
        )

        for name, spectrum in specs.items():
            root_interior = np.sqrt(
                np.maximum(spectrum[1:-1], 0)
            )[None, :, :]

            root_nyquist = np.sqrt(
                np.maximum(spectrum[-1], 0)
            )[None, :]

            for law, interior_noise, nyquist_noise in (
                ("shell", phase, nyq_shell),
                ("gaussian", z, nyq_gaussian),
            ):
                coeffs = np.zeros(
                    (b, nsensors, nfreq),
                    dtype=np.complex128
                )

                coeffs[:, :, 0] = (
                    np.sqrt(S37_TIME_POINTS) * dc_means
                )

                coeffs[:, :, 1:-1] = np.einsum(
                    "fsk,bfk->bsf",
                    vectors[1:-1],
                    interior_noise * root_interior,
                    optimize=True
                )

                coeffs[:, :, -1] = (
                    nyquist_noise * root_nyquist
                ) @ nyq_basis.T

                result = np.fft.irfft(
                    coeffs,
                    n=S37_TIME_POINTS,
                    axis=-1,
                    norm="ortho"
                )

                assert result.shape == (
                    b, 100, S37_TIME_POINTS
                )
                assert np.isfinite(result).all()

                out[f"{law}_{name}"].append(result)

    return {
        k: np.concatenate(v, axis=0)
        for k, v in out.items()
    }


# 3. Run prespecified seeds and measure the four arms

s37_records = []

s37_energy_distributions = {
    k: [] for k in (
        "shell_empirical",
        "shell_learned",
        "gaussian_empirical",
        "gaussian_learned"
    )
}

for s37_seed in S37_SEEDS:
    arms = s37_four_arm_sample(s37_seed)
    reference_arm = arms["shell_empirical"]

    for label, fields in arms.items():
        centered = fields - fields.mean(
            axis=-1, keepdims=True
        )

        per_field_energy = np.mean(
            centered**2, axis=(1, 2)
        )

        s37_energy_distributions[label].extend(
            per_field_energy.tolist()
        )

        sampled_corr = empirical_corr(fields)

        own_ref = (
            R_train if "empirical" in label
            else R_learned
        )

        control_metrics = metrics(
            fields, reference_arm
        )

        record = {
            "seed": int(s37_seed),
            "arm": label,
            "n_fields": int(fields.shape[0]),

            "between_field_temporal_energy_cv": float(
                per_field_energy.std()
                / per_field_energy.mean()
            ),

            "temporal_energy_mean_over_training_implied": float(
                per_field_energy.mean()
                / expected_avg_temporal_variance
            ),

            "spatial_corr_mae_vs_own_analytic": corr_mae(
                sampled_corr, own_ref
            ),

            "spatial_corr_mae_vs_train_empirical_analytic": corr_mae(
                sampled_corr, R_train
            ),

            "temporal_variance_ratio_vs_shell_empirical": float(
                control_metrics["temporal_variance_ratio"]
            ),

            "step_rms_ratio_vs_shell_empirical": float(
                control_metrics["step_rms_ratio"]
            ),

            "band_power_ratios_vs_shell_empirical":
                control_metrics["band_power_ratios"],
        }

        numeric_keys = (
            "between_field_temporal_energy_cv",
            "temporal_energy_mean_over_training_implied",
            "spatial_corr_mae_vs_own_analytic",
            "spatial_corr_mae_vs_train_empirical_analytic",
        )

        assert np.isfinite([
            record[k] for k in numeric_keys
        ]).all()

        s37_records.append(record)

    print("STAGE37_SEED_COMPLETE:", s37_seed)
    del arms


# 4. Aggregate multi-seed results

s37_summary = {}

for arm in s37_energy_distributions:
    subset = [
        row for row in s37_records
        if row["arm"] == arm
    ]

    s37_summary[arm] = {
        "seeds": len(subset),

        "fields_total": len(
            s37_energy_distributions[arm]
        ),

        "mean_energy_cv_by_seed": float(np.mean([
            x["between_field_temporal_energy_cv"]
            for x in subset
        ])),

        "max_energy_cv_by_seed": float(np.max([
            x["between_field_temporal_energy_cv"]
            for x in subset
        ])),

        "mean_spatial_corr_mae_vs_train_empirical_analytic":
            float(np.mean([
                x["spatial_corr_mae_vs_train_empirical_analytic"]
                for x in subset
            ])),

        "max_spatial_corr_mae_vs_train_empirical_analytic":
            float(np.max([
                x["spatial_corr_mae_vs_train_empirical_analytic"]
                for x in subset
            ])),

        "mean_energy_ratio_vs_training_implied":
            float(np.mean([
                x["temporal_energy_mean_over_training_implied"]
                for x in subset
            ])),
    }


# 5. Report structural outcome

s37_fixed_shell_invariance = all(
    s37_summary[k]["max_energy_cv_by_seed"] < 1e-8
    for k in ("shell_empirical", "shell_learned")
)

s37_gaussian_variability = all(
    s37_summary[k]["mean_energy_cv_by_seed"] > 1e-3
    for k in ("gaussian_empirical", "gaussian_learned")
)

print("\n=== STAGE 37 — FOUR-ARM INNOVATION ABLATION ===")
print(json.dumps(s37_summary, indent=2))

print(
    "FIXED_SHELL_ENERGY_INVARIANCE_CONFIRMED:",
    s37_fixed_shell_invariance
)

print(
    "GAUSSIAN_INNOVATIONS_REINTRODUCE_VARIABILITY:",
    s37_gaussian_variability
)

print("INDEPENDENT_TEST_ACCESSED: FALSE")


# 6. Preserve evidence without overwriting earlier work

s37_report = {
    "format": "wind_vision_stage37_innovation_ablation_v1",

    "purpose":
        "Isolate deterministic fixed-shell energy from "
        "spectral allocation effects",

    "input_checkpoints": {
        "stage33_sha256": EXPECTED_STAGE33_CHECKPOINT_SHA,
        "stage34_manifest_sha256": EXPECTED_MANIFEST_SHA,
        "stage36_manifest_sha256": s36_manifest_sha256,
    },

    "experimental_design": {
        "innovation_laws": [
            "unit_modulus_phase",
            "unit_variance_complex_gaussian",
        ],
        "eigenspectra": [
            "train_only_empirical",
            "stage34_learned",
        ],
        "shared_underlying_noise_within_seed": True,
        "seeds": list(S37_SEEDS),
        "fields_per_seed_per_arm": S37_FIELDS_PER_SEED,
        "original_stage33_generator_unchanged": True,
    },

    "summary": s37_summary,
    "per_seed_results": s37_records,

    "fixed_shell_invariance_confirmed":
        s37_fixed_shell_invariance,

    "gaussian_reintroduced_variability":
        s37_gaussian_variability,

    "development_hdf5_accessed": False,
    "independent_test_accessed": False,

    "limitations": (
        "All reference statistics here come from training-only "
        "moments. Gaussian coefficient sampling has the same "
        "analytical second-order covariance in expectation, "
        "but is neither a trained novel method nor demonstrated "
        "realistic wind-field energy variability. This is an "
        "ablation, not a replacement for the Stage 36 screen."
    ),
}

s37_tag = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)

s37_out = EVIDENCE / (
    "stage37_innovation_law_ablation_" + s37_tag
)

s37_out.mkdir(parents=False, exist_ok=False)

s37_report_path = s37_out / "stage37_report.json"

s37_report_path.write_text(
    json.dumps(
        s37_report,
        indent=2,
        sort_keys=True,
        allow_nan=False
    ) + "\n"
)

s37_manifest = {
    s37_report_path.name: file_sha256(s37_report_path)
}

s37_manifest_path = s37_out / "SHA256SUMS.json"

s37_manifest_path.write_text(
    json.dumps(
        s37_manifest,
        indent=2,
        sort_keys=True
    ) + "\n"
)

print("\nSTAGE37_EVIDENCE:", s37_out)
print(
    "STAGE37_MANIFEST_SHA256:",
    file_sha256(s37_manifest_path)
)
print(
    "STAGE36_DERIVED_SCREEN_UNCHANGED:",
    s36_screen_passed
)
print("STAGE37_COMPLETE")


STAGE37_SEED_COMPLETE: 371101
STAGE37_SEED_COMPLETE: 371102
STAGE37_SEED_COMPLETE: 371103
STAGE37_SEED_COMPLETE: 371104

=== STAGE 37 — FOUR-ARM INNOVATION ABLATION ===
{
  "shell_empirical": {
    "seeds": 4,
    "fields_total": 1024,
    "mean_energy_cv_by_seed": 2.2683093371396903e-16,
    "max_energy_cv_by_seed": 2.283970595696204e-16,
    "mean_spatial_corr_mae_vs_train_empirical_analytic": 0.0026660808817295725,
    "max_spatial_corr_mae_vs_train_empirical_analytic": 0.0031484437790649604,
    "mean_energy_ratio_vs_training_implied": 0.9999999999999999
  },
  "shell_learned": {
    "seeds": 4,
    "fields_total": 1024,
    "mean_energy_cv_by_seed": 2.971694529603884e-16,
    "max_energy_cv_by_seed": 3.0698725300017217e-16,
    "mean_spatial_corr_mae_vs_train_empirical_analytic": 0.005200417124788956,
    "max_spatial_corr_mae_vs_train_empirical_analytic": 0.005399373419247184,
    "mean_energy_ratio_vs_training_implied": 0.9999999947087259
  },
  "gaussian_empirical": {
    "seed

In [ ]:

# ================================================================
# STAGE 39 — REFERENCE-MATCHED ENERGY DISTRIBUTION AUDIT
# ================================================================
# No training, no HDF5 access, no independent test.
# Reuses the verified Stage 37 four-arm generator.
# Preserves all earlier evidence unchanged.

import json, uuid
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

assert s36_screen_passed
assert file_sha256(checkpoint_path) == EXPECTED_STAGE33_CHECKPOINT_SHA
assert file_sha256(s37_out / "SHA256SUMS.json") == (
    "cf0ee0285f64399ed281512c37d35f81b526a3eb16dea69a608c50ad2d82033d"
)

# Exact statistics from the completed Stage 38 Mac audit.
# These are transcribed reference summaries, not raw data.

S38_REFERENCE = {
    "source_sha256":
        "3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6",
    "train_ids_sha256":
        "f0b44c3c14a27b3d2bae90cf6ef8867bbaeca255c80803290f4954206fb3e33c",
    "dev_ids_sha256":
        "4de9038ec463633bf03a8c08b64ea6a155d54816818339a6647e6b39ac0ba6d7",
    "training": {
        "total":  {"mean": 3.2106091670893204, "cv": 8.017748301253513e-17},
        "low":    {"mean": 5299.1570893018825, "cv": 0.3464465714736885},
        "middle": {"mean": 394.68092793159934, "cv": 0.2260402940227991},
        "high":   {"mean": 16.48637577584998, "cv": 0.11311287628764687},
        "tail":   {"mean": 0.22229419608779363, "cv": 0.052986177333171694},
    },
    "development": {
        "total":  {"mean": 3.210609167089321, "cv": 1.7164973614466438e-16},
        "low":    {"mean": 5265.24277904571, "cv": 0.3547787788810269},
        "middle": {"mean": 397.8494116668855, "cv": 0.23056212300244802},
        "high":   {"mean": 16.40310299190703, "cv": 0.10918078681184742},
        "tail":   {"mean": 0.2232371940375639, "cv": 0.054321777130919256},
    },
}

assert S38_REFERENCE["source_sha256"] == EXPECTED_PARENT_TRAIN_SHA

s39_split_verified = (
    report["train_indices_sha256"] == S38_REFERENCE["train_ids_sha256"]
    and report["dev_indices_sha256"] == S38_REFERENCE["dev_ids_sha256"]
)

print("STAGE38_SPLIT_HASHES_MATCH_STAGE33:", s39_split_verified)

if not s39_split_verified:
    print("WARNING: Stage 38 split IDs do not match Stage 33.")
    print("Reference distributions may describe different partitions.")
    print("Results will be exploratory, not development validation.")
    print("STAGE33_TRAIN_HASH:", report["train_indices_sha256"])
    print("STAGE33_DEV_HASH:", report["dev_indices_sha256"])

S39_BANDS = {
    "low": (1, 7),
    "middle": (7, 17),
    "high": (17, 65),
    "tail": (65, 129),
}

S39_WINDOW = np.hanning(256)
S39_ARMS = (
    "shell_empirical",
    "shell_learned",
    "gaussian_empirical",
    "gaussian_learned",
)

s39_samples = {
    arm: {metric: [] for metric in ("total", *S39_BANDS)}
    for arm in S39_ARMS
}

# Replay the four frozen Stage 37 seeds.
for seed in S37_SEEDS:
    fields_by_arm = s37_four_arm_sample(seed)

    for arm, fields in fields_by_arm.items():
        assert arm in s39_samples
        assert fields.shape == (256, 100, 256)
        assert np.isfinite(fields).all()

        anomalies = fields - fields.mean(axis=-1, keepdims=True)

        # Match original Stage 38 total-energy definition.
        total_energy = np.mean(
            anomalies ** 2, axis=(1, 2)
        )

        s39_samples[arm]["total"].append(total_energy)

        # Match original Stage 38 Hann-periodogram definition.
        power = np.abs(
            np.fft.rfft(
                anomalies * S39_WINDOW,
                axis=-1
            )
        ) ** 2

        for band, (lo, hi) in S39_BANDS.items():
            energy = power[:, :, lo:hi].mean(axis=(1, 2))
            s39_samples[arm][band].append(energy)

    del fields_by_arm
    print("STAGE39_SEED_COMPLETE:", seed)


def s39_describe(chunks):
    x = np.concatenate(chunks).astype(np.float64)
    assert len(x) == 1024
    assert np.isfinite(x).all()
    assert np.all(x >= 0)
    mean = float(x.mean())
    assert mean > 0

    return {
        "n": len(x),
        "mean": mean,
        "cv": float(x.std(ddof=0) / mean),
        "q05": float(np.quantile(x, 0.05)),
        "median": float(np.median(x)),
        "q95": float(np.quantile(x, 0.95)),
    }


s39_results = {}

for arm in S39_ARMS:
    stats = {
        metric: s39_describe(chunks)
        for metric, chunks in s39_samples[arm].items()
    }

    # Compare against training statistics only for model diagnosis.
    train = S38_REFERENCE["training"]

    band_cv_relative_errors = {}
    band_mean_relative_errors = {}

    for band in S39_BANDS:
        band_cv_relative_errors[band] = float(
            stats[band]["cv"] / train[band]["cv"] - 1
        )
        band_mean_relative_errors[band] = float(
            stats[band]["mean"] / train[band]["mean"] - 1
        )

    s39_results[arm] = {
        "statistics": stats,
        "band_cv_relative_errors_vs_training":
            band_cv_relative_errors,
        "band_mean_relative_errors_vs_training":
            band_mean_relative_errors,
        "mean_absolute_band_cv_relative_error": float(
            np.mean(np.abs(list(
                band_cv_relative_errors.values()
            )))
        ),
        "mean_absolute_band_power_relative_error": float(
            np.mean(np.abs(list(
                band_mean_relative_errors.values()
            )))
        ),
    }


# Compact summary to paste back into chat.
print("\n=== STAGE 39 REFERENCE-MATCHED DISTRIBUTIONS ===")
print("REFERENCE_TRAIN_TOTAL_CV:",
      S38_REFERENCE["training"]["total"]["cv"])

for arm, outcome in s39_results.items():
    stats = outcome["statistics"]

    print("\nARM:", arm)
    print("TOTAL_ENERGY_CV:", stats["total"]["cv"])
    print(
        "TOTAL_MEAN_RATIO:",
        stats["total"]["mean"]
        / S38_REFERENCE["training"]["total"]["mean"]
    )
    print("BAND_CVS:", json.dumps({
        band: stats[band]["cv"] for band in S39_BANDS
    }))
    print("BAND_MEAN_RATIOS:", json.dumps({
        band: (
            stats[band]["mean"]
            / S38_REFERENCE["training"][band]["mean"]
        ) for band in S39_BANDS
    }))
    print(
        "MEAN_ABSOLUTE_BAND_CV_RELATIVE_ERROR:",
        outcome["mean_absolute_band_cv_relative_error"]
    )

# Preserve an immutable evidence record.
s39_report = {
    "format": "wind_vision_stage39_reference_band_audit_v1",
    "source": "Stage38 Mac terminal audit summary, transcribed",
    "reference": S38_REFERENCE,
    "stage38_split_verified_against_stage33":
        s39_split_verified,
    "stage37_manifest_sha256":
        "cf0ee0285f64399ed281512c37d35f81b526a3eb16dea69a608c50ad2d82033d",
    "generation_seeds": list(S37_SEEDS),
    "fields_per_arm": 1024,
    "hann_window": "np.hanning(256)",
    "frequency_bands": {
        k: list(v) for k, v in S39_BANDS.items()
    },
    "results": s39_results,
    "development_data_used_for_model_selection": False,
    "raw_development_hdf5_accessed": False,
    "independent_test_accessed": False,
    "status": (
        "reference_matched_exploratory_ablation"
        if s39_split_verified
        else "reference_partition_unverified"
    ),
    "limitations": (
        "Reference values were transcribed from Stage38 stdout. "
        "This audit compares simulator statistics and generated "
        "samples; it is not independent-test validation or "
        "evidence of operational atmospheric realism."
    ),
}

s39_tag = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)

s39_out = EVIDENCE / (
    "stage39_reference_band_audit_" + s39_tag
)
s39_out.mkdir(parents=False, exist_ok=False)

s39_result_path = s39_out / "stage39_report.json"
s39_result_path.write_text(
    json.dumps(
        s39_report, indent=2, sort_keys=True, allow_nan=False
    ) + "\n"
)

s39_manifest_path = s39_out / "SHA256SUMS.json"
s39_manifest_path.write_text(
    json.dumps({
        s39_result_path.name: file_sha256(s39_result_path)
    }, indent=2, sort_keys=True) + "\n"
)

print("\nSTAGE39_EVIDENCE:", s39_out)
print("STAGE39_MANIFEST_SHA256:",
      file_sha256(s39_manifest_path))
print("STAGE39_SPLIT_VERIFIED:", s39_split_verified)
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("STAGE39_COMPLETE")


STAGE38_SPLIT_HASHES_MATCH_STAGE33: True
STAGE39_SEED_COMPLETE: 371101
STAGE39_SEED_COMPLETE: 371102
STAGE39_SEED_COMPLETE: 371103
STAGE39_SEED_COMPLETE: 371104

=== STAGE 39 REFERENCE-MATCHED DISTRIBUTIONS ===
REFERENCE_TRAIN_TOTAL_CV: 8.017748301253513e-17

ARM: shell_empirical
TOTAL_ENERGY_CV: 2.2683745673360247e-16
TOTAL_MEAN_RATIO: 0.999999999999999
BAND_CVS: {"low": 0.3536473730141232, "middle": 0.22396337661462326, "high": 0.11091831125886072, "tail": 0.053585205356947836}
BAND_MEAN_RATIOS: {"low": 0.9867178825565937, "middle": 1.005308181282472, "high": 1.001121952368244, "tail": 1.0002447451931158}
MEAN_ABSOLUTE_BAND_CV_RELATIVE_ERROR: 0.015169978489416841

ARM: shell_learned
TOTAL_ENERGY_CV: 2.972469867211984e-16
TOTAL_MEAN_RATIO: 0.9999999947087251
BAND_CVS: {"low": 0.3520204386865972, "middle": 0.22458626542748011, "high": 0.1105734696538057, "tail": 0.053513095145819804}
BAND_MEAN_RATIOS: {"low": 0.9866964116402014, "middle": 1.0054488402218638, "high": 1.0012584915899372,

In [ ]:

# ================================================================
# STAGE 40 — ORIGINAL NASA c-LFM / PHYSICSNEMO RE-ENTRY
# ================================================================
# Read-only source/checkpoint inventory.
# No training, installs, extraction, or test evaluation.
# No previous research artifacts modified.

from pathlib import Path
import ast
import hashlib
import json
import zipfile

ROOT40 = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

RELEASE40 = (
    ROOT40
    / "evidence/fullscale_physicsnemo_training/"
      "final_completion_20261108_v1/final_release/"
      "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

EXPECTED_RELEASE40_SHA = (
    "49f28c0dd34a7f0da20433905092d6dbc691124fc70439e813ae1adf281f5c52"
)

assert RELEASE40.is_file(), (
    f"Original November 8 release not found: {RELEASE40}"
)

def sha256_40(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

actual_sha40 = sha256_40(RELEASE40)
assert actual_sha40 == EXPECTED_RELEASE40_SHA, (
    "Original release checksum mismatch. "
    "Do not proceed using an unverified archive."
)

print("=" * 70)
print("STAGE 40 — ORIGINAL NASA c-LFM / PHYSICSNEMO")
print("=" * 70)
print("ORIGINAL_RELEASE_SHA256: PASS")
print("GPU_AVAILABLE:", __import__("torch").cuda.is_available())

with zipfile.ZipFile(RELEASE40, "r") as z:
    names = sorted(
        n for n in z.namelist()
        if not n.endswith("/")
    )

    def find_suffix(suffix):
        return [
            n for n in names
            if n.endswith(suffix)
        ]

    manifest_names = find_suffix("/MANIFEST.json")
    assert len(manifest_names) == 1

    release_manifest = json.loads(
        z.read(manifest_names[0]).decode("utf-8")
    )

    print("\nRELEASE_MEMBER_COUNT:", len(names))
    print("MANIFEST_KEYS:", list(release_manifest.keys()))

    # Confirm original and native checkpoint presence.
    required = (
        "checkpoints/original_vae.ckpt",
        "checkpoints/original_flow.pth",
        "checkpoints/native_vae_v2.pt",
        "checkpoints/native_flow_v3.pt",
        "code/models.py",
        "code/replay.py",
    )

    print("\n=== REQUIRED MODEL ARTIFACTS ===")
    all_present = True

    for suffix in required:
        matches = find_suffix("/" + suffix)
        present = len(matches) == 1
        all_present &= present
        print(
            "PASS" if present else "MISSING",
            suffix
        )

    # Inventory only source files; do not execute them.
    code_files = [
        n for n in names
        if "/code/" in n and n.endswith(".py")
    ]

    print("\n=== RESEARCH RELEASE CODE FILES ===")

    for name in code_files:
        print(name)

    print("\n=== MODEL INTERFACES ===")

    for suffix in ("/code/models.py", "/code/replay.py"):
        matches = find_suffix(suffix)

        if len(matches) != 1:
            continue

        src = z.read(matches[0]).decode("utf-8")
        tree = ast.parse(src, filename=matches[0])

        classes = [
            node.name for node in tree.body
            if isinstance(node, ast.ClassDef)
        ]
        functions = [
            node.name for node in tree.body
            if isinstance(
                node, (ast.FunctionDef, ast.AsyncFunctionDef)
            )
        ]

        print("\nSOURCE:", suffix)
        print("CLASSES:", classes)
        print("FUNCTIONS:", functions)

    # Determine whether independent training scripts
    # were actually packaged with the release.
    training_scripts = [
        n for n in names
        if n.endswith(".py")
        and (
            "train" in Path(n).stem.lower()
            or "fit" in Path(n).stem.lower()
        )
    ]

    print("\n=== AVAILABLE TRAINING ENTRYPOINTS ===")

    for name in training_scripts[:40]:
        print(name)

    if not training_scripts:
        print("No explicitly named training scripts in release.")

    print("\n=== UPSTREAM NASA CODE ===")
    upstream = [
        n for n in names
        if "/upstream_nasa/" in n and n.endswith(".py")
    ]
    print("UPSTREAM_PYTHON_FILES:", len(upstream))

print("\n" + "=" * 70)
print("STAGE40_RELEASE_INTEGRITY: PASS")
print("STAGE40_REQUIRED_ARTIFACTS_PRESENT:", all_present)
print("HDF5_DATASETS_OPENED: FALSE")
print("TRAINING_EXECUTED: FALSE")
print("INDEPENDENT_TEST_EVALUATED: FALSE")
print("STAGE40_SOURCE_INVENTORY_COMPLETE")
print("=" * 70)


STAGE 40 — ORIGINAL NASA c-LFM / PHYSICSNEMO
ORIGINAL_RELEASE_SHA256: PASS
GPU_AVAILABLE: True

RELEASE_MEMBER_COUNT: 102
MANIFEST_KEYS: ['files_sha256', 'format', 'fresh_process_replays', 'nasa_commit', 'physicsnemo', 'python', 'torch', 'w1_generation', 'w2_tested_parity', 'w3_generation_execution', 'w3_physical_fidelity', 'w4_end_to_end_target', 'w4_vae_optimizer_step_target']

=== REQUIRED MODEL ARTIFACTS ===
PASS checkpoints/original_vae.ckpt
PASS checkpoints/original_flow.pth
PASS checkpoints/native_vae_v2.pt
PASS checkpoints/native_flow_v3.pt
PASS code/models.py
PASS code/replay.py

=== RESEARCH RELEASE CODE FILES ===
WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1/code/models.py
WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1/code/replay.py

=== MODEL INTERFACES ===

SOURCE: /code/models.py
CLASSES: ['PhysicsNeMoWindVAE', 'PhysicsNeMoWindFlow']
FUNCTIONS: ['load_vae', 'load_flow']

SOURCE: /code/replay.py
CLASSES: []
FUNCTIONS: ['sha', 'main']

=== AVAILABLE TRAINING ENTRYPOINTS ===
WIND_VI

In [ ]:
# ================================================================
# STAGE 41 — RUNTIME RECOVERY
# ================================================================
# Restore variables needed by Stage 41 after a Colab restart.
# Read-only. Does not modify any previous research results.

from pathlib import Path
import hashlib
import torch

# Reconnect Google Drive if necessary
from google.colab import drive
drive.mount("/content/drive")

ROOT40 = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

RELEASE40 = (
    ROOT40
    / "evidence/fullscale_physicsnemo_training/"
      "final_completion_20261108_v1/final_release/"
      "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

EXPECTED_RELEASE40_SHA = (
    "49f28c0dd34a7f0da20433905092d6dbc691124fc70439e813ae1adf281f5c52"
)

def sha256_40(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(chunk)

    return h.hexdigest()

# Verify original research archive
assert RELEASE40.is_file(), (
    f"Research archive not found: {RELEASE40}"
)

assert sha256_40(RELEASE40) == EXPECTED_RELEASE40_SHA, (
    "Original research archive checksum mismatch"
)

print("=" * 70)
print("STAGE 41 — RUNTIME RECOVERY")
print("=" * 70)
print("GOOGLE_DRIVE: CONNECTED")
print("RESEARCH_RELEASE: VERIFIED")
print("RELEASE_SHA256: PASS")
print("CUDA_AVAILABLE:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "None")
print("PREVIOUS_EVIDENCE: UNCHANGED")
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("RUNTIME_RECOVERY: PASS")
print("=" * 70)

Mounted at /content/drive
STAGE 41 — RUNTIME RECOVERY
GOOGLE_DRIVE: CONNECTED
RESEARCH_RELEASE: VERIFIED
RELEASE_SHA256: PASS
CUDA_AVAILABLE: True
GPU: NVIDIA L4
PREVIOUS_EVIDENCE: UNCHANGED
INDEPENDENT_TEST_ACCESSED: FALSE
RUNTIME_RECOVERY: PASS


In [ ]:

# ================================================================
# STAGE 41 — NEURAL GENERATOR BOTTLENECK PREPARATION
# ================================================================
# Verified November 8 NASA/PhysicsNeMo release.
# Read-only. No training. No HDF5 access.
# No independent test evaluation.
# No extraction or modification of original artifacts.

import ast, io, json, hashlib, zipfile
import numpy as np
from pathlib import Path

assert RELEASE40.is_file()
assert sha256_40(RELEASE40) == EXPECTED_RELEASE40_SHA

REFERENCE = {
    "temporal_variance": 3.2106091670893204,
    "step_rms": 0.241722,
    "bands": {
        "low": 5299.1570893018825,
        "middle": 394.68092793159934,
        "high": 16.48637577584998,
        "tail": 0.22229419608779363,
    }
}

BANDS = {
    "low": (1, 7),
    "middle": (7, 17),
    "high": (17, 65),
    "tail": (65, 129),
}

stage41_results = {}
stage41_array_catalog = []

print("=" * 70)
print("STAGE 41 — NASA c-LFM NEURAL FIDELITY INVESTIGATION")
print("=" * 70)

with zipfile.ZipFile(RELEASE40, "r") as z:

    assert z.testzip() is None, "Archive integrity failure"

    names = sorted(
        n for n in z.namelist()
        if not n.endswith("/")
    )

    root = next(
        n[:-len("code/models.py")]
        for n in names
        if n.endswith("/code/models.py")
    )

    manifest = json.loads(
        z.read(root + "MANIFEST.json")
    )

    # Verify all archived component hashes.
    expected_files = manifest["files_sha256"]
    assert isinstance(expected_files, dict)

    for relpath, expected_hash in expected_files.items():
        candidates = (root + relpath, relpath)

        matches = [p for p in candidates if p in names]
        assert matches, f"Missing manifest member: {relpath}"

        actual = hashlib.sha256(
            z.read(matches[0])
        ).hexdigest()

        assert actual == expected_hash, relpath

    print("ARCHIVED_COMPONENT_HASHES_VERIFIED:",
          len(expected_files))

    # ------------------------------------------------------------
    # 1. Identify the actual VAE and flow interfaces
    # ------------------------------------------------------------

    print("\n=== NATIVE MODEL INTERFACES ===")

    model_source = z.read(
        root + "code/models.py"
    ).decode("utf-8")

    model_tree = ast.parse(model_source)
    model_lines = model_source.splitlines()

    for node in model_tree.body:

        if isinstance(node, ast.ClassDef):
            print("\nCLASS:", node.name)

            for method in node.body:
                if not isinstance(
                    method, (ast.FunctionDef, ast.AsyncFunctionDef)
                ):
                    continue

                if method.name in (
                    "__init__", "forward", "encode",
                    "decode", "sample"
                ):
                    params = [
                        arg.arg for arg in method.args.args
                    ]
                    print(
                        " METHOD:", method.name,
                        "ARGS:", params
                    )

        elif isinstance(node, ast.FunctionDef):
            if node.name in ("load_vae", "load_flow"):
                print(
                    "\nLOADER:", node.name,
                    "ARGS:",
                    [a.arg for a in node.args.args]
                )
                print("SOURCE:")
                for line in model_lines[
                    node.lineno - 1 : node.end_lineno
                ][:45]:
                    print(line)

    # ------------------------------------------------------------
    # 2. Inspect replay's decoder and latent handling
    # ------------------------------------------------------------

    print("\n=== REPLAY LATENT/DECODER OPERATIONS ===")

    replay_source = z.read(
        root + "code/replay.py"
    ).decode("utf-8")

    replay_lines = replay_source.splitlines()

    keywords = (
        "decode(", "encode(", "load_vae(",
        "load_flow(", "np.savez", "torch.save",
        "z_mean", "z_std", "latent",
        "coordinates", "points", "grid"
    )

    selected = [
        (i, line)
        for i, line in enumerate(replay_lines, 1)
        if any(word in line for word in keywords)
    ]

    for line_no, line in selected[:45]:
        print(f"L{line_no}: {line.strip()}")

    # ------------------------------------------------------------
    # 3. Inventory archived evidence arrays
    # ------------------------------------------------------------

    print("\n=== ARCHIVED GENERATION AND LATENT ARRAYS ===")

    npz_members = [
        n for n in names
        if n.lower().endswith(".npz")
        and "/evidence/" in n
        and "test" not in n.lower()
    ]

    for member in npz_members:
        print("\nEVIDENCE_FILE:", member[len(root):])

        try:
            with np.load(
                io.BytesIO(z.read(member)),
                allow_pickle=False
            ) as pack:

                for key in pack.files:
                    try:
                        arr = pack[key]
                    except (ValueError, TypeError) as exc:
                        print(
                            " SKIPPED_ARRAY:", key, str(exc)
                        )
                        continue

                    entry = {
                        "file": member[len(root):],
                        "key": key,
                        "shape": list(arr.shape),
                        "dtype": str(arr.dtype),
                    }
                    stage41_array_catalog.append(entry)

                    print(
                        " ARRAY:", key,
                        "SHAPE:", arr.shape,
                        "DTYPE:", arr.dtype
                    )

                    # Compute metrics only for recognizable
                    # physical wind arrays, never for latents.
                    if (
                        arr.ndim == 3
                        and arr.shape[1:] == (100, 256)
                        and np.issubdtype(
                            arr.dtype, np.number
                        )
                    ):
                        field = np.asarray(
                            arr, dtype=np.float64
                        )

                        if not np.isfinite(field).all():
                            print(" NONFINITE_FIELD: SKIPPED")
                            continue

                        centered = field - field.mean(
                            axis=-1, keepdims=True
                        )

                        temporal_variance = float(
                            np.mean(centered**2)
                        )

                        step_rms = float(np.sqrt(
                            np.mean(
                                np.diff(field, axis=-1)**2
                            )
                        ))

                        power = np.abs(
                            np.fft.rfft(
                                centered * np.hanning(256),
                                axis=-1
                            )
                        )**2

                        band_ratios = {
                            band: float(
                                power[:, :, lo:hi].mean()
                                / REFERENCE["bands"][band]
                            )
                            for band, (lo, hi)
                            in BANDS.items()
                        }

                        result = {
                            "fields": int(len(field)),
                            "temporal_variance": temporal_variance,
                            "variance_ratio_vs_reference":
                                temporal_variance
                                / REFERENCE["temporal_variance"],
                            "step_rms": step_rms,
                            "step_rms_ratio_vs_reference":
                                step_rms
                                / REFERENCE["step_rms"],
                            "band_power_ratios_vs_training":
                                band_ratios,
                        }

                        identifier = (
                            member[len(root):] + "::" + key
                        )

                        stage41_results[identifier] = result

                        print(" WIND_FIDELITY:")
                        print(json.dumps(
                            result, indent=2
                        ))

        except Exception as exc:
            print(
                "ARRAY_INSPECTION_UNAVAILABLE:",
                type(exc).__name__, str(exc)
            )

print("\n" + "=" * 70)
print("STAGE 41 — DIAGNOSTIC SUMMARY")
print("=" * 70)

print(
    "NATIVE_MODEL_CLASSES:",
    ["PhysicsNeMoWindVAE", "PhysicsNeMoWindFlow"]
)
print("ARCHIVED_ARRAYS_INSPECTED:",
      len(stage41_array_catalog))
print("WIND_FIELD_ARRAYS_EVALUATED:",
      len(stage41_results))

for key, result in stage41_results.items():
    print("\nFIELD:", key)
    print(
        "TEMPORAL_VARIANCE_RATIO:",
        result["variance_ratio_vs_reference"]
    )
    print(
        "ADJACENT_STEP_RMS_RATIO:",
        result["step_rms_ratio_vs_reference"]
    )
    print(
        "BAND_POWER_RATIOS:",
        result["band_power_ratios_vs_training"]
    )

print("\nSOURCE_RELEASE_SHA256_VERIFIED: TRUE")
print("COMPONENT_MANIFEST_VERIFIED: TRUE")
print("NEW_TRAINING: FALSE")
print("INDEPENDENT_TEST_EVALUATED: FALSE")
print("STAGE41_COMPLETE")


STAGE 41 — NASA c-LFM NEURAL FIDELITY INVESTIGATION
ARCHIVED_COMPONENT_HASHES_VERIFIED: 101

=== NATIVE MODEL INTERFACES ===

CLASS: PhysicsNeMoWindVAE
 METHOD: __init__ ARGS: ['self', 'cfg']
 METHOD: encode ARGS: ['self', 'u']
 METHOD: decode ARGS: ['self', 'z', 'x']
 METHOD: forward ARGS: ['self', 'u', 'x', 'epsilon']

CLASS: PhysicsNeMoWindFlow
 METHOD: __init__ ARGS: ['self', 'cfg']
 METHOD: forward ARGS: ['self', 'z', 't']

LOADER: load_vae ARGS: ['path', 'device']
SOURCE:
def load_vae(path, device="cuda"):
    item = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    assert item["format"] == (
        "wind_vision_physicsnemo_vae_v1"
    )

    model = PhysicsNeMoWindVAE(
        item["config"]
    ).to(device).eval()

    model.load_state_dict(
        item["model_state"],
        strict=True
    )

    return model, item["config"]

LOADER: load_flow ARGS: ['path', 'device']
SOURCE:
def load_flow(path, device="cuda"):
    item = torch.load

In [ ]:

# ================================================================
# STAGE 42 — DIRECT HDF5 EXTRACTION IN COLAB (CORRECTED)
# ================================================================
# Uses the 5000-field HDF5 already uploaded to Colab.
#
# NO NEW UPLOAD
# NO TRAINING
# NO DEVELOPMENT FIELDS EXPORTED
# NO INDEPENDENT TEST ACCESS
# NO OVERWRITING PREVIOUS RESEARCH EVIDENCE
# ================================================================

from pathlib import Path
from google.colab import drive
import numpy as np
import h5py
import hashlib
import io
import gc

print("=" * 70)
print("WIND VISION — STAGE 42")
print("VERIFIED TRAINING REFERENCE EXTRACTION")
print("=" * 70)

# ------------------------------------------------
# 1. Recover the already-uploaded HDF5
# ------------------------------------------------

drive.mount("/content/drive")

# Free memory held by the previous files.upload() call.
# The uploaded HDF5 itself remains on disk.
globals().pop("uploaded", None)
globals().pop("payload", None)
gc.collect()

candidates = [
    Path("/content/wind_train_data.hdf5"),
    Path.cwd() / "wind_train_data.hdf5",
]

source = next(
    (p for p in candidates if p.is_file()),
    None
)

assert source is not None, (
    "Previously uploaded wind_train_data.hdf5 not found. "
    "Check the Colab Files sidebar."
)

print("SOURCE_FILE:", source)
print("SOURCE_SIZE_MIB:", round(
    source.stat().st_size / 2**20, 3
))

# ------------------------------------------------
# 2. Verify original dataset integrity
# ------------------------------------------------

EXPECTED_SOURCE_SHA = (
    "3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6"
)

EXPECTED_TRAIN_SHA = (
    "f0b44c3c14a27b3d2bae90cf6ef8867bbaeca255c80803290f4954206fb3e33c"
)

EXPECTED_DEV_SHA = (
    "4de9038ec463633bf03a8c08b64ea6a155d54816818339a6647e6b39ac0ba6d7"
)

def stage42_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

actual_sha = stage42_sha256(source)

print("ACTUAL_SOURCE_SHA256:", actual_sha)

assert actual_sha == EXPECTED_SOURCE_SHA, (
    "Dataset SHA256 mismatch. "
    "Do not use an unverified or different dataset."
)

print("ORIGINAL_TRAINING_HDF5_SHA256: PASS")

# ------------------------------------------------
# 3. Reconstruct verified Stage 33 split
# ------------------------------------------------

rng = np.random.default_rng(331001)
all_ids = rng.permutation(5000)

dev_ids = all_ids[:500]
train_ids = all_ids[500:]

def stage42_ids_hash(arr):
    return hashlib.sha256(
        np.asarray(arr, dtype="<i8").tobytes()
    ).hexdigest()

assert stage42_ids_hash(train_ids) == EXPECTED_TRAIN_SHA
assert stage42_ids_hash(dev_ids) == EXPECTED_DEV_SHA

assert len(train_ids) == 4500
assert len(dev_ids) == 500
assert not np.isin(train_ids, dev_ids).any()

print("ORIGINAL_TRAIN_SPLIT: PASS")
print("ORIGINAL_DEVELOPMENT_SPLIT: PASS")

# ------------------------------------------------
# 4. Extract 64 TRAINING fields only
# ------------------------------------------------

selected_ids = train_ids[:64]

assert len(np.unique(selected_ids)) == 64
assert not np.isin(selected_ids, dev_ids).any()

with h5py.File(source, "r") as h:

    group = h["wind_samples"]

    assert len(group) == 5000

    fields = np.stack([
        group[str(int(i))]["v1"][:]
        for i in selected_ids
    ]).astype(np.float32)

    xyz = np.asarray(
        h["x_grid"], dtype=np.float32
    )

    times = np.asarray(
        h["t_grid"], dtype=np.float32
    )

assert fields.shape == (64, 100, 256)
assert xyz.shape == (100, 3)
assert times.shape == (256,)

assert np.isfinite(fields).all()
assert np.isfinite(xyz).all()
assert np.isfinite(times).all()

print("TRAINING_FIELDS_EXTRACTED:", len(fields))
print("WIND_SHAPE:", fields.shape)
print("XYZ_SHAPE:", xyz.shape)
print("TIME_SHAPE:", times.shape)

# ------------------------------------------------
# 5. Build compact reference package
# ------------------------------------------------

memory_buffer = io.BytesIO()

np.savez_compressed(
    memory_buffer,
    wind_mps=fields,
    xyz=xyz,
    t_grid=times,
    field_indices=selected_ids,
    source_sha256=np.array(EXPECTED_SOURCE_SHA),
    train_ids_sha256=np.array(EXPECTED_TRAIN_SHA),
    dev_ids_sha256=np.array(EXPECTED_DEV_SHA),
)

package = memory_buffer.getvalue()

package_sha = hashlib.sha256(package).hexdigest()

print("PACKAGE_SIZE_MIB:", round(
    len(package) / 2**20, 3
))

# ------------------------------------------------
# 6. Save permanently to Google Drive
# ------------------------------------------------

destination = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction/"
    "evidence/stage42_reference_inputs"
)

destination.mkdir(
    parents=True,
    exist_ok=True
)

output = destination / (
    f"training_only_64_{package_sha[:16]}.npz"
)

if output.exists():

    assert stage42_sha256(output) == package_sha, (
        "Existing reference package checksum mismatch"
    )

    print("EXISTING_PACKAGE_REUSED: TRUE")

else:

    with output.open("xb") as f:
        f.write(package)

assert stage42_sha256(output) == package_sha

# ------------------------------------------------
# 7. Verify saved package contents
# ------------------------------------------------

with np.load(output, allow_pickle=False) as saved:

    assert np.array_equal(
        saved["wind_mps"], fields
    )

    assert np.array_equal(
        saved["field_indices"], selected_ids
    )

    assert np.array_equal(
        saved["xyz"], xyz
    )

    assert np.array_equal(
        saved["t_grid"], times
    )

    assert saved["source_sha256"].item() == EXPECTED_SOURCE_SHA
    assert saved["train_ids_sha256"].item() == EXPECTED_TRAIN_SHA
    assert saved["dev_ids_sha256"].item() == EXPECTED_DEV_SHA

print()
print("=" * 70)
print("STAGE 42 — REFERENCE INPUT READY")
print("=" * 70)

print("STAGE42_REFERENCE_INPUT_READY: TRUE")
print("ORIGINAL_DATASET_SHA256: VERIFIED")
print("TRAIN_SPLIT_SHA256: VERIFIED")
print("DEV_SPLIT_SHA256: VERIFIED")

print("REFERENCE_INPUT:", output)
print("REFERENCE_INPUT_SHA256:", package_sha)

print("TRAIN_ONLY_FIELDS:", 64)
print("DEVELOPMENT_FIELDS_EXPORTED:", 0)
print("INDEPENDENT_TEST_ACCESSED: FALSE")

print("PREVIOUS_EVIDENCE_UNCHANGED: TRUE")
print("STAGE42_COMPLETE")
print("=" * 70)


WIND VISION — STAGE 42
VERIFIED TRAINING REFERENCE EXTRACTION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SOURCE_FILE: /content/wind_train_data.hdf5
SOURCE_SIZE_MIB: 983.276
ACTUAL_SOURCE_SHA256: 3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6
ORIGINAL_TRAINING_HDF5_SHA256: PASS
ORIGINAL_TRAIN_SPLIT: PASS
ORIGINAL_DEVELOPMENT_SPLIT: PASS
TRAINING_FIELDS_EXTRACTED: 64
WIND_SHAPE: (64, 100, 256)
XYZ_SHAPE: (100, 3)
TIME_SHAPE: (256,)
PACKAGE_SIZE_MIB: 5.292

STAGE 42 — REFERENCE INPUT READY
STAGE42_REFERENCE_INPUT_READY: TRUE
ORIGINAL_DATASET_SHA256: VERIFIED
TRAIN_SPLIT_SHA256: VERIFIED
DEV_SPLIT_SHA256: VERIFIED
REFERENCE_INPUT: /content/drive/MyDrive/CLFM_Research_Results/physicsnemo_proposal_reproduction/evidence/stage42_reference_inputs/training_only_64_bb8fe406dd21025d.npz
REFERENCE_INPUT_SHA256: bb8fe406dd21025d9698ee420bbee0b68d039a13c36ef4ffaf7a28054888e978
TRAIN_ONLY_FIELDS: 64

In [ ]:

# ================================================================
# WIND VISION — STAGE 43 V4
# VERIFIED SPARSE-ENCODER / DECODER / FLOW DIAGNOSIS
# ================================================================
#
# COMPLETE REPLACEMENT — SINGLE COLAB CELL
#
# CORRECTION:
#   The archived VAE encoder was trained with 40 sensors.
#   Select those sensors using the frozen model configuration.
#   Decode the resulting latents at all 100 sensor locations.
#
# SCIENTIFIC CONTROLS:
#   - Frozen original checkpoint and architecture
#   - Archived two-field decoder replay required
#   - 64 verified training-only wind fields
#   - Original normalization and coordinate ordering
#   - No independent test or development data accessed
#   - No training or checkpoint modifications
#
# STORAGE CONTROLS:
#   - No new package installations
#   - No new HDF5 copies
#   - No duplicate checkpoints
#   - Temporary Python source under /content only
#   - Only a small JSON report and hash manifest on Drive
#   - Maximum total new Drive write: 64 KiB
#
# BACKEND:
#   PyTorch-compatible inference, NOT a native
#   PhysicsNeMo performance benchmark.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

import ast
import gc
import hashlib
import io
import json
import shutil
import sys
import tempfile
import types
import uuid
import zipfile

import numpy as np
import torch


# ================================================================
# 1. RESTORE VERIFIED INPUT PATHS
# ================================================================

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

ROOT43 = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

RELEASE43 = (
    ROOT43
    / "evidence/fullscale_physicsnemo_training/"
      "final_completion_20261108_v1/final_release/"
      "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

REFERENCE43 = (
    ROOT43
    / "evidence/stage42_reference_inputs/"
      "training_only_64_bb8fe406dd21025d.npz"
)

EXPECTED_RELEASE43 = (
    "49f28c0dd34a7f0da20433905092d6dbc691124fc70439e813ae1adf281f5c52"
)

EXPECTED_REFERENCE43 = (
    "bb8fe406dd21025d9698ee420bbee0b68d039a13c36ef4ffaf7a28054888e978"
)

EXPECTED_SOURCE43 = (
    "3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6"
)

EXPECTED_TRAIN_IDS43 = (
    "f0b44c3c14a27b3d2bae90cf6ef8867bbaeca255c80803290f4954206fb3e33c"
)

EXPECTED_DEV_IDS43 = (
    "4de9038ec463633bf03a8c08b64ea6a155d54816818339a6647e6b39ac0ba6d7"
)


def sha43(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(block)

    return h.hexdigest()


print("=" * 72)
print("WIND VISION — STAGE 43 V4")
print("SPARSE ENCODER / DECODER / FLOW DIAGNOSIS")
print("=" * 72)

assert RELEASE43.is_file(), "Verified release not found"
assert REFERENCE43.is_file(), "Stage 42 reference missing"

assert sha43(RELEASE43) == EXPECTED_RELEASE43
assert sha43(REFERENCE43) == EXPECTED_REFERENCE43

assert torch.cuda.is_available(), (
    "NVIDIA CUDA runtime required"
)

free_disk43 = shutil.disk_usage("/content").free

assert free_disk43 >= 256 * 2**20

print("RELEASE_SHA256: PASS")
print("REFERENCE_SHA256: PASS")
print("GPU:", torch.cuda.get_device_name(0))
print("COLAB_LOCAL_FREE_GIB:", round(free_disk43 / 2**30, 2))
print("GOOGLE_DRIVE_QUOTA: NOT_INFERRED_FROM_LOCAL_DISK")
print("NEW_PACKAGE_INSTALLATIONS: NONE")
print("DATASET_DUPLICATION: NONE")
print("CHECKPOINT_DUPLICATION: NONE")


# ================================================================
# 2. LOAD 64 VERIFIED TRAINING FIELDS
# ================================================================

with np.load(REFERENCE43, allow_pickle=False) as p:

    real43 = p["wind_mps"].astype(np.float32)
    xyz43 = p["xyz"].astype(np.float32)
    times43 = p["t_grid"].astype(np.float32)
    ids43 = p["field_indices"].copy()

    assert p["source_sha256"].item() == EXPECTED_SOURCE43

    assert (
        p["train_ids_sha256"].item()
        == EXPECTED_TRAIN_IDS43
    )

    assert (
        p["dev_ids_sha256"].item()
        == EXPECTED_DEV_IDS43
    )

assert real43.shape == (64, 100, 256)
assert xyz43.shape == (100, 3)
assert times43.shape == (256,)

assert np.isfinite(real43).all()
assert np.isfinite(xyz43).all()
assert np.isfinite(times43).all()

frozen_split43 = (
    np.random.default_rng(331001).permutation(5000)
)

assert np.array_equal(
    ids43, frozen_split43[500:564]
)

assert not np.isin(
    ids43, frozen_split43[:500]
).any()

print("\nTRAINING_FIELDS_VERIFIED: 64")
print("REFERENCE_SENSOR_COUNT: 100")
print("REFERENCE_TIME_STEPS: 256")
print("DEVELOPMENT_FIELDS_ACCESSED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")


# ================================================================
# 3. ORIGINAL NORMALIZATION AND FULL DECODER GRID
# ================================================================

# NASA reference velocity normalization:
# u_normalized = (u_mps - (-2.0)) / 49.0

real_norm43 = np.ascontiguousarray(
    (real43 + 2.0) / 49.0,
    dtype=np.float32
)

# Reconstruct all 100 sensors, all 256 time points.
# Sensor-major ordering matches archived generation.

coords43 = np.concatenate(
    [
        np.broadcast_to(
            xyz43[:, None, :],
            (100, 256, 3)
        ),
        np.broadcast_to(
            times43[None, :, None],
            (100, 256, 1)
        )
    ],
    axis=-1
).reshape(25600, 4).copy()

assert coords43.shape == (25600, 4)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False

DEVICE43 = "cuda"
BATCH43 = 2


# ================================================================
# 4. COMMON SCIENTIFIC METRICS
# ================================================================

BANDS43 = {
    "low": (1, 7),
    "middle": (7, 17),
    "high": (17, 65),
    "tail": (65, 129)
}


def statistics43(fields):

    a = np.asarray(fields, dtype=np.float64)

    assert a.ndim == 3
    assert a.shape[-1] == 256
    assert a.shape[1] > 0
    assert np.isfinite(a).all()

    centered = a - a.mean(
        axis=-1, keepdims=True
    )

    field_energy = np.mean(
        centered**2,
        axis=(1, 2)
    )

    power = np.abs(
        np.fft.rfft(
            centered * np.hanning(256),
            axis=-1
        )
    )**2

    assert field_energy.mean() > 0

    return {
        "n_fields": int(a.shape[0]),
        "n_sensors": int(a.shape[1]),

        "temporal_variance": float(
            np.mean(centered**2)
        ),

        "step_rms": float(
            np.sqrt(np.mean(
                np.diff(a, axis=-1)**2
            ))
        ),

        "between_field_energy_cv": float(
            field_energy.std(ddof=0)
            / field_energy.mean()
        ),

        "band_power": {
            name: float(
                power[:, :, lo:hi].mean()
            )
            for name, (lo, hi) in BANDS43.items()
        }
    }


def compare43(generated, reference, paired=False):

    a = statistics43(generated)
    b = statistics43(reference)

    assert a["n_sensors"] == b["n_sensors"]

    result = {
        "generated_fields": a["n_fields"],
        "reference_fields": b["n_fields"],
        "sensor_count": a["n_sensors"],

        "temporal_variance_ratio": float(
            a["temporal_variance"]
            / b["temporal_variance"]
        ),

        "step_rms_ratio": float(
            a["step_rms"]
            / b["step_rms"]
        ),

        "between_field_energy_cv": (
            a["between_field_energy_cv"]
        ),

        "band_power_ratios": {
            name: float(
                a["band_power"][name]
                / b["band_power"][name]
            )
            for name in BANDS43
        }
    }

    if paired:

        assert generated.shape == reference.shape

        result["paired_rmse_mps"] = float(
            np.sqrt(np.mean(
                (
                    generated.astype(np.float64)
                    - reference.astype(np.float64)
                )**2
            ))
        )

    return result


# ================================================================
# 5. VERIFIED RELEASE COMPONENTS
# ================================================================

results43 = {}

with zipfile.ZipFile(RELEASE43, "r") as archive43:

    assert archive43.testzip() is None

    names43 = archive43.namelist()

    prefix43 = next(
        name[:-len("code/models.py")]
        for name in names43
        if name.endswith("/code/models.py")
    )

    manifest43 = json.loads(
        archive43.read(prefix43 + "MANIFEST.json")
    )

    hashes43 = manifest43["files_sha256"]

    assert len(hashes43) == 101

    def read_verified43(relative):

        data = archive43.read(
            prefix43 + relative
        )

        assert hashlib.sha256(
            data
        ).hexdigest() == hashes43[relative], (
            "Archived SHA mismatch: " + relative
        )

        return data

    source_bytes43 = read_verified43(
        "code/models.py"
    )

    origin_bytes43 = read_verified43(
        "checkpoints/native_vae_origin.pt"
    )

    trained_bytes43 = read_verified43(
        "checkpoints/native_vae_v2.pt"
    )

    generated_bytes43 = read_verified43(
        "evidence/cell02_native_fields.npz"
    )

    with np.load(
        io.BytesIO(generated_bytes43),
        allow_pickle=False
    ) as p:

        flow_fields43 = p["wind_mps"].astype(
            np.float32
        )

        flow_z43 = p[
            "generated_decoder_z"
        ].astype(np.float32)

        archived_xyz43 = p["xyz"].astype(
            np.float32
        )

        archived_times43 = p["t_grid"].astype(
            np.float32
        )

    assert flow_fields43.shape == (16, 100, 256)
    assert flow_z43.shape == (16, 128)

    assert np.array_equal(
        xyz43, archived_xyz43
    )

    assert np.array_equal(
        times43, archived_times43
    )

    print("\nARCHIVED_MODEL_COMPONENTS: VERIFIED")
    print("ARCHIVED_FLOW_FIELDS: 16")


    # ============================================================
    # 6. VERIFY AND ADAPT FRAMEWORK BASE-CLASS IMPORT
    # ============================================================

    original_source43 = source_bytes43.decode("utf-8")
    parsed43 = ast.parse(original_source43)

    imports43 = []

    for node in parsed43.body:

        if not isinstance(node, ast.ImportFrom):
            continue

        for alias in node.names:

            defined = alias.asname or alias.name

            if defined == "PhysicsNeMoModule":
                imports43.append((node, alias))

    assert len(imports43) == 1

    import_node43, alias43 = imports43[0]

    assert alias43.name == "Module"

    assert import_node43.module is not None
    assert import_node43.module.startswith("physicsnemo")
    assert len(import_node43.names) == 1

    framework_imports43 = [
        node
        for node in ast.walk(parsed43)
        if (
            (
                isinstance(node, ast.ImportFrom)
                and node.module is not None
                and node.module.startswith("physicsnemo")
            )
            or (
                isinstance(node, ast.Import)
                and any(
                    a.name.startswith("physicsnemo")
                    for a in node.names
                )
            )
        )
    ]

    assert len(framework_imports43) == 1
    assert framework_imports43[0] is import_node43

    base_uses43 = [
        node
        for node in ast.walk(parsed43)
        if (
            isinstance(node, ast.Name)
            and node.id == "PhysicsNeMoModule"
            and isinstance(node.ctx, ast.Load)
        )
    ]

    class_bases43 = [
        base
        for node in ast.walk(parsed43)
        if isinstance(node, ast.ClassDef)
        for base in node.bases
        if (
            isinstance(base, ast.Name)
            and base.id == "PhysicsNeMoModule"
        )
    ]

    assert len(base_uses43) == len(class_bases43) == 2

    lines43 = original_source43.splitlines(
        keepends=True
    )

    lines43[
        import_node43.lineno - 1:
        import_node43.end_lineno
    ] = [
        "from torch.nn import Module as PhysicsNeMoModule\n"
    ]

    compatible_source43 = "".join(lines43)

    assert ast.parse(compatible_source43)

    original_model_sha43 = hashlib.sha256(
        source_bytes43
    ).hexdigest()

    compatible_model_sha43 = hashlib.sha256(
        compatible_source43.encode("utf-8")
    ).hexdigest()

    print("MODEL_BASE_CLASS: torch.nn.Module")
    print("ARCHIVED_MODEL_SOURCE_CHANGED: FALSE")
    print("CHECKPOINT_REPLAY_GATE_REQUIRED: TRUE")


    # ============================================================
    # 7. TEMPORARY SOURCE IMPORTS
    # ============================================================

    with tempfile.TemporaryDirectory(
        prefix="wind_stage43_v4_",
        dir="/content"
    ) as temp_name43:

        temp43 = Path(temp_name43)
        upstream_prefix43 = (
            prefix43 + "upstream_nasa/"
        )

        for member in names43:

            if not (
                member.startswith(
                    upstream_prefix43 + "clfm/"
                )
                and member.endswith(".py")
            ):
                continue

            relative = member[len(upstream_prefix43):]

            assert ".." not in Path(relative).parts

            source_data = read_verified43(
                "upstream_nasa/" + relative
            )

            local_path = temp43 / relative

            local_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            local_path.write_bytes(source_data)

        # Temporarily isolate pinned NASA code.
        previous_clfm43 = {
            key: value
            for key, value in sys.modules.items()
            if (
                key == "clfm"
                or key.startswith("clfm.")
            )
        }

        for key in previous_clfm43:
            del sys.modules[key]

        sys.path.insert(0, str(temp43))

        try:

            module43 = types.ModuleType(
                "wind_stage43_v4_models"
            )

            module43.__file__ = str(
                temp43 / "models.py"
            )

            exec(
                compile(
                    compatible_source43,
                    module43.__file__,
                    "exec"
                ),
                module43.__dict__
            )

            print("MODEL_IMPORT: PASS")


            # ====================================================
            # 8. RESTORE FROZEN VAE AND SENSOR CONFIGURATION
            # ====================================================

            origin43 = torch.load(
                io.BytesIO(origin_bytes43),
                map_location="cpu",
                weights_only=True
            )

            trained43 = torch.load(
                io.BytesIO(trained_bytes43),
                map_location="cpu",
                weights_only=True
            )

            cfg43 = origin43["architecture_config"]

            vae43 = (
                module43.PhysicsNeMoWindVAE(
                    cfg43
                ).to(DEVICE43).eval()
            )

            vae43.load_state_dict(
                trained43["model_state"],
                strict=True
            )

            vae43.requires_grad_(False)

            assert sum(
                p.numel()
                for p in vae43.parameters()
            ) == 877057

            print("FROZEN_VAE_CHECKPOINT: STRICT PASS")

            # ----------------------------------------------------
            # CRITICAL V4 CORRECTION:
            # Obtain the exact sensor selection from the
            # frozen VAE's original architecture configuration.
            # Never feed 100 channels to a 40-channel encoder.
            # ----------------------------------------------------

            sensor_indices43 = np.asarray(
                cfg43["sensor_indices"],
                dtype=np.int64
            ).reshape(-1)

            configured_channels43 = int(
                cfg43["sensor_count"]
            )

            encoder_channels43 = int(
                vae43._encoder.input_layer.in_channels
            )

            assert configured_channels43 == 40
            assert encoder_channels43 == 40

            assert sensor_indices43.shape == (40,)

            assert np.unique(
                sensor_indices43
            ).size == 40

            assert sensor_indices43.min() >= 0
            assert sensor_indices43.max() < 100

            # Confirm the original NASA sparse-sensor
            # configuration is used.
            expected_sparse43 = np.concatenate([
                np.arange(start, start + 10)
                for start in (0, 30, 60, 90)
            ])

            assert np.array_equal(
                sensor_indices43,
                expected_sparse43
            ), (
                "Unexpected sparse-sensor ordering. "
                "Do not silently change encoder inputs."
            )

            # IMPORTANT: Select sensors BEFORE encoding.
            observed_norm43 = np.ascontiguousarray(
                real_norm43[:, sensor_indices43, :],
                dtype=np.float32
            )

            assert observed_norm43.shape == (
                64, 40, 256
            )

            withheld_indices43 = np.setdiff1d(
                np.arange(100, dtype=np.int64),
                sensor_indices43
            )

            assert len(withheld_indices43) == 60

            print("\nSPARSE_ENCODER_CONFIGURATION: VERIFIED")
            print("ENCODER_INPUT_CHANNELS:", encoder_channels43)
            print("ENCODER_INPUT_SHAPE:", observed_norm43.shape)
            print("DECODER_OUTPUT_SENSORS:", 100)
            print("OBSERVED_SENSOR_COUNT:", 40)
            print("WITHHELD_SENSOR_COUNT:", 60)


            # ====================================================
            # 9. DECODER IMPLEMENTATION
            # ====================================================

            x43 = torch.from_numpy(
                coords43
            ).to(DEVICE43)

            @torch.inference_mode()
            def decode43(latents):

                outputs = []

                for start in range(
                    0, len(latents), BATCH43
                ):

                    z_batch = torch.from_numpy(
                        np.ascontiguousarray(
                            latents[
                                start:start+BATCH43
                            ],
                            dtype=np.float32
                        )
                    ).to(DEVICE43)

                    assert z_batch.shape[1] == 128

                    decoded = vae43.decode(
                        z_batch,
                        x43[None].expand(
                            len(z_batch), -1, -1
                        )
                    )

                    decoded = decoded.reshape(
                        len(z_batch), 100, 256
                    )

                    assert torch.isfinite(
                        decoded
                    ).all()

                    physical = (
                        decoded * 49.0 - 2.0
                    )

                    outputs.append(
                        physical.float().cpu().numpy()
                    )

                return np.concatenate(
                    outputs,
                    axis=0
                ).astype(np.float32)


            # ====================================================
            # 10. ARCHIVED NUMERICAL REPLAY GATE
            # ====================================================

            replay43 = decode43(
                flow_z43[:2]
            )

            replay_diff43 = float(
                np.max(np.abs(
                    replay43 - flow_fields43[:2]
                ))
            )

            print(
                "\nARCHIVED_DECODER_REPLAY_MAX_DIFF_MPS:",
                replay_diff43
            )

            assert replay_diff43 < 1e-3, (
                "Compatibility replay failed. "
                "Do not interpret reconstruction results."
            )

            print("FROZEN_DECODER_REPLAY: PASS")


            # ====================================================
            # 11. ENCODE ACTUAL 40-SENSOR OBSERVATIONS
            # ====================================================

            mean_latents43 = []
            posterior_latents43 = []

            rng43 = torch.Generator(
                device="cpu"
            ).manual_seed(430043)

            with torch.inference_mode():

                for start in range(
                    0, len(observed_norm43), BATCH43
                ):

                    u_batch = torch.from_numpy(
                        np.ascontiguousarray(
                            observed_norm43[
                                start:start+BATCH43
                            ]
                        )
                    ).to(DEVICE43)

                    assert u_batch.shape[1:] == (
                        40, 256
                    )

                    mu, logvar = vae43.encode(
                        u_batch
                    )

                    assert mu.shape == (
                        len(u_batch), 128
                    )

                    assert logvar.shape == mu.shape

                    assert torch.isfinite(mu).all()
                    assert torch.isfinite(logvar).all()

                    epsilon = torch.randn(
                        mu.shape,
                        generator=rng43,
                        dtype=torch.float32
                    ).to(DEVICE43)

                    sampled = (
                        mu
                        + torch.exp(
                            0.5 * logvar
                        ) * epsilon
                    )

                    assert torch.isfinite(
                        sampled
                    ).all()

                    mean_latents43.append(
                        mu.float().cpu().numpy()
                    )

                    posterior_latents43.append(
                        sampled.float().cpu().numpy()
                    )

            mean_latents43 = np.concatenate(
                mean_latents43,
                axis=0
            )

            posterior_latents43 = np.concatenate(
                posterior_latents43,
                axis=0
            )

            assert mean_latents43.shape == (
                64, 128
            )

            assert posterior_latents43.shape == (
                64, 128
            )

            print("\n40_SENSOR_REFERENCE_ENCODING: PASS")


            # ====================================================
            # 12. RECONSTRUCT ALL 100 WIND SENSORS
            # ====================================================

            reconstructed_mean43 = decode43(
                mean_latents43
            )

            reconstructed_posterior43 = decode43(
                posterior_latents43
            )

            assert reconstructed_mean43.shape == (
                64, 100, 256
            )

            assert reconstructed_posterior43.shape == (
                64, 100, 256
            )

            assert np.isfinite(
                reconstructed_mean43
            ).all()

            assert np.isfinite(
                reconstructed_posterior43
            ).all()

            print("100_SENSOR_RECONSTRUCTION: PASS")


            # ====================================================
            # 13. DIAGNOSTIC COMPARISONS
            # ====================================================

            # Full-field reference reconstruction.
            results43 = {
                "reference_64": statistics43(real43),

                "posterior_mean_reconstruction_64":
                    compare43(
                        reconstructed_mean43,
                        real43,
                        paired=True
                    ),

                "posterior_sample_reconstruction_64":
                    compare43(
                        reconstructed_posterior43,
                        real43,
                        paired=True
                    ),

                "posterior_mean_reconstruction_16":
                    compare43(
                        reconstructed_mean43[:16],
                        real43[:16],
                        paired=True
                    ),

                "flow_generated_16":
                    compare43(
                        flow_fields43,
                        real43[:16],
                        paired=False
                    ),
            }

            # Additional diagnostic: can the encoder–decoder
            # reconstruct sensors it actually observed?
            results43[
                "posterior_mean_observed_40"
            ] = compare43(
                reconstructed_mean43[
                    :, sensor_indices43, :
                ],
                real43[:, sensor_indices43, :],
                paired=True
            )

            # And how does it perform on 60 unobserved
            # sensor histories from these TRAINING fields?
            results43[
                "posterior_mean_withheld_60"
            ] = compare43(
                reconstructed_mean43[
                    :, withheld_indices43, :
                ],
                real43[:, withheld_indices43, :],
                paired=True
            )

            print("\nSCIENTIFIC_METRICS_COMPUTED: PASS")

            # Release GPU model memory.
            del vae43
            del x43

        finally:

            if str(temp43) in sys.path:
                sys.path.remove(str(temp43))

            for key in list(sys.modules):

                if (
                    key == "clfm"
                    or key.startswith("clfm.")
                ):
                    del sys.modules[key]

            sys.modules.update(previous_clfm43)

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ================================================================
# 14. PRINT SCIENTIFIC RESULTS
# ================================================================

print()
print("=" * 72)
print("STAGE 43 V4 — DECODER / FLOW BOTTLENECK RESULTS")
print("=" * 72)

for arm in (
    "posterior_mean_reconstruction_64",
    "posterior_sample_reconstruction_64",
    "posterior_mean_reconstruction_16",
    "flow_generated_16",
    "posterior_mean_observed_40",
    "posterior_mean_withheld_60"
):

    print("\nARM:", arm)

    print(
        json.dumps(
            results43[arm],
            indent=2,
            allow_nan=False
        )
    )

print(
    "\nREFERENCE_64_TEMPORAL_VARIANCE:",
    results43["reference_64"]["temporal_variance"]
)

print(
    "REFERENCE_64_STEP_RMS:",
    results43["reference_64"]["step_rms"]
)

print("\nINTERPRETATION CONTROLS:")
print(
    "Reconstruction: 40 observed sensors -> 100 decoded sensors."
)
print(
    "Flow generation: 16 archived independent draws."
)
print(
    "Reconstruction tests the encoder-decoder pipeline."
)
print(
    "Poor reconstruction alone cannot isolate the decoder."
)
print(
    "No development or independent test fields were used."
)


# ================================================================
# 15. STORAGE-LIMITED EVIDENCE PRESERVATION
# ================================================================

record43 = {
    "format": (
        "wind_vision_stage43_sparse_encoder_"
        "decoder_flow_v4"
    ),

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "release_sha256": EXPECTED_RELEASE43,
    "training_subset_sha256": EXPECTED_REFERENCE43,

    "original_model_source_sha256":
        original_model_sha43,

    "compatible_model_source_sha256":
        compatible_model_sha43,

    "compatibility_backend":
        "PyTorch nn.Module inference",

    "frozen_checkpoint_strict_load": True,

    "archived_decoder_replay_max_diff_mps":
        replay_diff43,

    "encoder_input_sensors": 40,
    "decoder_output_sensors": 100,

    "encoder_sensor_indices":
        sensor_indices43.tolist(),

    "training_fields_used": 64,
    "development_fields_used": 0,
    "independent_test_accessed": False,

    "new_training_performed": False,

    "results": results43,

    "limitations": [
        "Training-field reconstruction only",
        "Sparse 40-sensor encoder configuration",
        "Reconstruction assesses encoder and decoder jointly",
        "Flow generation uses 16 archived independent draws",
        "Not a native PhysicsNeMo performance benchmark",
        "No independent-test validation",
        "Frozen checkpoints and original source unchanged"
    ]
}

report_bytes43 = (
    json.dumps(
        record43,
        sort_keys=True,
        indent=2,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

MAX_NEW_DRIVE_BYTES43 = 64 * 1024

assert len(report_bytes43) <= MAX_NEW_DRIVE_BYTES43

manifest_content43 = {
    "stage43_report.json": hashlib.sha256(
        report_bytes43
    ).hexdigest()
}

manifest_bytes43 = (
    json.dumps(
        manifest_content43,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

total_write43 = (
    len(report_bytes43)
    + len(manifest_bytes43)
)

assert total_write43 <= MAX_NEW_DRIVE_BYTES43

evidence43 = ROOT43 / "evidence"

assert evidence43.is_dir()

tag43 = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

folder43 = evidence43 / (
    "stage43_sparse_encoder_diagnostic_" + tag43
)

folder43.mkdir(
    parents=False,
    exist_ok=False
)

report_path43 = (
    folder43 / "stage43_report.json"
)

manifest_path43 = (
    folder43 / "SHA256SUMS.json"
)

with report_path43.open("xb") as f:
    f.write(report_bytes43)

with manifest_path43.open("xb") as f:
    f.write(manifest_bytes43)

assert sha43(report_path43) == (
    manifest_content43["stage43_report.json"]
)

print("\n" + "=" * 72)
print("STAGE 43 V4 — EVIDENCE AND STORAGE")
print("=" * 72)

print("STAGE43_EVIDENCE:", folder43)
print("STAGE43_MANIFEST_SHA256:", sha43(manifest_path43))
print("NEW_DRIVE_EVIDENCE_BYTES:", total_write43)
print("DRIVE_WRITE_LIMIT_BYTES:", MAX_NEW_DRIVE_BYTES43)
print("NEW_HDF5_COPIES: 0")
print("NEW_CHECKPOINT_COPIES: 0")
print("NEW_PACKAGE_INSTALLATIONS: 0")
print("TEMPORARY_SOURCE_FILES: CLEANED")

print("\nSTAGE43_V4_COMPLETE")
print("FROZEN_DECODER_REPLAY: PASS")
print("SPARSE_ENCODER_CONFIGURATION: VERIFIED")
print("TRAINING_FIELDS_EVALUATED: 64")
print("DEVELOPMENT_FIELDS_ACCESSED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("TRAINING_PERFORMED: FALSE")
print("=" * 72)


WIND VISION — STAGE 43 V4
SPARSE ENCODER / DECODER / FLOW DIAGNOSIS
RELEASE_SHA256: PASS
REFERENCE_SHA256: PASS
GPU: NVIDIA L4
COLAB_LOCAL_FREE_GIB: 234.55
GOOGLE_DRIVE_QUOTA: NOT_INFERRED_FROM_LOCAL_DISK
NEW_PACKAGE_INSTALLATIONS: NONE
DATASET_DUPLICATION: NONE
CHECKPOINT_DUPLICATION: NONE

TRAINING_FIELDS_VERIFIED: 64
REFERENCE_SENSOR_COUNT: 100
REFERENCE_TIME_STEPS: 256
DEVELOPMENT_FIELDS_ACCESSED: FALSE
INDEPENDENT_TEST_ACCESSED: FALSE

ARCHIVED_MODEL_COMPONENTS: VERIFIED
ARCHIVED_FLOW_FIELDS: 16
MODEL_BASE_CLASS: torch.nn.Module
ARCHIVED_MODEL_SOURCE_CHANGED: FALSE
CHECKPOINT_REPLAY_GATE_REQUIRED: TRUE
MODEL_IMPORT: PASS
FROZEN_VAE_CHECKPOINT: STRICT PASS

SPARSE_ENCODER_CONFIGURATION: VERIFIED
ENCODER_INPUT_CHANNELS: 40
ENCODER_INPUT_SHAPE: (64, 40, 256)
DECODER_OUTPUT_SENSORS: 100
OBSERVED_SENSOR_COUNT: 40
WITHHELD_SENSOR_COUNT: 60

ARCHIVED_DECODER_REPLAY_MAX_DIFF_MPS: 2.288818359375e-05
FROZEN_DECODER_REPLAY: PASS

40_SENSOR_REFERENCE_ENCODING: PASS
100_SENSOR_RECONSTRUCTION: 

In [ ]:

# ================================================================
# WIND VISION — STAGE 44
# FROZEN DECODER BASIS CAPACITY AUDIT
# ================================================================
# One complete, restart-safe Colab cell.
#
# OBJECTIVE:
# Determine the best least-squares approximation of genuine
# training wind fields in the frozen decoder trunk's basis.
#
# This is a diagnostic upper envelope for the fixed basis,
# NOT a generative model and NOT a new trained checkpoint.
#
# STORAGE:
# - No installs, downloads, HDF5 copies or checkpoint copies
# - Temporary Python source only in /content
# - Full calculations in RAM / GPU
# - Drive writes <= 64 KiB, JSON evidence only
#
# DATA ISOLATION:
# - 64 previously verified TRAINING fields
# - 0 development fields
# - 0 independent test fields
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import ast, gc, hashlib, io, json, shutil, sys
import tempfile, types, uuid, zipfile

import numpy as np
import torch

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

RELEASE = ROOT / (
    "evidence/fullscale_physicsnemo_training/"
    "final_completion_20261108_v1/final_release/"
    "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

REFERENCE = ROOT / (
    "evidence/stage42_reference_inputs/"
    "training_only_64_bb8fe406dd21025d.npz"
)

RELEASE_SHA = (
    "49f28c0dd34a7f0da20433905092d6dbc691124fc70439e813ae1adf281f5c52"
)

REFERENCE_SHA = (
    "bb8fe406dd21025d9698ee420bbee0b68d039a13c36ef4ffaf7a28054888e978"
)

SOURCE_SHA = (
    "3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6"
)

TRAIN_IDS_SHA = (
    "f0b44c3c14a27b3d2bae90cf6ef8867bbaeca255c80803290f4954206fb3e33c"
)

DEV_IDS_SHA = (
    "4de9038ec463633bf03a8c08b64ea6a155d54816818339a6647e6b39ac0ba6d7"
)

def sha44(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("STAGE 44 — FROZEN DECODER BASIS CAPACITY")
print("=" * 70)

assert torch.cuda.is_available()
assert RELEASE.is_file() and REFERENCE.is_file()
assert sha44(RELEASE) == RELEASE_SHA
assert sha44(REFERENCE) == REFERENCE_SHA

local_free = shutil.disk_usage("/content").free
assert local_free > 512 * 2**20

print("RESEARCH_INPUT_HASHES: PASS")
print("GPU:", torch.cuda.get_device_name(0))
print("COLAB_LOCAL_FREE_GIB:", round(local_free / 2**30, 2))
print("DRIVE_QUOTA_INFERRED_FROM_LOCAL_DISK: FALSE")
print("PACKAGE_INSTALLATIONS: NONE")
print("NEW_DATASET_COPIES: NONE")

# ------------------------------------------------
# 1. VERIFY TRAINING INPUTS
# ------------------------------------------------

with np.load(REFERENCE, allow_pickle=False) as p:
    real = p["wind_mps"].astype(np.float32)
    xyz = p["xyz"].astype(np.float32)
    times = p["t_grid"].astype(np.float32)
    indices = p["field_indices"].copy()

    assert p["source_sha256"].item() == SOURCE_SHA
    assert p["train_ids_sha256"].item() == TRAIN_IDS_SHA
    assert p["dev_ids_sha256"].item() == DEV_IDS_SHA

assert real.shape == (64, 100, 256)
assert xyz.shape == (100, 3)
assert times.shape == (256,)
assert np.isfinite(real).all()

split = np.random.default_rng(331001).permutation(5000)
assert np.array_equal(indices, split[500:564])
assert not np.isin(indices, split[:500]).any()

normalized = np.ascontiguousarray(
    (real + 2.0) / 49.0, dtype=np.float32
)

coords = np.concatenate([
    np.broadcast_to(xyz[:, None, :], (100, 256, 3)),
    np.broadcast_to(times[None, :, None], (100, 256, 1))
], axis=-1).reshape(25600, 4).copy()

print("VERIFIED_TRAINING_FIELDS:", len(real))
print("DEVELOPMENT_FIELDS_ACCESSED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")

# ------------------------------------------------
# 2. FIDELITY METRICS
# ------------------------------------------------

BANDS = {
    "low": (1, 7),
    "middle": (7, 17),
    "high": (17, 65),
    "tail": (65, 129),
}

def describe44(a):
    a = np.asarray(a, dtype=np.float64)
    assert a.shape == (64, 100, 256)
    assert np.isfinite(a).all()

    centered = a - a.mean(axis=-1, keepdims=True)
    window = np.hanning(256)

    power = np.abs(
        np.fft.rfft(
            centered * window, axis=-1
        )
    ) ** 2

    energy = np.mean(centered**2, axis=(1, 2))

    return {
        "variance": float(energy.mean()),
        "step_rms": float(np.sqrt(
            np.mean(np.diff(a, axis=-1)**2)
        )),
        "energy_cv": float(
            energy.std(ddof=0) / energy.mean()
        ),
        "band_power": {
            name: float(power[:, :, lo:hi].mean())
            for name, (lo, hi) in BANDS.items()
        },
    }

reference_stats = describe44(real)

def compare44(fields):
    stat = describe44(fields)

    return {
        "paired_rmse_mps": float(np.sqrt(
            np.mean(
                (fields.astype(np.float64)
                 - real.astype(np.float64))**2
            )
        )),
        "variance_ratio": (
            stat["variance"] /
            reference_stats["variance"]
        ),
        "step_rms_ratio": (
            stat["step_rms"] /
            reference_stats["step_rms"]
        ),
        "between_field_energy_cv": stat["energy_cv"],
        "band_power_ratios": {
            name: (
                stat["band_power"][name]
                / reference_stats["band_power"][name]
            )
            for name in BANDS
        },
    }

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False

# ------------------------------------------------
# 3. RESTORE VERIFIED FROZEN VAE
# ------------------------------------------------

results = {}
ranks = {}

with zipfile.ZipFile(RELEASE, "r") as archive:
    names = archive.namelist()

    prefix = next(
        name[:-len("code/models.py")]
        for name in names
        if name.endswith("/code/models.py")
    )

    manifest = json.loads(
        archive.read(prefix + "MANIFEST.json")
    )
    hashes = manifest["files_sha256"]

    assert len(hashes) == 101

    def read_verified(relative):
        data = archive.read(prefix + relative)
        assert hashlib.sha256(data).hexdigest() == (
            hashes[relative]
        )
        return data

    model_bytes = read_verified("code/models.py")
    origin_bytes = read_verified(
        "checkpoints/native_vae_origin.pt"
    )
    trained_bytes = read_verified(
        "checkpoints/native_vae_v2.pt"
    )
    flow_bytes = read_verified(
        "evidence/cell02_native_fields.npz"
    )

    with np.load(
        io.BytesIO(flow_bytes), allow_pickle=False
    ) as p:
        archived_fields = p["wind_mps"].astype(np.float32)
        archived_z = p[
            "generated_decoder_z"
        ].astype(np.float32)

        assert np.array_equal(p["xyz"], xyz)
        assert np.array_equal(p["t_grid"], times)

    assert archived_fields.shape == (16, 100, 256)
    assert archived_z.shape == (16, 128)

    # Import adaptation: exact archived source, with only
    # PhysicsNeMo base class changed for PyTorch inference.
    source = model_bytes.decode("utf-8")
    tree = ast.parse(source)

    imports = [
        node for node in tree.body
        if (
            isinstance(node, ast.ImportFrom)
            and any(
                a.asname == "PhysicsNeMoModule"
                and a.name == "Module"
                for a in node.names
            )
        )
    ]

    assert len(imports) == 1
    imp = imports[0]
    assert imp.module == "physicsnemo.core.module"
    assert len(imp.names) == 1

    lines = source.splitlines(keepends=True)
    lines[imp.lineno-1:imp.end_lineno] = [
        "from torch.nn import Module as PhysicsNeMoModule\n"
    ]
    compatible_source = "".join(lines)

    assert ast.parse(compatible_source)

    with tempfile.TemporaryDirectory(
        prefix="wind_stage44_", dir="/content"
    ) as temporary:

        tmp = Path(temporary)
        upstream = prefix + "upstream_nasa/"

        for member in names:
            if not (
                member.startswith(upstream + "clfm/")
                and member.endswith(".py")
            ):
                continue

            rel = member[len(upstream):]
            assert ".." not in Path(rel).parts

            dst = tmp / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(
                read_verified("upstream_nasa/" + rel)
            )

        previous_modules = {
            key: value
            for key, value in sys.modules.items()
            if key == "clfm" or key.startswith("clfm.")
        }

        for key in previous_modules:
            del sys.modules[key]

        sys.path.insert(0, str(tmp))

        try:
            module = types.ModuleType(
                "wind_stage44_verified_models"
            )
            module.__file__ = str(tmp / "models.py")

            exec(
                compile(
                    compatible_source,
                    module.__file__,
                    "exec"
                ),
                module.__dict__
            )

            origin = torch.load(
                io.BytesIO(origin_bytes),
                map_location="cpu",
                weights_only=True
            )
            checkpoint = torch.load(
                io.BytesIO(trained_bytes),
                map_location="cpu",
                weights_only=True
            )

            cfg = origin["architecture_config"]

            vae = module.PhysicsNeMoWindVAE(
                cfg
            ).to("cuda").eval()

            vae.load_state_dict(
                checkpoint["model_state"],
                strict=True
            )
            vae.requires_grad_(False)

            assert sum(
                p.numel() for p in vae.parameters()
            ) == 877057

            sensor_ids = np.asarray(
                cfg["sensor_indices"], dtype=np.int64
            )

            assert sensor_ids.shape == (40,)
            assert np.array_equal(
                sensor_ids,
                np.concatenate([
                    np.arange(i, i+10)
                    for i in (0, 30, 60, 90)
                ])
            )

            print("FROZEN_VAE_STRICT_LOAD: PASS")

            # ------------------------------------------------
            # 4. FORM DECODER TRUNK BASIS ONCE
            # ------------------------------------------------

            with torch.inference_mode():
                x = torch.from_numpy(
                    coords
                ).to("cuda")[None]

                trunk = vae._trunk(
                    vae.grid.normalize(x)
                )

                assert trunk.shape == (
                    1, 25600, 512, 1
                )

                F = trunk.reshape(
                    25600, 512
                ).contiguous()

                bias = vae._bias.reshape(()).detach()

                # Exact frozen-model replay check.
                z_check = torch.from_numpy(
                    archived_z[:2]
                ).to("cuda")

                coefficients = vae._branch(z_check)

                predicted = (
                    coefficients @ F.T + bias
                ).reshape(2, 100, 256)

                predicted = (
                    predicted * 49.0 - 2.0
                ).cpu().numpy()

                replay_diff = float(
                    np.max(np.abs(
                        predicted - archived_fields[:2]
                    ))
                )

                assert replay_diff < 1e-3, (
                    "Trunk-basis decoder does not reproduce "
                    "the archived PhysicsNeMo output."
                )

            print(
                "TRUNK_BASIS_ARCHIVED_REPLAY_MAX_DIFF_MPS:",
                replay_diff
            )
            print("TRUNK_BASIS_REPLAY: PASS")

            # ------------------------------------------------
            # 5. RECOMPUTE THE STAGE 43 ENCODER BASELINE
            # ------------------------------------------------

            observed = normalized[:, sensor_ids, :]
            mu_parts = []

            with torch.inference_mode():
                for start in range(0, 64, 8):
                    u = torch.from_numpy(
                        np.ascontiguousarray(
                            observed[start:start+8]
                        )
                    ).to("cuda")

                    mu, logvar = vae.encode(u)
                    assert mu.shape == (len(u), 128)
                    mu_parts.append(mu)

                mu_all = torch.cat(mu_parts)

                baseline_norm = (
                    vae._branch(mu_all) @ F.T + bias
                ).reshape(64, 100, 256)

                baseline = (
                    baseline_norm * 49.0 - 2.0
                ).cpu().numpy()

            results["encoded_posterior_mean"] = (
                compare44(baseline)
            )

            baseline_rmse = results[
                "encoded_posterior_mean"
            ]["paired_rmse_mps"]

            # Consistency with successfully completed Stage43.
            assert abs(
                baseline_rmse - 1.5763280875889771
            ) < 0.01, (
                "Stage43 reconstruction baseline mismatch"
            )

            print("STAGE43_BASELINE_REPLAY: PASS")
            print("BASELINE_RMSE_MPS:", baseline_rmse)

            # ------------------------------------------------
            # 6. OPTIMAL LEAST-SQUARES SUBSPACE PROJECTION
            # ------------------------------------------------
            #
            # Decoder output is:
            #     u = F @ branch(z) + bias
            #
            # We allow unrestricted coefficients instead of
            # requiring coefficients reachable by branch(z).
            #
            # This gives the best L2 fit in the truncated
            # numerical column space of the frozen trunk.
            #
            # Rank thresholds are sensitivity checks.
            # No model parameters or latent vectors trained.
            # ------------------------------------------------

            target = torch.from_numpy(
                np.ascontiguousarray(
                    normalized.reshape(64, -1).T
                )
            ).to("cuda")

            assert target.shape == (25600, 64)

            print("\nBUILDING_TRUNK_ORTHOGONAL_BASIS...")

            with torch.inference_mode():
                Q, R = torch.linalg.qr(
                    F, mode="reduced"
                )

                assert Q.shape == (25600, 512)
                assert R.shape == (512, 512)

                # Small, CPU float64 SVD for stable rank analysis.
                U, singular, Vh = torch.linalg.svd(
                    R.double().cpu(),
                    full_matrices=False
                )

                assert torch.isfinite(singular).all()

                relative_singular = (
                    singular / singular[0]
                )

                centered_target = target - bias

                for threshold in (1e-4, 1e-6):

                    keep = relative_singular > threshold
                    rank = int(keep.sum().item())

                    assert 0 < rank <= 512

                    # Q @ U[:, keep] is an orthonormal basis
                    # for the stable trunk-feature subspace.
                    stable_basis = Q @ (
                        U[:, keep]
                        .to(device="cuda", dtype=Q.dtype)
                    )

                    fitted = (
                        stable_basis @ (
                            stable_basis.T @ centered_target
                        )
                        + bias
                    )

                    fitted = (
                        fitted.T
                        .reshape(64, 100, 256)
                        * 49.0 - 2.0
                    )

                    fitted_np = (
                        fitted.float().cpu().numpy()
                    )

                    name = (
                        "oracle_trunk_least_squares_"
                        + str(threshold)
                    )

                    results[name] = compare44(
                        fitted_np
                    )
                    ranks[name] = rank

                    print(
                        "ORACLE_RANK_THRESHOLD:",
                        threshold,
                        "EFFECTIVE_RANK:",
                        rank,
                        "PAIRED_RMSE_MPS:",
                        results[name]["paired_rmse_mps"]
                    )

                    del stable_basis, fitted, fitted_np

            del vae, F, Q, R, trunk

        finally:
            if str(tmp) in sys.path:
                sys.path.remove(str(tmp))

            for key in list(sys.modules):
                if key == "clfm" or key.startswith("clfm."):
                    del sys.modules[key]

            sys.modules.update(previous_modules)

            gc.collect()
            torch.cuda.empty_cache()

# ------------------------------------------------
# 7. RESULTS
# ------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 44 — DECODER BASIS CAPACITY RESULTS")
print("=" * 70)

for arm, metrics in results.items():
    print("\nARM:", arm)
    print("EFFECTIVE_RANK:", ranks.get(arm, "encoder"))
    print(json.dumps(metrics, indent=2))

print("\nINTERPRETATION:")
print("Oracle arms fit all 100 training-sensor histories.")
print("They bypass the encoder and the latent-flow model.")
print("They use unrestricted decoder trunk coefficients.")
print("This is an in-sample representational capacity test.")
print("It is NOT a practical generation or prediction method.")

# ------------------------------------------------
# 8. SAVE SMALL, IMMUTABLE EVIDENCE
# ------------------------------------------------

record = {
    "format": "wind_vision_stage44_decoder_basis_capacity_v1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "release_sha256": RELEASE_SHA,
    "training_subset_sha256": REFERENCE_SHA,
    "archived_model_source_sha256": hashlib.sha256(
        model_bytes
    ).hexdigest(),
    "backend": "PyTorch-compatible frozen-model inference",
    "training_fields": 64,
    "development_fields": 0,
    "independent_test_accessed": False,
    "trunk_basis_size": 512,
    "effective_ranks": ranks,
    "archived_replay_max_diff_mps": replay_diff,
    "results": results,
    "training_performed": False,
    "limitations": [
        "Oracle uses complete training-field target values",
        "Oracle coefficients are not constrained to branch(z)",
        "Numerical rank truncation changes attainable subspace",
        "Least-squares minimizes MSE, not spectral discrepancy",
        "Poor spectral fidelity alone does not prove impossibility",
        "No independent test or generative performance claim"
    ]
}

report = (
    json.dumps(
        record, indent=2, sort_keys=True,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

report_hash = hashlib.sha256(report).hexdigest()

manifest = (
    json.dumps({
        "stage44_report.json": report_hash
    }, indent=2, sort_keys=True) + "\n"
).encode("utf-8")

assert len(report) + len(manifest) <= 64 * 1024

evidence = ROOT / "evidence"
assert evidence.is_dir()

tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    ) + "_" + uuid.uuid4().hex[:8]
)

folder = evidence / (
    "stage44_decoder_basis_capacity_" + tag
)

folder.mkdir(parents=False, exist_ok=False)

with (folder / "stage44_report.json").open("xb") as f:
    f.write(report)

with (folder / "SHA256SUMS.json").open("xb") as f:
    f.write(manifest)

assert sha44(folder / "stage44_report.json") == report_hash

print("\nSTAGE44_EVIDENCE:", folder)
print("STAGE44_MANIFEST_SHA256:",
      sha44(folder / "SHA256SUMS.json"))
print("NEW_DRIVE_BYTES:", len(report) + len(manifest))
print("NEW_HDF5_FILES:", 0)
print("NEW_CHECKPOINT_FILES:", 0)
print("NEW_PACKAGE_INSTALLATIONS:", 0)
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("STAGE44_COMPLETE")


STAGE 44 — FROZEN DECODER BASIS CAPACITY
RESEARCH_INPUT_HASHES: PASS
GPU: NVIDIA L4
COLAB_LOCAL_FREE_GIB: 234.55
DRIVE_QUOTA_INFERRED_FROM_LOCAL_DISK: FALSE
PACKAGE_INSTALLATIONS: NONE
NEW_DATASET_COPIES: NONE
VERIFIED_TRAINING_FIELDS: 64
DEVELOPMENT_FIELDS_ACCESSED: FALSE
INDEPENDENT_TEST_ACCESSED: FALSE
FROZEN_VAE_STRICT_LOAD: PASS
TRUNK_BASIS_ARCHIVED_REPLAY_MAX_DIFF_MPS: 2.288818359375e-05
TRUNK_BASIS_REPLAY: PASS
STAGE43_BASELINE_REPLAY: PASS
BASELINE_RMSE_MPS: 1.5763281022681346

BUILDING_TRUNK_ORTHOGONAL_BASIS...
ORACLE_RANK_THRESHOLD: 0.0001 EFFECTIVE_RANK: 40 PAIRED_RMSE_MPS: 1.345833218946872
ORACLE_RANK_THRESHOLD: 1e-06 EFFECTIVE_RANK: 88 PAIRED_RMSE_MPS: 1.2129153465290283

STAGE 44 — DECODER BASIS CAPACITY RESULTS

ARM: encoded_posterior_mean
EFFECTIVE_RANK: encoder
{
  "paired_rmse_mps": 1.5763281022681346,
  "variance_ratio": 0.28245429708074216,
  "step_rms_ratio": 0.09644225172770735,
  "between_field_energy_cv": 0.33358369328697174,
  "band_power_ratios": {
    "low":

In [ ]:

# ================================================================
# WIND VISION — STAGE 45
# CONTROLLED FOURIER-FEATURE DECODER PILOT
# ================================================================
# COMPLETE, SELF-CONTAINED COLAB CELL
#
# Two equal-parameter residual decoder heads:
#   A: raw coordinates + latent code
#   B: same inputs + temporal Fourier features
#
# Original VAE encoder, decoder and checkpoint stay frozen.
# The original latent-flow model is not retrained.
#
# 48 training-subset fields: adapter training
# 16 training-subset fields: internal evaluation
# 0 development fields
# 0 independent-test fields
#
# No new downloads, installations, or dataset copies.
# Model source is temporarily extracted to /content.
# JSON evidence <= 64 KiB.
# An adapter checkpoint <= 1 MiB is saved ONLY if
# prespecified pilot acceptance checks pass.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

import ast
import copy
import gc
import hashlib
import io
import json
import shutil
import sys
import tempfile
import types
import uuid
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as TF

# ------------------------------------------------
# 1. Verified inputs and storage preflight
# ------------------------------------------------

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

ROOT45 = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

RELEASE45 = ROOT45 / (
    "evidence/fullscale_physicsnemo_training/"
    "final_completion_20261108_v1/final_release/"
    "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

REFERENCE45 = ROOT45 / (
    "evidence/stage42_reference_inputs/"
    "training_only_64_bb8fe406dd21025d.npz"
)

EXPECTED_RELEASE45 = (
    "49f28c0dd34a7f0da20433905092d6dbc691124fc70439e813ae1adf281f5c52"
)
EXPECTED_REFERENCE45 = (
    "bb8fe406dd21025d9698ee420bbee0b68d039a13c36ef4ffaf7a28054888e978"
)
EXPECTED_TRAIN45 = (
    "3c5f35ad3b8707de18b708ec22132731dd2256b70cbbd0bb4e15b2b70d164bd6"
)
EXPECTED_TRAIN_IDS45 = (
    "f0b44c3c14a27b3d2bae90cf6ef8867bbaeca255c80803290f4954206fb3e33c"
)
EXPECTED_DEV_IDS45 = (
    "4de9038ec463633bf03a8c08b64ea6a155d54816818339a6647e6b39ac0ba6d7"
)

def sha45(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

print("=" * 72)
print("WIND VISION — STAGE 45")
print("CONTROLLED FOURIER-FEATURE DECODER PILOT")
print("=" * 72)

assert RELEASE45.is_file()
assert REFERENCE45.is_file()
assert sha45(RELEASE45) == EXPECTED_RELEASE45
assert sha45(REFERENCE45) == EXPECTED_REFERENCE45
assert torch.cuda.is_available()

free45 = shutil.disk_usage("/content").free
assert free45 > 512 * 2**20

print("RESEARCH_RELEASE_SHA256: PASS")
print("TRAINING_SUBSET_SHA256: PASS")
print("GPU:", torch.cuda.get_device_name(0))
print("LOCAL_FREE_GIB:", round(free45 / 2**30, 2))
print("DRIVE_QUOTA_NOT_INFERRED_FROM_LOCAL_DISK: TRUE")
print("NEW_DATASET_COPIES: 0")
print("NEW_PACKAGE_INSTALLATIONS: 0")

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False

DEVICE45 = "cuda"

# ------------------------------------------------
# 2. Recover verified 64-field sample
# ------------------------------------------------

with np.load(REFERENCE45, allow_pickle=False) as p:
    real45 = p["wind_mps"].astype(np.float32)
    xyz45 = p["xyz"].astype(np.float32)
    times45 = p["t_grid"].astype(np.float32)
    indices45 = p["field_indices"].copy()

    assert p["source_sha256"].item() == EXPECTED_TRAIN45
    assert p["train_ids_sha256"].item() == EXPECTED_TRAIN_IDS45
    assert p["dev_ids_sha256"].item() == EXPECTED_DEV_IDS45

assert real45.shape == (64, 100, 256)
assert xyz45.shape == (100, 3)
assert times45.shape == (256,)
assert np.isfinite(real45).all()

frozen_ids45 = np.random.default_rng(
    331001
).permutation(5000)

assert np.array_equal(
    indices45, frozen_ids45[500:564]
)

# Internal adapter split, fixed before training.
# All 64 were members of the original VAE training pool;
# this is NOT an independent test split.
ADAPTER_FIT45 = np.arange(0, 48)
ADAPTER_EVAL45 = np.arange(48, 64)

assert not np.isin(
    indices45, frozen_ids45[:500]
).any()

print("ADAPTER_FIT_FIELDS: 48")
print("ADAPTER_EVALUATION_FIELDS: 16")
print("DEVELOPMENT_FIELDS_USED: 0")
print("INDEPENDENT_TEST_ACCESSED: FALSE")

real_norm45 = np.ascontiguousarray(
    (real45 + 2.0) / 49.0,
    dtype=np.float32
)

coords45 = np.concatenate([
    np.broadcast_to(
        xyz45[:, None, :], (100, 256, 3)
    ),
    np.broadcast_to(
        times45[None, :, None], (100, 256, 1)
    )
], axis=-1).reshape(25600, 4).copy()

# ------------------------------------------------
# 3. Define matched coordinate features
# ------------------------------------------------

spatial_min45 = xyz45.min(axis=0)
spatial_span45 = xyz45.max(axis=0) - spatial_min45
spatial_span45 = np.where(
    spatial_span45 > 0, spatial_span45, 1.0
)

space01_45 = (
    xyz45 - spatial_min45
) / spatial_span45

time_span45 = float(
    times45.max() - times45.min()
)
assert time_span45 > 0

time01_45 = (
    (times45 - times45.min()) / time_span45
)

# Eight prespecified temporal harmonics.
HARMONICS45 = (1, 2, 4, 8, 16, 32, 64, 96)

# Use sample-index phase for exact discrete harmonics.
phase45 = np.arange(
    256, dtype=np.float64
) / 256.0

fourier45 = np.stack(
    [
        fn(2.0 * np.pi * k * phase45)
        for k in HARMONICS45
        for fn in (np.sin, np.cos)
    ],
    axis=-1
).astype(np.float32)

assert fourier45.shape == (256, 16)

space_points45 = np.broadcast_to(
    space01_45[:, None, :],
    (100, 256, 3)
)

time_points45 = np.broadcast_to(
    time01_45[None, :, None],
    (100, 256, 1)
)

fourier_points45 = np.broadcast_to(
    fourier45[None, :, :],
    (100, 256, 16)
)

coord_features45 = np.concatenate([
    space_points45,
    time_points45,
    fourier_points45
], axis=-1).reshape(25600, 20).copy()

# Control has identical architecture and parameter
# count but no harmonic information.
control_features45 = coord_features45.copy()
control_features45[:, 4:] = 0.0

assert coord_features45.shape == (25600, 20)
assert control_features45.shape == (25600, 20)

# ------------------------------------------------
# 4. Evaluation functions
# ------------------------------------------------

BANDS45 = {
    "low": (1, 7),
    "middle": (7, 17),
    "high": (17, 65),
    "tail": (65, 129),
}

def stats45(a):
    a = np.asarray(a, dtype=np.float64)

    assert a.ndim == 3
    assert a.shape[1:] == (100, 256)
    assert np.isfinite(a).all()

    centered = a - a.mean(
        axis=-1, keepdims=True
    )

    power = np.abs(
        np.fft.rfft(
            centered * np.hanning(256),
            axis=-1
        )
    )**2

    return {
        "variance": float(np.mean(centered**2)),
        "step_rms": float(np.sqrt(
            np.mean(np.diff(a, axis=-1)**2)
        )),
        "bands": {
            name: float(power[:, :, lo:hi].mean())
            for name, (lo, hi) in BANDS45.items()
        }
    }

real_eval45 = real45[ADAPTER_EVAL45]
reference_stats45 = stats45(real_eval45)

def evaluate45(prediction):
    assert prediction.shape == real_eval45.shape

    s = stats45(prediction)
    r = reference_stats45

    return {
        "paired_rmse_mps": float(np.sqrt(
            np.mean(
                (
                    prediction.astype(np.float64)
                    - real_eval45.astype(np.float64)
                )**2
            )
        )),
        "temporal_variance_ratio": (
            s["variance"] / r["variance"]
        ),
        "step_rms_ratio": (
            s["step_rms"] / r["step_rms"]
        ),
        "band_power_ratios": {
            name: s["bands"][name] / r["bands"][name]
            for name in BANDS45
        },
    }

# ------------------------------------------------
# 5. Restore frozen original model
# ------------------------------------------------

with zipfile.ZipFile(RELEASE45, "r") as archive45:
    assert archive45.testzip() is None

    names45 = archive45.namelist()
    prefix45 = next(
        n[:-len("code/models.py")]
        for n in names45
        if n.endswith("/code/models.py")
    )

    manifest45 = json.loads(
        archive45.read(prefix45 + "MANIFEST.json")
    )
    hashes45 = manifest45["files_sha256"]
    assert len(hashes45) == 101

    def read_verified45(relative):
        data = archive45.read(prefix45 + relative)
        assert hashlib.sha256(data).hexdigest() == (
            hashes45[relative]
        ), relative
        return data

    model_bytes45 = read_verified45("code/models.py")
    origin_bytes45 = read_verified45(
        "checkpoints/native_vae_origin.pt"
    )
    vae_bytes45 = read_verified45(
        "checkpoints/native_vae_v2.pt"
    )
    replay_bytes45 = read_verified45(
        "evidence/cell02_native_fields.npz"
    )

    with np.load(
        io.BytesIO(replay_bytes45),
        allow_pickle=False
    ) as p:
        replay_fields45 = p["wind_mps"].astype(np.float32)
        replay_z45 = p[
            "generated_decoder_z"
        ].astype(np.float32)
        assert np.array_equal(p["xyz"], xyz45)
        assert np.array_equal(p["t_grid"], times45)

    # Adapt framework base-class import only.
    source45 = model_bytes45.decode("utf-8")
    tree45 = ast.parse(source45)

    imports45 = [
        node for node in tree45.body
        if isinstance(node, ast.ImportFrom)
        and node.module == "physicsnemo.core.module"
        and len(node.names) == 1
        and node.names[0].name == "Module"
        and node.names[0].asname == "PhysicsNeMoModule"
    ]
    assert len(imports45) == 1

    framework_imports45 = [
        node for node in ast.walk(tree45)
        if (
            isinstance(node, ast.ImportFrom)
            and (node.module or "").startswith("physicsnemo")
        ) or (
            isinstance(node, ast.Import)
            and any(
                a.name.startswith("physicsnemo")
                for a in node.names
            )
        )
    ]
    assert len(framework_imports45) == 1

    lines45 = source45.splitlines(keepends=True)
    imp45 = imports45[0]
    lines45[imp45.lineno-1:imp45.end_lineno] = [
        "from torch.nn import Module as PhysicsNeMoModule\n"
    ]
    adapted_source45 = "".join(lines45)
    ast.parse(adapted_source45)

    # Extract only verified source into temporary local disk.
    with tempfile.TemporaryDirectory(
        prefix="wind_stage45_", dir="/content"
    ) as temporary45:
        tmp45 = Path(temporary45)
        upstream45 = prefix45 + "upstream_nasa/"

        for member in names45:
            if not (
                member.startswith(upstream45 + "clfm/")
                and member.endswith(".py")
            ):
                continue

            relative = member[len(upstream45):]
            assert ".." not in Path(relative).parts

            dest = tmp45 / relative
            dest.parent.mkdir(
                parents=True, exist_ok=True
            )
            dest.write_bytes(
                read_verified45("upstream_nasa/" + relative)
            )

        old_clfm45 = {
            k: v for k, v in sys.modules.items()
            if k == "clfm" or k.startswith("clfm.")
        }
        for k in old_clfm45:
            del sys.modules[k]

        sys.path.insert(0, str(tmp45))

        try:
            mod45 = types.ModuleType(
                "wind_stage45_compatible_model"
            )
            mod45.__file__ = str(tmp45 / "models.py")

            exec(
                compile(
                    adapted_source45,
                    mod45.__file__,
                    "exec"
                ),
                mod45.__dict__
            )

            origin45 = torch.load(
                io.BytesIO(origin_bytes45),
                map_location="cpu",
                weights_only=True
            )
            trained45 = torch.load(
                io.BytesIO(vae_bytes45),
                map_location="cpu",
                weights_only=True
            )

            cfg45 = origin45["architecture_config"]
            vae45 = mod45.PhysicsNeMoWindVAE(
                cfg45
            ).to(DEVICE45).eval()

            vae45.load_state_dict(
                trained45["model_state"],
                strict=True
            )
            vae45.requires_grad_(False)

            sensor_ids45 = np.asarray(
                cfg45["sensor_indices"],
                dtype=np.int64
            )
            assert sensor_ids45.shape == (40,)
            assert np.array_equal(
                sensor_ids45,
                np.concatenate([
                    np.arange(k, k+10)
                    for k in (0, 30, 60, 90)
                ])
            )

            # --------------------------------------------
            # 6. Replay the exact frozen decoder
            # --------------------------------------------

            coords_gpu45 = torch.from_numpy(
                coords45
            ).to(DEVICE45)

            @torch.inference_mode()
            def frozen_decode45(z):
                outputs = []

                for j in range(0, len(z), 2):
                    latent = torch.from_numpy(
                        np.ascontiguousarray(
                            z[j:j+2], dtype=np.float32
                        )
                    ).to(DEVICE45)

                    y = vae45.decode(
                        latent,
                        coords_gpu45[None].expand(
                            len(latent), -1, -1
                        )
                    ).reshape(len(latent), 100, 256)

                    assert torch.isfinite(y).all()
                    outputs.append(
                        y.float().cpu().numpy()
                    )

                return np.concatenate(outputs)

            replay_norm45 = frozen_decode45(
                replay_z45[:2]
            )
            replay_mps45 = (
                replay_norm45 * 49.0 - 2.0
            )

            replay_error45 = float(np.max(
                np.abs(
                    replay_mps45 - replay_fields45[:2]
                )
            ))

            assert replay_error45 < 1e-3
            print("FROZEN_DECODER_REPLAY: PASS")
            print(
                "REPLAY_MAX_DIFF_MPS:",
                replay_error45
            )

            # --------------------------------------------
            # 7. Cache 64 frozen encoder codes and outputs
            # --------------------------------------------

            latents45 = []

            with torch.inference_mode():
                for j in range(0, 64, 2):
                    observed = torch.from_numpy(
                        np.ascontiguousarray(
                            real_norm45[
                                j:j+2, sensor_ids45, :
                            ]
                        )
                    ).to(DEVICE45)

                    assert observed.shape[1:] == (
                        40, 256
                    )

                    mu, logvar = vae45.encode(observed)
                    assert mu.shape == (len(observed), 128)
                    assert torch.isfinite(mu).all()
                    latents45.append(
                        mu.float().cpu().numpy()
                    )

            latents45 = np.concatenate(latents45)
            assert latents45.shape == (64, 128)

            baseline_norm45 = frozen_decode45(
                latents45
            )

            baseline45 = (
                baseline_norm45 * 49.0 - 2.0
            )

            assert baseline45.shape == real45.shape
            baseline_metric45 = evaluate45(
                baseline45[ADAPTER_EVAL45]
            )

            print("FROZEN_ENCODER_DECODER: PASS")
            print(
                "EVALUATION_BASELINE_RMSE_MPS:",
                baseline_metric45["paired_rmse_mps"]
            )

            del vae45, coords_gpu45
            torch.cuda.empty_cache()

        finally:
            if str(tmp45) in sys.path:
                sys.path.remove(str(tmp45))

            for k in list(sys.modules):
                if k == "clfm" or k.startswith("clfm."):
                    del sys.modules[k]

            sys.modules.update(old_clfm45)

# ------------------------------------------------
# 8. Prepare small GPU-resident training inputs
# ------------------------------------------------

latent_gpu45 = torch.from_numpy(
    latents45
).to(DEVICE45)

baseline_gpu45 = torch.from_numpy(
    baseline_norm45.reshape(64, 25600)
).to(DEVICE45)

target_gpu45 = torch.from_numpy(
    real_norm45.reshape(64, 25600)
).to(DEVICE45)

features_control45 = torch.from_numpy(
    control_features45
).to(DEVICE45)

features_fourier45 = torch.from_numpy(
    coord_features45
).to(DEVICE45)

assert (
    features_control45.shape
    == features_fourier45.shape
    == (25600, 20)
)

# ------------------------------------------------
# 9. Identical residual-network architecture
# ------------------------------------------------

class ResidualHead45(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(148, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
            nn.Linear(96, 1)
        )

        # Exactly zero residual before training.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, z, coordinate_features):
        # z: (fields, 128)
        # coordinates: (points, 20)
        nf = z.shape[0]
        npnt = coordinate_features.shape[0]

        x = torch.cat([
            z[:, None, :].expand(nf, npnt, 128),
            coordinate_features[None].expand(
                nf, npnt, 20
            )
        ], dim=-1)

        return self.net(
            x.reshape(-1, 148)
        ).reshape(nf, npnt)


torch.manual_seed(450045)

initial_head45 = ResidualHead45()

control45 = copy.deepcopy(
    initial_head45
).to(DEVICE45)

fourier_head45 = copy.deepcopy(
    initial_head45
).to(DEVICE45)

nparams45 = sum(
    p.numel() for p in control45.parameters()
)

assert nparams45 == sum(
    p.numel() for p in fourier_head45.parameters()
)

# Same initial weights and equal parameter count.
for p, q in zip(
    control45.parameters(),
    fourier_head45.parameters()
):
    assert torch.equal(p, q)

print("\nRESIDUAL_HEAD_PARAMETERS:", nparams45)
print("INITIAL_WEIGHTS_MATCHED: TRUE")
print("CONTROL_HARMONIC_FEATURES: ZERO")
print("FOURIER_HARMONICS:", HARMONICS45)

# ------------------------------------------------
# 10. Prespecified optimizer schedule
# ------------------------------------------------

STEPS45 = 160
LR45 = 3e-4
SPECTRAL_WEIGHT45 = 2e-5
FIELDS_PER_STEP45 = 2
SENSORS_PER_STEP45 = 8

schedule_rng45 = np.random.default_rng(450045)

schedule45 = [
    (
        schedule_rng45.choice(
            ADAPTER_FIT45,
            size=FIELDS_PER_STEP45,
            replace=False
        ),
        schedule_rng45.choice(
            100,
            size=SENSORS_PER_STEP45,
            replace=False
        )
    )
    for _ in range(STEPS45)
]

window45 = torch.from_numpy(
    np.hanning(256).astype(np.float32)
).to(DEVICE45)

time_offsets45 = torch.arange(
    256, device=DEVICE45
)

def train_arm45(model, feature_matrix, arm_name):
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR45,
        weight_decay=0.0
    )

    history = []

    for step, (field_ids, sensor_ids) in enumerate(
        schedule45, start=1
    ):
        fids = torch.as_tensor(
            field_ids,
            dtype=torch.long,
            device=DEVICE45
        )

        sids = torch.as_tensor(
            sensor_ids,
            dtype=torch.long,
            device=DEVICE45
        )

        point_ids = (
            sids[:, None] * 256
            + time_offsets45[None, :]
        ).reshape(-1)

        z = latent_gpu45[fids]
        features = feature_matrix[point_ids]

        base = baseline_gpu45[fids][
            :, point_ids
        ].reshape(2, 8, 256)

        truth = target_gpu45[fids][
            :, point_ids
        ].reshape(2, 8, 256)

        correction = model(
            z, features
        ).reshape(2, 8, 256)

        prediction = base + correction

        mse = TF.mse_loss(
            prediction, truth
        )

        # Match the reference's frequency-band definition.
        pred_centered = prediction - prediction.mean(
            dim=-1, keepdim=True
        )
        true_centered = truth - truth.mean(
            dim=-1, keepdim=True
        )

        pred_power = torch.abs(
            torch.fft.rfft(
                pred_centered * window45, dim=-1
            )
        )**2

        true_power = torch.abs(
            torch.fft.rfft(
                true_centered * window45, dim=-1
            )
        )**2

        band_losses = []

        for lo, hi in BANDS45.values():
            pred_band = pred_power[
                ..., lo:hi
            ].mean()
            true_band = true_power[
                ..., lo:hi
            ].mean()

            band_losses.append(
                torch.log(
                    (pred_band + 1e-8)
                    / (true_band + 1e-8)
                )**2
            )

        spectral_loss = torch.stack(
            band_losses
        ).mean()

        loss = (
            mse
            + SPECTRAL_WEIGHT45 * spectral_loss
        )

        assert torch.isfinite(loss)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        history.append({
            "step": step,
            "mse": float(mse.detach().item()),
            "spectral_loss": float(
                spectral_loss.detach().item()
            ),
            "total": float(loss.detach().item())
        })

        if step % 40 == 0:
            print(
                arm_name,
                "STEP:", step,
                "MSE:", round(history[-1]["mse"], 8),
                "SPECTRAL:", round(
                    history[-1]["spectral_loss"], 5
                )
            )

    return history

print("\nTRAINING_CONTROL_ARM...")
history_control45 = train_arm45(
    control45,
    features_control45,
    "RAW_TIME_CONTROL"
)

print("\nTRAINING_FOURIER_ARM...")
history_fourier45 = train_arm45(
    fourier_head45,
    features_fourier45,
    "FOURIER_FEATURES"
)

# ------------------------------------------------
# 11. Evaluate all 100 sensors on 16 fields
# ------------------------------------------------

@torch.inference_mode()
def infer_arm45(model, feature_matrix):
    model.eval()
    output = np.empty(
        (16, 100, 256),
        dtype=np.float32
    )

    for index, field_id in enumerate(ADAPTER_EVAL45):
        corrections = []

        for start in range(0, 25600, 4096):
            stop = min(start + 4096, 25600)

            correction = model(
                latent_gpu45[
                    field_id:field_id+1
                ],
                feature_matrix[start:stop]
            )

            corrections.append(
                correction.reshape(-1)
            )

        full_correction = torch.cat(
            corrections, dim=0
        )

        normalized_prediction = (
            baseline_gpu45[field_id]
            + full_correction
        )

        result = (
            normalized_prediction.reshape(100, 256)
            * 49.0 - 2.0
        )

        assert torch.isfinite(result).all()

        output[index] = result.cpu().numpy()

    return output

control_predictions45 = infer_arm45(
    control45,
    features_control45
)

fourier_predictions45 = infer_arm45(
    fourier_head45,
    features_fourier45
)

control_metric45 = evaluate45(
    control_predictions45
)

fourier_metric45 = evaluate45(
    fourier_predictions45
)

# ------------------------------------------------
# 12. Prespecified go/no-go criterion
# ------------------------------------------------

base_rmse45 = baseline_metric45[
    "paired_rmse_mps"
]
control_rmse45 = control_metric45[
    "paired_rmse_mps"
]
fourier_rmse45 = fourier_metric45[
    "paired_rmse_mps"
]

base_step45 = baseline_metric45[
    "step_rms_ratio"
]
fourier_step45 = fourier_metric45[
    "step_rms_ratio"
]

# A pilot improvement must satisfy ALL conditions:
# - RMSE improves >= 2% versus frozen VAE
# - RMSE improves >= 1% versus matched control
# - adjacent-step RMS improves >= 50% relatively
# - adjacent-step RMS does not overshoot reference
# - middle/high-band power is not > 3x reference
# - outputs remain finite
#
# This is a pilot acceptance gate, not proof of
# generalization or a paper-level scientific result.

gate45 = {
    "rmse_vs_frozen_2pct": (
        fourier_rmse45 <= 0.98 * base_rmse45
    ),
    "rmse_vs_control_1pct": (
        fourier_rmse45 <= 0.99 * control_rmse45
    ),
    "step_rms_relative_gain_50pct": (
        fourier_step45 >= 1.5 * base_step45
    ),
    "step_rms_not_above_reference": (
        fourier_step45 <= 1.0
    ),
    "middle_band_not_above_3x": (
        fourier_metric45[
            "band_power_ratios"
        ]["middle"] <= 3.0
    ),
    "high_band_not_above_3x": (
        fourier_metric45[
            "band_power_ratios"
        ]["high"] <= 3.0
    ),
    "finite_output": bool(
        np.isfinite(fourier_predictions45).all()
    )
}

pilot_pass45 = bool(all(gate45.values()))

print("\n" + "=" * 72)
print("STAGE 45 — PRESPECIFIED PILOT RESULTS")
print("=" * 72)

for name, metrics in (
    ("frozen_vae", baseline_metric45),
    ("matched_raw_time_control", control_metric45),
    ("fourier_feature_adapter", fourier_metric45)
):
    print("\nARM:", name)
    print(json.dumps(metrics, indent=2))

print("\nPILOT_GATE:")
print(json.dumps(gate45, indent=2))
print("STAGE45_PILOT_PASS:", pilot_pass45)

# ------------------------------------------------
# 13. Preserve tiny evidence, optional small adapter
# ------------------------------------------------

record45 = {
    "format": "wind_vision_stage45_fourier_adapter_pilot_v1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "release_sha256": EXPECTED_RELEASE45,
    "training_subset_sha256": EXPECTED_REFERENCE45,
    "model_source_sha256": hashlib.sha256(
        model_bytes45
    ).hexdigest(),
    "backend": "PyTorch-compatible frozen VAE",
    "training_fields": 48,
    "internal_evaluation_fields": 16,
    "development_fields": 0,
    "independent_test_accessed": False,
    "original_vae_updated": False,
    "latent_flow_updated": False,
    "adapter_trained": True,
    "matched_parameter_count": nparams45,
    "harmonics": list(HARMONICS45),
    "steps_per_arm": STEPS45,
    "learning_rate": LR45,
    "spectral_weight": SPECTRAL_WEIGHT45,
    "fields_per_step": FIELDS_PER_STEP45,
    "sensors_per_step": SENSORS_PER_STEP45,
    "schedule_seed": 450045,
    "archived_replay_max_diff_mps": replay_error45,
    "metrics": {
        "frozen_vae": baseline_metric45,
        "raw_time_control": control_metric45,
        "fourier_adapter": fourier_metric45
    },
    "pilot_gate": gate45,
    "pilot_pass": pilot_pass45,
    "training_history": {
        "control": history_control45,
        "fourier": history_fourier45
    },
    "limitations": [
        "Internal pilot on 64 original-training-pool fields",
        "The 16 evaluation fields were not used for adapter optimization",
        "The frozen VAE was pretrained on the broader training pool",
        "Not independent-test or out-of-distribution validation",
        "Not a full new PhysicsNeMo VAE training run",
        "No native PhysicsNeMo speed or full-pipeline claim",
        "Adapters cannot establish operational wind realism"
    ]
}

payload45 = (
    json.dumps(
        record45,
        sort_keys=True,
        indent=2,
        allow_nan=False
    ) + "\n"
).encode("utf-8")

# Training histories are small; enforce a strict cap.
assert len(payload45) < 64 * 1024

tag45 = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    ) + "_" + uuid.uuid4().hex[:8]
)

out45 = ROOT45 / "evidence" / (
    "stage45_fourier_adapter_pilot_" + tag45
)

assert out45.parent.is_dir()
out45.mkdir(parents=False, exist_ok=False)

report45 = out45 / "stage45_report.json"
with report45.open("xb") as f:
    f.write(payload45)

files_saved45 = {
    report45.name: sha45(report45)
}

# Only retain new model weights when all pilot gates pass.
if pilot_pass45:
    adapter_buffer45 = io.BytesIO()

    torch.save({
        "format": "wind_stage45_fourier_adapter_v1",
        "model_state": {
            k: v.detach().cpu()
            for k, v in fourier_head45.state_dict().items()
        },
        "latent_dim": 128,
        "input_dim": 148,
        "hidden_dim": 96,
        "harmonics": HARMONICS45,
        "release_sha256": EXPECTED_RELEASE45,
        "reference_sha256": EXPECTED_REFERENCE45,
        "training_seed": 450045
    }, adapter_buffer45)

    adapter_bytes45 = adapter_buffer45.getvalue()

    assert len(adapter_bytes45) <= 1024 * 1024

    adapter_path45 = out45 / "fourier_adapter.pt"

    with adapter_path45.open("xb") as f:
        f.write(adapter_bytes45)

    files_saved45[adapter_path45.name] = (
        sha45(adapter_path45)
    )

manifest_bytes45 = (
    json.dumps(
        files_saved45,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

manifest_path45 = out45 / "SHA256SUMS.json"
with manifest_path45.open("xb") as f:
    f.write(manifest_bytes45)

for file_name, expected_hash in files_saved45.items():
    assert sha45(out45 / file_name) == expected_hash

saved_bytes45 = sum(
    (out45 / name).stat().st_size
    for name in files_saved45
) + manifest_path45.stat().st_size

print("\n" + "=" * 72)
print("STAGE 45 — EVIDENCE AND STORAGE")
print("=" * 72)
print("STAGE45_EVIDENCE:", out45)
print("STAGE45_MANIFEST_SHA256:", sha45(manifest_path45))
print("NEW_DRIVE_BYTES:", saved_bytes45)
print("ADAPTER_CHECKPOINT_SAVED:", pilot_pass45)
print("NEW_DATASET_COPIES: 0")
print("NEW_ORIGINAL_VAE_CHECKPOINTS: 0")
print("NEW_PACKAGE_INSTALLATIONS: 0")
print("DEVELOPMENT_FIELDS_ACCESSED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")
print("STAGE45_COMPLETE")


WIND VISION — STAGE 45
CONTROLLED FOURIER-FEATURE DECODER PILOT
RESEARCH_RELEASE_SHA256: PASS
TRAINING_SUBSET_SHA256: PASS
GPU: NVIDIA L4
LOCAL_FREE_GIB: 234.55
DRIVE_QUOTA_NOT_INFERRED_FROM_LOCAL_DISK: TRUE
NEW_DATASET_COPIES: 0
NEW_PACKAGE_INSTALLATIONS: 0
ADAPTER_FIT_FIELDS: 48
ADAPTER_EVALUATION_FIELDS: 16
DEVELOPMENT_FIELDS_USED: 0
INDEPENDENT_TEST_ACCESSED: FALSE
FROZEN_DECODER_REPLAY: PASS
REPLAY_MAX_DIFF_MPS: 2.288818359375e-05
FROZEN_ENCODER_DECODER: PASS
EVALUATION_BASELINE_RMSE_MPS: 1.555941545741743

RESIDUAL_HEAD_PARAMETERS: 23713
INITIAL_WEIGHTS_MATCHED: TRUE
CONTROL_HARMONIC_FEATURES: ZERO
FOURIER_HARMONICS: (1, 2, 4, 8, 16, 32, 64, 96)

TRAINING_CONTROL_ARM...
RAW_TIME_CONTROL STEP: 40 MSE: 0.00038739 SPECTRAL: 42.69176
RAW_TIME_CONTROL STEP: 80 MSE: 0.0011587 SPECTRAL: 107.01772
RAW_TIME_CONTROL STEP: 120 MSE: 0.00039995 SPECTRAL: 49.03004
RAW_TIME_CONTROL STEP: 160 MSE: 0.00096894 SPECTRAL: 100.73071

TRAINING_FOURIER_ARM...
FOURIER_FEATURES STEP: 40 MSE: 0.00041189 S

In [1]:

# ============================================================
# WIND VISION — NASA EXIT PRESENTATION EVIDENCE COLLECTOR
# ============================================================
#
# READ-ONLY GOOGLE DRIVE OPERATION
#
# Creates one ZIP in temporary Colab /content storage,
# then downloads it directly to your Mac.
#
# NO TRAINING
# NO GPU REQUIRED
# NO DATASET DOWNLOAD
# NO GOOGLE DRIVE FILES CREATED OR MODIFIED
# NO LARGE CHECKPOINTS COPIED
# NO INDEPENDENT TEST ACCESS
# ============================================================

from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import zipfile

print("=" * 70)
print("WIND VISION — EXIT PRESENTATION EVIDENCE COLLECTOR")
print("=" * 70)

# ------------------------------------------------------------
# 1. Connect to existing Google Drive
# ------------------------------------------------------------

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/CLFM_Research_Results/"
    "physicsnemo_proposal_reproduction"
)

EVIDENCE = ROOT / "evidence"

assert EVIDENCE.is_dir(), (
    "Research evidence folder not found on Google Drive."
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

# ------------------------------------------------------------
# 2. Exact saved experiment paths
# ------------------------------------------------------------

reference = EVIDENCE / (
    "stage42_reference_inputs/"
    "training_only_64_bb8fe406dd21025d.npz"
)

stage43 = EVIDENCE / (
    "stage43_sparse_encoder_diagnostic_"
    "20261110T204242Z_d37ba384/"
    "stage43_report.json"
)

stage44 = EVIDENCE / (
    "stage44_decoder_basis_capacity_"
    "20261110T204924Z_46c6cdfc/"
    "stage44_report.json"
)

stage45 = EVIDENCE / (
    "stage45_fourier_adapter_pilot_"
    "20261110T205833Z_3a26fa7e/"
    "stage45_report.json"
)

release = EVIDENCE / (
    "fullscale_physicsnemo_training/"
    "final_completion_20261108_v1/"
    "final_release/"
    "WIND_VISION_PHYSICSNEMO_FINAL_20261108_V1.zip"
)

EXPECTED_RELEASE_SHA = (
    "49f28c0dd34a7f0da20433905092d"
    "6dbc691124fc70439e813ae1adf281f5c52"
)

EXPECTED_REFERENCE_SHA = (
    "bb8fe406dd21025d9698ee420bbee0b68"
    "d039a13c36ef4ffaf7a28054888e978"
)

required = {
    "reference_training_64.npz": reference,
    "stage43_report.json": stage43,
    "stage44_report.json": stage44,
    "stage45_report.json": stage45,
}

for name, path in required.items():
    assert path.is_file(), f"Missing required file: {path}"
    print("FOUND:", name, "(", path.stat().st_size, "bytes)")

assert release.is_file(), "Final research ZIP not found"

assert sha256_file(release) == EXPECTED_RELEASE_SHA
assert sha256_file(reference) == EXPECTED_REFERENCE_SHA

print("\nRESEARCH_RELEASE_SHA256: VERIFIED")
print("STAGE42_REFERENCE_SHA256: VERIFIED")

# ------------------------------------------------------------
# 3. Prepare a unique temporary ZIP on Colab only
# ------------------------------------------------------------

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

output = Path(
    f"/content/WIND_VISION_EXIT_PRESENTATION_"
    f"EVIDENCE_{timestamp}.zip"
)

assert not output.exists()

saved_files = {}
MAX_BUNDLE_BYTES = 40 * 1024 * 1024
raw_budget = 0

def add_bytes(bundle, destination, data, origin):
    global raw_budget

    raw_budget += len(data)

    assert raw_budget <= MAX_BUNDLE_BYTES, (
        "Evidence exceeds the 40 MiB extraction budget."
    )

    bundle.writestr(destination, data)

    saved_files[destination] = {
        "source": origin,
        "size_bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest()
    }

# ------------------------------------------------------------
# 4. Collect source arrays, figures and benchmark reports
# ------------------------------------------------------------

with zipfile.ZipFile(
    output, "x",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=5
) as bundle:

    # Verified small files directly from Google Drive.
    for name, path in required.items():
        data = path.read_bytes()

        add_bytes(
            bundle,
            "recent_experiments/" + name,
            data,
            str(path)
        )

    # Extract presentation-specific artifacts directly from
    # the verified November 8 research release.
    #
    # No model checkpoints or training datasets extracted.
    with zipfile.ZipFile(release, "r") as original:

        names = original.namelist()

        prefix = next(
            name[:-len("MANIFEST.json")]
            for name in names
            if name.endswith("/MANIFEST.json")
        )

        release_manifest = json.loads(
            original.read(prefix + "MANIFEST.json")
        )

        hashes = release_manifest["files_sha256"]

        selected_count = 0

        for member in names:

            if not member.startswith(prefix):
                continue

            relative = member[len(prefix):]

            is_figure = (
                relative.startswith("figures/")
                and relative.lower().endswith(
                    (".png", ".jpg", ".jpeg", ".svg")
                )
            )

            is_wind_array = relative in {
                "evidence/cell02_native_fields.npz",
                "evidence/cell03_original_fields.npz",
            }

            is_evidence_report = (
                relative.startswith("evidence/")
                and relative.endswith(".json")
            )

            is_documentation = relative in {
                "MANIFEST.json",
                "TECHNICAL_CLOSEOUT.md",
                "README.md",
            }

            if not any([
                is_figure,
                is_wind_array,
                is_evidence_report,
                is_documentation,
            ]):
                continue

            info = original.getinfo(member)

            if info.file_size > 8 * 1024 * 1024:
                print("SKIPPED LARGE MEMBER:", relative)
                continue

            data = original.read(member)

            if relative != "MANIFEST.json":
                assert (
                    hashlib.sha256(data).hexdigest()
                    == hashes[relative]
                ), f"Archived hash mismatch: {relative}"

            add_bytes(
                bundle,
                "nov8_verified_release/" + relative,
                data,
                str(release) + "::" + relative
            )

            selected_count += 1

        print(
            "\nNOVEMBER 8 RELEASE COMPONENTS EXTRACTED:",
            selected_count
        )

    # Optional: Stage 39 statistical-model evidence.
    # Include only small JSON files, not large arrays.
    stage39_folder = EVIDENCE / (
        "fullscale_physicsnemo_training/"
        "stage39_reference_band_audit_"
        "20261110T165230Z_709551a9"
    )

    if stage39_folder.is_dir():

        for path in sorted(stage39_folder.rglob("*.json")):

            if path.stat().st_size > 1024 * 1024:
                continue

            relative = path.relative_to(
                stage39_folder
            ).as_posix()

            add_bytes(
                bundle,
                "stage39_statistical_analysis/" + relative,
                path.read_bytes(),
                str(path)
            )

    # File provenance index.
    index = {
        "purpose": "NASA Langley Nov 20 exit presentation",
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "original_release_sha256": EXPECTED_RELEASE_SHA,
        "stage42_reference_sha256": EXPECTED_REFERENCE_SHA,
        "files": saved_files,
        "stage46_executed": False,
        "independent_test_accessed": False,
        "original_checkpoints_included": False,
        "google_drive_modified": False,
    }

    bundle.writestr(
        "EVIDENCE_INDEX.json",
        json.dumps(
            index, indent=2, sort_keys=True
        )
    )

# ------------------------------------------------------------
# 5. Verify completed local bundle
# ------------------------------------------------------------

assert output.stat().st_size <= MAX_BUNDLE_BYTES

with zipfile.ZipFile(output, "r") as check:
    assert check.testzip() is None

    included_names = set(check.namelist())

    for name in required:
        assert (
            "recent_experiments/" + name
        ) in included_names

    for expected in (
        "evidence/cell02_native_fields.npz",
        "evidence/cell03_original_fields.npz",
    ):
        assert (
            "nov8_verified_release/" + expected
        ) in included_names, (
            "Required generated wind array missing: " + expected
        )

print("\n" + "=" * 70)
print("PRESENTATION EVIDENCE PACKAGE: READY")
print("=" * 70)

print("FILE:", output.name)
print("SIZE_MIB:", round(output.stat().st_size / 2**20, 3))
print("SHA256:", sha256_file(output))
print("FILES_INCLUDED:", len(saved_files) + 1)
print("NEW_GOOGLE_DRIVE_BYTES:", 0)
print("CHECKPOINTS_MODIFIED: FALSE")
print("INDEPENDENT_TEST_ACCESSED: FALSE")

print("\nDOWNLOADING PACKAGE TO YOUR MAC...")

files.download(str(output))


WIND VISION — EXIT PRESENTATION EVIDENCE COLLECTOR
Mounted at /content/drive
FOUND: reference_training_64.npz ( 5549507 bytes)
FOUND: stage43_report.json ( 5073 bytes)
FOUND: stage44_report.json ( 2461 bytes)
FOUND: stage45_report.json ( 53505 bytes)

RESEARCH_RELEASE_SHA256: VERIFIED
STAGE42_REFERENCE_SHA256: VERIFIED

NOVEMBER 8 RELEASE COMPONENTS EXTRACTED: 17

PRESENTATION EVIDENCE PACKAGE: READY
FILE: WIND_VISION_EXIT_PRESENTATION_EVIDENCE_20261111T225254Z.zip
SIZE_MIB: 8.456
SHA256: 837efe09dd3184beb4bd4b7efb38c223ce1fbd7d0dcde1a809c2f873662162a7
FILES_INCLUDED: 24
NEW_GOOGLE_DRIVE_BYTES: 0
CHECKPOINTS_MODIFIED: FALSE
INDEPENDENT_TEST_ACCESSED: FALSE

DOWNLOADING PACKAGE TO YOUR MAC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:

# ============================================================
# WIND VISION — DRIVE + COLAB SOURCE INVENTORY
# READ-ONLY RESEARCH DISCOVERY
# ============================================================

from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import json, os

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

mydrive = Path("/content/drive/MyDrive")

roots = [
    mydrive / "CLFM_Research_Results",
    mydrive / "CLFM_CLOUD_BUNDLE.zip",
]

# Include possible saved research notebooks.
notebook_folder = mydrive / "Colab Notebooks"

if notebook_folder.is_dir():
    for p in notebook_folder.iterdir():
        if p.is_file() and any(
            word in p.name.lower()
            for word in (
                "wind", "clfm", "physicsnemo"
            )
        ):
            roots.append(p)

# Detect surviving temporary local research files.
for p in Path("/content").iterdir():
    if p.name == "drive":
        continue
    if any(
        word in p.name.lower()
        for word in (
            "wind", "clfm", "physicsnemo",
            "stage33", "stage34", "stage35",
            "stage36", "stage37", "stage38",
            "stage39", "stage40", "stage41",
            "stage42", "stage43", "stage44",
            "stage45",
        )
    ):
        roots.append(p)

records = []
summaries = []

for root in roots:
    if not root.exists() or root.is_symlink():
        continue

    total = 0
    count = 0
    errors = 0

    if root.is_file():
        iterator = [(str(root.parent), [], [root.name])]
    else:
        iterator = os.walk(root, followlinks=False)

    for directory, dirs, filenames in iterator:
        dirs[:] = [
            d for d in dirs
            if d != "_research_vault"
            and not (Path(directory) / d).is_symlink()
        ]

        for filename in filenames:
            path = Path(directory) / filename
            try:
                if path.is_symlink():
                    continue
                stat = path.stat()
                records.append({
                    "source_root": str(root),
                    "path": str(path),
                    "size_bytes": stat.st_size,
                    "modified_epoch": stat.st_mtime,
                })
                total += stat.st_size
                count += 1
            except (OSError, PermissionError):
                errors += 1

    summaries.append({
        "root": str(root),
        "files": count,
        "size_gib": round(total / 2**30, 3),
        "read_errors": errors,
    })

report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sources": summaries,
    "files": records,
    "last_completed_research_stage": 45,
    "stage46_status": "NOT EXECUTED",
    "independent_test_accessed_by_inventory": False,
    "storage_mutation": "NONE",
}

output = Path(
    "/content/WIND_VISION_DRIVE_COLAB_INVENTORY.json"
)

output.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("=" * 70)
print("WIND VISION — DRIVE + COLAB INVENTORY")
print("=" * 70)

for item in summaries:
    print(
        f"{item['files']:>7} files | "
        f"{item['size_gib']:>8.3f} GiB | "
        f"{item['root']}"
    )

print("\nTOTAL_IDENTIFIED_FILES:", len(records))
print("STAGE45_LAST_EXECUTED: TRUE")
print("STAGE46_EXECUTED: FALSE")
print("GOOGLE_DRIVE_MODIFIED: FALSE")
print("INVENTORY_COMPLETE")

files.download(str(output))


Mounted at /content/drive
WIND VISION — DRIVE + COLAB INVENTORY
    719 files |    1.916 GiB | /content/drive/MyDrive/CLFM_Research_Results
      1 files |    0.980 GiB | /content/drive/MyDrive/CLFM_CLOUD_BUNDLE.zip
      1 files |    0.000 GiB | /content/drive/MyDrive/Colab Notebooks/Colab_Pro_Day0_CLFM.ipynb
      1 files |    0.001 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_PhysicsNeMo_Day1.ipynb
      1 files |    0.006 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_PhysicsNeMo_Audited.ipynb
      1 files |    0.001 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_P32_P43_Live_Snapshot_20261103.ipynb
      1 files |    0.002 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_PhysicsNeMo_Focused_Continuation_Gate_Audit.ipynb
      1 files |    0.002 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_PhysicsNeMo_FINAL_COMPLETION_20261108.ipynb
      1 files |    0.000 GiB | /content/drive/MyDrive/Colab Notebooks/Wind_VISION_Stage34_Colab_Co

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>